In [ ]:
library(Seurat)

In [ ]:
own <- readRDS("a-scientific data数据+分析代码/未整合-新猕猴数据整合+跨物种代码 copy.ipynb/classFiltSeurat_filt_allSubtypes.rds")

# 1. 基本结构
own
head(rownames(own), 20)        # 基因名
head(colnames(own), 5)         # 细胞 barcode
dim(own)                       # 基因数 x 细胞数

# 2. 元数据列
colnames(own@meta.data)

# 3. 大类/细类分布
sort(table(own$cell_type_fine), decreasing = TRUE)

# 4. 找 5-HT 相关类群
grep("5-?HT|serot|raphe", unique(own$cell_type_fine),
     value = TRUE, ignore.case = TRUE)

# 5. 预测大类的分布（merge 时做粗注释用）
if ("maxPredClass" %in% colnames(own@meta.data)) {
  sort(table(own$maxPredClass), decreasing = TRUE)
} else {
  cat("没有 maxPredClass 列\n")
}

In [ ]:
# 跨物种

In [ ]:
library(Seurat)

all_seu <- readRDS("./data/classFiltSeurat_filt_allSubtypes.rds")

# ---- 1. 对象概览：assay、细胞数、基因数、降维 ----
all_seu

# ---- 2. metadata 全部列名 ----
colnames(all_seu@meta.data)

# ---- 3. 逐列看内容（前 10 行）----
for (cc in colnames(all_seu@meta.data)) {
  cat("\n========== ", cc, " ==========\n")
  print(head(all_seu@meta.data[[cc]], 10))
}

# ---- 4. 关键列的取值分布 ----
# 大类
table(all_seu$cell_class)

# 亚类（可能很长，直接看 levels 和数量）
length(levels(factor(all_seu$cell_type_new)))
sort(table(all_seu$cell_type_new), decreasing = TRUE)

# donor
table(all_seu$donor)

# ---- 5. 有哪些 assay / layer ----
Assays(all_seu)
Layers(all_seu[["RNA"]])
Layers(all_seu[["SCT"]])

# ---- 6. 有哪些降维、每个降维的维度数 ----
Reductions(all_seu)
sapply(Reductions(all_seu), function(r) dim(Embeddings(all_seu, r)))

# ---- 7. 聚类结果 ----
grep("snn_res", colnames(all_seu@meta.data), value = TRUE)
table(all_seu$seurat_clusters)

# ---- 8. Idents 当前是什么 ----
levels(Idents(all_seu))

In [ ]:
suppressMessages({
  library(tidyverse)
  library(Seurat)
  library(patchwork)
  library(Matrix)
})

set.seed(42)
out_dir <- "./data/"
save_dir <- "./data/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

## =========================================================================
## 第一部分：读取已注释数据，只取 5-HT 细胞
## =========================================================================

own <- readRDS("./data/ht_recluster/ht_5ht_nFeature1500_reclustered_named.rds")
own[["RNA"]] <- JoinLayers(own[["RNA"]])

mcq <- subset(own, subset = cell_type_new %in%
                grep("^5-HT_", unique(own$cell_type_new), value = TRUE))
mcq$renew_type <- as.character(mcq$cell_type_new)

cat("--- macaque 5-HT 亚型构成 ---\n")
print(table(mcq$renew_type, mcq$donor))

## =========================================================================
## 第二部分：跨物种整合（macaque × mouse，CCA，★ 全量细胞不抽样）
## =========================================================================

PAR <- list(
  variable_genes = 3000,
  npcs           = 100,
  cca_dims       = 1:80,
  nb_dims        = 1:30,
  resolution     = 0.1,
  min_cells      = 100,
  k_anchor       = 5
)

## ---- 2.1 读取数据 + 清理 ----
macaque <- mcq
DefaultAssay(macaque) <- "RNA"
macaque <- DietSeurat(macaque, assays = "RNA", dimreducs = FALSE, graphs = FALSE)
macaque[["RNA"]] <- JoinLayers(macaque[["RNA"]])
macaque$donor <- as.character(macaque$donor)
macaque$maxPredClass <- "5-HT"
Idents(macaque) <- "renew_type"

mouse <- readRDS("./data/mouse_sero.combined_res0.5.rds")
mouse$maxPredClass <- "5-HT"
mouse$donor <- mouse$group
Idents(mouse) <- "renew_type"

## ---- 2.2 同源基因转换 ----
test <- read.table('./data/mart_export.txt',
                   sep = "\t", header = TRUE)
comGeneTb <- test[c("Gene.name", "Mouse.gene.name")]
colnames(comGeneTb) <- c('macaqueGene', 'mouseGene')
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$macaqueGene), ]
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$mouseGene), ]
comGeneTb <- comGeneTb[!duplicated(comGeneTb$macaqueGene), ]

prepareHomoSeurat <- function(seurat, originalGeneName, homoGeneName,
                              keepMetaCol, commonMetaCol,
                              speciesPrefix = "macaque_", clusterCol = "renew_type") {
  homoCountMx <- GetAssayData(seurat[["RNA"]], slot = "counts")[originalGeneName, ]
  rownames(homoCountMx) <- homoGeneName
  colnames(homoCountMx) <- paste0(speciesPrefix, colnames(homoCountMx))
  homoSeurat <- CreateSeuratObject(counts = homoCountMx)
  tmpDf <- seurat@meta.data[sub(paste0("^", speciesPrefix), "",
                                colnames(homoSeurat)), keepMetaCol]
  tmpDf$species <- sub("(_|-)", "", speciesPrefix)
  tmpDf$clusterName <- paste0(speciesPrefix, tmpDf[[clusterCol]])
  commonMetaCol <- c(commonMetaCol, "species", "clusterName")
  colnames(tmpDf) <- ifelse(colnames(tmpDf) %in% commonMetaCol,
                            colnames(tmpDf),
                            paste0(speciesPrefix, colnames(tmpDf)))
  rownames(tmpDf) <- paste0(speciesPrefix, rownames(tmpDf))
  homoSeurat <- AddMetaData(homoSeurat, metadata = tmpDf)
  return(homoSeurat)
}

comGeneTb$homoGeneSymbol <- comGeneTb$macaqueGene
comGeneTb <- subset(comGeneTb,
  macaqueGene %in% rownames(GetAssayData(macaque[["RNA"]], slot = "counts")) &
  mouseGene   %in% rownames(GetAssayData(mouse[["RNA"]], slot = "counts")))
message("一对一同源基因: ", nrow(comGeneTb))

macaqueHomoSeurat <- prepareHomoSeurat(
  macaque, comGeneTb$macaqueGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "macaque_"
)
mouseHomoSeurat <- prepareHomoSeurat(
  mouse, comGeneTb$mouseGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "group", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "mouse_"
)
mouseHomoSeurat$species   <- "mouse"
macaqueHomoSeurat$species <- "macaque"

## ---- 2.3 batch 标记（★ 不抽平，全量保留）----
own_donors <- c("Mq246", "Mq277")
macaqueHomoSeurat$batch <- ifelse(macaqueHomoSeurat$donor %in% own_donors,
                                  macaqueHomoSeurat$donor, "NHP_published")
mouseHomoSeurat$batch <- mouseHomoSeurat$donor

cat("★ 全量细胞（不抽样）:\n")
print(table(macaqueHomoSeurat$batch))
print(table(mouseHomoSeurat$batch))
cat("macaque 总:", ncol(macaqueHomoSeurat),
    " mouse 总:", ncol(mouseHomoSeurat),
    " 合计:", ncol(macaqueHomoSeurat) + ncol(mouseHomoSeurat), "\n")

## ---- 2.4 split + merge ----
mouseHomoSeurat[["RNA"]]   <- split(mouseHomoSeurat[["RNA"]],   f = mouseHomoSeurat$batch)
macaqueHomoSeurat[["RNA"]] <- split(macaqueHomoSeurat[["RNA"]], f = macaqueHomoSeurat$batch)
classFiltSeurat <- merge(mouseHomoSeurat, macaqueHomoSeurat)
print(sapply(Layers(classFiltSeurat[["RNA"]]),
             function(l) ncol(LayerData(classFiltSeurat[["RNA"]], layer = l))))

## ---- 2.5 SCT + PCA + CCA ----
classFiltSeurat <- SCTransform(classFiltSeurat, assay = "RNA",
                               ncells = ncol(classFiltSeurat[["RNA"]]),
                               variable.features.n = PAR$variable_genes,
                               method = "glmGamPoi")
classFiltSeurat <- RunPCA(classFiltSeurat, npcs = PAR$npcs)
classFiltSeurat <- IntegrateLayers(classFiltSeurat,
                                   method = CCAIntegration,
                                   orig.reduction = "pca",
                                   new.reduction = "integrated.cca",
                                   normalization.method = "SCT",
                                   dims = PAR$cca_dims, verbose = TRUE)

## ---- 2.6 聚类 + UMAP ----
classFiltSeurat <- FindNeighbors(classFiltSeurat, reduction = "integrated.cca",
                                 dims = PAR$nb_dims)
classFiltSeurat <- FindClusters(classFiltSeurat, resolution = PAR$resolution)
classFiltSeurat <- RunUMAP(classFiltSeurat, reduction = "integrated.cca",
                           dims = PAR$nb_dims, reduction.name = "umap")

## ---- 2.7 可视化 ----
library(ggplot2)
colorLeg  <- c(macaque = "#EB6368", mouse = "#82ADD0")
batchCols <- c("mouse1" = "#82ADD0", "mouse2" = "#6A93B8",
               "Mq246" = "#EB6368", "Mq277" = "#D14E5B",
               "NHP_published" = "#F5B971")

p1 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "species",
              cols = colorLeg, raster = FALSE, pt.size = 1.0) +
  coord_fixed() + ggtitle("5-HT cross-species (all cells, CCA dims80)")
print(p1)
ggsave(paste0(save_dir, "UMAP_CCA_allcells_species.pdf"), p1, width = 6, height = 5)

p2 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "batch",
              cols = batchCols, pt.size = 1.0) +
  coord_fixed() + ggtitle("batch check (all cells)")
print(p2)
ggsave(paste0(save_dir, "UMAP_CCA_allcells_batch.pdf"), p2, width = 6.5, height = 5)

mac_labels <- setNames(paste0("mac_", macaque$renew_type), colnames(macaque))
mac_cells_cf <- colnames(classFiltSeurat)[classFiltSeurat$species == "macaque"]
mac_stripped <- sub("^macaque_", "", mac_cells_cf)

classFiltSeurat$subtype_recl <- NA
classFiltSeurat$subtype_recl[classFiltSeurat$species == "macaque"] <- mac_labels[mac_stripped]
classFiltSeurat$subtype_recl[classFiltSeurat$species == "mouse"] <-
  sub("^mouse_", "", as.character(classFiltSeurat$clusterName[classFiltSeurat$species == "mouse"]))

n_missing <- sum(is.na(classFiltSeurat$subtype_recl))
cat("标签缺失细胞数:", n_missing, "/", ncol(classFiltSeurat), "\n")
classFiltSeurat$subtype_recl[is.na(classFiltSeurat$subtype_recl)] <- "Unassigned"
classFiltSeurat$subtype_recl <- factor(classFiltSeurat$subtype_recl)

print(table(classFiltSeurat$subtype_recl, classFiltSeurat$species))
write.csv(as.matrix(table(classFiltSeurat$subtype_recl, classFiltSeurat$species)),
          paste0(save_dir, "subtype_recl_vs_species_allcells.csv"))

all_types <- levels(classFiltSeurat$subtype_recl)
mac_types_lbl <- grep("^mac_", all_types, value = TRUE)
mus_types_lbl <- setdiff(all_types, mac_types_lbl)
subtype_cols <- c(
  setNames(colorRampPalette(c("#F4A0A4", "#B2182B"))(max(length(mac_types_lbl), 2))[seq_along(mac_types_lbl)], mac_types_lbl),
  setNames(colorRampPalette(c("#9ECAE1", "#08519C"))(max(length(mus_types_lbl), 2))[seq_along(mus_types_lbl)], mus_types_lbl)
)
legend_theme <- theme(legend.position = "right",
                      legend.title = element_text(size = 10),
                      legend.text = element_text(size = 8),
                      legend.key.size = unit(0.4, "cm"))
legend_guide <- guides(color = guide_legend(title = "Subtype", ncol = 1,
                                            override.aes = list(size = 3)))

p3 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              split.by = "species", label = TRUE, repel = TRUE,
              label.size = 3.5, ncol = 2, raster = FALSE, pt.size = 1.2) +
  coord_fixed() + scale_color_manual(values = subtype_cols) +
  ggtitle("subtypes split (all cells)") +
  legend_theme + legend_guide
print(p3)
ggsave(paste0(save_dir, "UMAP_CCA_allcells_subtype_split.pdf"),
       p3, width = 15, height = 6.5)

p4 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              label = TRUE, repel = TRUE, label.size = 3, raster = FALSE,
              pt.size = 0.6) +
  coord_fixed() + scale_color_manual(values = subtype_cols) +
  legend_theme + legend_guide
print(p4)
ggsave(paste0(save_dir, "UMAP_CCA_allcells_subtype_all.pdf"),
       p4, width = 10, height = 6.5)

## ---- 2.8 Overlap 热图 ----
suppressMessages({ library(ComplexHeatmap); library(circlize); library(grid) })

compare_cl <- function(cl, ref.cl) {
  conf1 <- table(cl, ref.cl)
  conf1 <- sweep(conf1, 1, rowSums(conf1), "/")
  cl.prop.cocl <- apply(conf1, 2, function(x) apply(expand.grid(x, x), 1, min))
  cl.prop.cocl.m <- matrix(apply(cl.prop.cocl, 1, sum), nrow(conf1), nrow(conf1),
                           dimnames = list(rownames(conf1), rownames(conf1)))
  return(list(conf = conf1, cocl = cl.prop.cocl.m))
}

cl.conf <- compare_cl(classFiltSeurat$subtype_recl, classFiltSeurat$seurat_clusters)
cocl.subset <- cl.conf$cocl[grepl("^mac_",    rownames(cl.conf$cocl)),
                            grepl("^5-HT",    colnames(cl.conf$cocl))]
cat("Overlap 矩阵:", nrow(cocl.subset), "x", ncol(cocl.subset), "\n")

row_max <- apply(cocl.subset, 1, max)
bar_colors <- ifelse(row_max > 0.3, "#D6251F", "#3778AE")
left_anno <- rowAnnotation("Max Overlap\nscore" = anno_barplot(
  row_max, gp = gpar(fill = bar_colors, col = NA), bar_width = 0.9,
  border = FALSE, axis_param = list(direction = "reverse")), width = unit(3, "cm"))
bar_legend <- Legend(title = "Type",
                     at = c("macaque-specific", "macaque/mouse-conserved"),
                     legend_gp = gpar(fill = c("#3778AE", "#D6251F")),
                     border = "black", labels_gp = gpar(fontsize = 10))

ht <- Heatmap(cocl.subset, name = "Overlap score",
              col = colorRampPalette(c("white", "grey70", "black"))(100),
              cluster_rows = FALSE, cluster_columns = FALSE,
              left_annotation = left_anno, row_names_side = "right",
              row_names_gp = gpar(fontsize = 9), column_names_gp = gpar(fontsize = 9),
              width = unit(6, "cm"), height = unit(8, "cm"))

png(paste0(save_dir, "overlap_cca_allcells_mac5HT.png"),
    width = 8.5, height = 8, units = "in", res = 300)
draw(ht, annotation_legend_side = "bottom", heatmap_legend_list = list(bar_legend))
decorate_annotation("Max Overlap\nscore", {
  n <- length(row_max)
  for (j in seq_len(n))
    grid.text(sprintf("%.2f", row_max[j]), x = unit(row_max[j], "native"),
              y = unit(1 - (j - 0.5)/n, "npc"), just = "right", gp = gpar(fontsize = 8))
})
decorate_annotation("Max Overlap\nscore", {
  grid.lines(x = unit(0.7, "native"), y = unit(c(0,1), "npc"), gp = gpar(lty = 2, col = "black"))
})
dev.off()

write.csv(cocl.subset, paste0(save_dir, "overlap_cca_allcells_mac5HT.csv"))
cat("Overlap 热图已保存\n")

# ## ---- 2.9 保存整合对象（★ 全量版很大，务必存）----
# saveRDS(classFiltSeurat, paste0(save_dir, "cross_species_5HT_allcells_cca.rds"))
# cat("整合对象已保存\n")

In [ ]:
suppressMessages({
  library(tidyverse)
  library(Seurat)
  library(patchwork)
  library(Matrix)
})

set.seed(42)
out_dir <- "./data/"
save_dir <- "./data/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

## =========================================================================
## 第一部分：读取已注释数据，只取 5-HT 细胞
## =========================================================================

own <- readRDS("./data/classFiltSeurat_filt_allSubtypes.rds")
own[["RNA"]] <- JoinLayers(own[["RNA"]])

mcq <- subset(own, subset = cell_type_new %in%
                grep("^5-HT_", unique(own$cell_type_new), value = TRUE))
mcq$renew_type <- as.character(mcq$cell_type_new)

cat("--- macaque 5-HT 亚型构成 ---\n")
print(table(mcq$renew_type, mcq$donor))

## =========================================================================
## 第二部分：跨物种整合（macaque × mouse，CCA，batch 抽平）
## =========================================================================

PAR <- list(
  variable_genes = 3000,
  npcs           = 100,
  cca_dims       = 1:80,
  nb_dims        = 1:30,
  resolution     = 0.1,
  min_cells      = 100,
  k_anchor       = 5
)

## ---- 2.1 读取数据 + 清理 ----
macaque <- mcq
DefaultAssay(macaque) <- "RNA"
macaque <- DietSeurat(macaque, assays = "RNA", dimreducs = FALSE, graphs = FALSE)
macaque[["RNA"]] <- JoinLayers(macaque[["RNA"]])
macaque$donor <- as.character(macaque$donor)
macaque$maxPredClass <- "5-HT"
Idents(macaque) <- "renew_type"

mouse <- readRDS("./data/mouse_sero.combined_res0.5.rds")
mouse$maxPredClass <- "5-HT"
mouse$donor <- mouse$group
Idents(mouse) <- "renew_type"

## ---- 2.2 同源基因转换 ----
test <- read.table('./data/mart_export.txt',
                   sep = "\t", header = TRUE)
comGeneTb <- test[c("Gene.name", "Mouse.gene.name")]
colnames(comGeneTb) <- c('macaqueGene', 'mouseGene')
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$macaqueGene), ]
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$mouseGene), ]
comGeneTb <- comGeneTb[!duplicated(comGeneTb$macaqueGene), ]

prepareHomoSeurat <- function(seurat, originalGeneName, homoGeneName,
                              keepMetaCol, commonMetaCol,
                              speciesPrefix = "macaque_", clusterCol = "renew_type") {
  homoCountMx <- GetAssayData(seurat[["RNA"]], slot = "counts")[originalGeneName, ]
  rownames(homoCountMx) <- homoGeneName
  colnames(homoCountMx) <- paste0(speciesPrefix, colnames(homoCountMx))
  homoSeurat <- CreateSeuratObject(counts = homoCountMx)
  tmpDf <- seurat@meta.data[sub(paste0("^", speciesPrefix), "",
                                colnames(homoSeurat)), keepMetaCol]
  tmpDf$species <- sub("(_|-)", "", speciesPrefix)
  tmpDf$clusterName <- paste0(speciesPrefix, tmpDf[[clusterCol]])
  commonMetaCol <- c(commonMetaCol, "species", "clusterName")
  colnames(tmpDf) <- ifelse(colnames(tmpDf) %in% commonMetaCol,
                            colnames(tmpDf),
                            paste0(speciesPrefix, colnames(tmpDf)))
  rownames(tmpDf) <- paste0(speciesPrefix, rownames(tmpDf))
  homoSeurat <- AddMetaData(homoSeurat, metadata = tmpDf)
  return(homoSeurat)
}

comGeneTb$homoGeneSymbol <- comGeneTb$macaqueGene
comGeneTb <- subset(comGeneTb,
  macaqueGene %in% rownames(GetAssayData(macaque[["RNA"]], slot = "counts")) &
  mouseGene   %in% rownames(GetAssayData(mouse[["RNA"]], slot = "counts")))
message("一对一同源基因: ", nrow(comGeneTb))

macaqueHomoSeurat <- prepareHomoSeurat(
  macaque, comGeneTb$macaqueGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "macaque_"
)
mouseHomoSeurat <- prepareHomoSeurat(
  mouse, comGeneTb$mouseGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "group", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "mouse_"
)
mouseHomoSeurat$species   <- "mouse"
macaqueHomoSeurat$species <- "macaque"

## ---- 2.3 合并小 donor + 每 batch 抽平 ----
own_donors <- c("Mq246", "Mq277")
macaqueHomoSeurat$batch <- ifelse(macaqueHomoSeurat$donor %in% own_donors,
                                  macaqueHomoSeurat$donor, "NHP_published")
mouseHomoSeurat$batch <- mouseHomoSeurat$donor

target_per_batch <- min(c(table(macaqueHomoSeurat$batch),
                          table(mouseHomoSeurat$batch)))
message("每个 batch 抽平到 ", target_per_batch)

downsample_by_batch <- function(obj, target_per_batch) {
  ct <- obj$clusterName; bt <- obj$batch
  cells <- unlist(lapply(split(seq_len(ncol(obj)), bt), function(idx) {
    n_target <- min(length(idx), target_per_batch)
    tab <- table(ct[idx])
    n_per <- round(tab / sum(tab) * n_target)
    n_per <- pmin(n_per, tab)
    n_per[names(which.max(n_per))] <- n_per[names(which.max(n_per))] + (n_target - sum(n_per))
    unlist(lapply(names(n_per), function(x) {
      cells_x <- colnames(obj)[idx][ct[idx] == x]
      sample(cells_x, min(n_per[x], length(cells_x)))
    }))
  }))
  sample(cells, min(length(cells), sum(pmin(table(bt), target_per_batch))))
}
macaqueHomoSeurat <- subset(macaqueHomoSeurat,
                            cells = downsample_by_batch(macaqueHomoSeurat, target_per_batch))
mouseHomoSeurat   <- subset(mouseHomoSeurat,
                            cells = downsample_by_batch(mouseHomoSeurat, target_per_batch))
print(table(macaqueHomoSeurat$batch)); print(table(mouseHomoSeurat$batch))

## ---- 2.4 split + merge ----
mouseHomoSeurat[["RNA"]]   <- split(mouseHomoSeurat[["RNA"]],   f = mouseHomoSeurat$batch)
macaqueHomoSeurat[["RNA"]] <- split(macaqueHomoSeurat[["RNA"]], f = macaqueHomoSeurat$batch)
classFiltSeurat <- merge(mouseHomoSeurat, macaqueHomoSeurat)
print(sapply(Layers(classFiltSeurat[["RNA"]]),
             function(l) ncol(LayerData(classFiltSeurat[["RNA"]], layer = l))))

## ---- 2.5 SCT + PCA + CCA ----
classFiltSeurat <- SCTransform(classFiltSeurat, assay = "RNA",
                               ncells = ncol(classFiltSeurat[["RNA"]]),
                               variable.features.n = PAR$variable_genes,
                               method = "glmGamPoi")
classFiltSeurat <- RunPCA(classFiltSeurat, npcs = PAR$npcs)
classFiltSeurat <- IntegrateLayers(classFiltSeurat,
                                   method = CCAIntegration,
                                   orig.reduction = "pca",
                                   new.reduction = "integrated.cca",
                                   normalization.method = "SCT",
                                   dims = PAR$cca_dims, verbose = TRUE)

## ---- 2.6 聚类 + UMAP ----
classFiltSeurat <- FindNeighbors(classFiltSeurat, reduction = "integrated.cca",
                                 dims = PAR$nb_dims)
classFiltSeurat <- FindClusters(classFiltSeurat, resolution = PAR$resolution)
classFiltSeurat <- RunUMAP(classFiltSeurat, reduction = "integrated.cca",
                           dims = PAR$nb_dims, reduction.name = "umap")

## ---- 2.7 可视化（★ pt.size 已再调大） ----
library(ggplot2)
colorLeg  <- c(macaque = "#EB6368", mouse = "#82ADD0")
batchCols <- c("mouse1" = "#82ADD0", "mouse2" = "#6A93B8",
               "Mq246" = "#EB6368", "Mq277" = "#D14E5B",
               "NHP_published" = "#F5B971")

p1 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "species",
              cols = colorLeg, raster = FALSE,
              pt.size = 1.0) +
  coord_fixed() + ggtitle("5-HT cross-species (own macaque, CCA dims50)")
print(p1)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_species1.pdf"), p1, width = 6, height = 5)

p2 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "batch",
              cols = batchCols,
              pt.size = 1.0) +
  coord_fixed() + ggtitle("batch check")
print(p2)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_batch1.pdf"), p2, width = 6.5, height = 5)

mac_labels <- setNames(paste0("mac_", macaque$renew_type), colnames(macaque))
mac_cells_cf <- colnames(classFiltSeurat)[classFiltSeurat$species == "macaque"]
mac_stripped <- sub("^macaque_", "", mac_cells_cf)

classFiltSeurat$subtype_recl <- NA
classFiltSeurat$subtype_recl[classFiltSeurat$species == "macaque"] <- mac_labels[mac_stripped]
classFiltSeurat$subtype_recl[classFiltSeurat$species == "mouse"] <-
  sub("^mouse_", "", as.character(classFiltSeurat$clusterName[classFiltSeurat$species == "mouse"]))

n_missing <- sum(is.na(classFiltSeurat$subtype_recl))
cat("标签缺失细胞数:", n_missing, "/", ncol(classFiltSeurat), "\n")
classFiltSeurat$subtype_recl[is.na(classFiltSeurat$subtype_recl)] <- "Unassigned"
classFiltSeurat$subtype_recl <- factor(classFiltSeurat$subtype_recl)

print(table(classFiltSeurat$subtype_recl, classFiltSeurat$species))
write.csv(as.matrix(table(classFiltSeurat$subtype_recl, classFiltSeurat$species)),
          paste0(save_dir, "subtype_recl_vs_species.csv"))

all_types <- levels(classFiltSeurat$subtype_recl)
mac_types_lbl <- grep("^mac_", all_types, value = TRUE)
mus_types_lbl <- setdiff(all_types, mac_types_lbl)
subtype_cols <- c(
  setNames(colorRampPalette(c("#F4A0A4", "#B2182B"))(max(length(mac_types_lbl), 2))[seq_along(mac_types_lbl)], mac_types_lbl),
  setNames(colorRampPalette(c("#9ECAE1", "#08519C"))(max(length(mus_types_lbl), 2))[seq_along(mus_types_lbl)], mus_types_lbl)
)
legend_theme <- theme(legend.position = "right",
                      legend.title = element_text(size = 10),
                      legend.text = element_text(size = 8),
                      legend.key.size = unit(0.4, "cm"))
legend_guide <- guides(color = guide_legend(title = "Subtype", ncol = 1,
                                            override.aes = list(size = 3)))

p3 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              split.by = "species", label = TRUE, repel = TRUE,
              label.size = 3.5, ncol = 2, raster = FALSE,
              pt.size = 1.2) +
  coord_fixed() + scale_color_manual(values = subtype_cols) +
  ggtitle("subtypes split (macaque = annotated 5-HT)") +
  legend_theme + legend_guide
print(p3)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_subtype_split_legend1.pdf"),
       p3, width = 15, height = 6.5)

p4 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              label = TRUE, repel = TRUE, label.size = 3, raster = FALSE,
              pt.size = 0.6) +
  coord_fixed() + scale_color_manual(values = subtype_cols) +
  legend_theme + legend_guide
print(p4)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_subtype_all_legend1.pdf"),
       p4, width = 10, height = 6.5)

## ---- 2.8 Overlap 热图 ----
suppressMessages({ library(ComplexHeatmap); library(circlize); library(grid) })

compare_cl <- function(cl, ref.cl) {
  conf1 <- table(cl, ref.cl)
  conf1 <- sweep(conf1, 1, rowSums(conf1), "/")
  cl.prop.cocl <- apply(conf1, 2, function(x) apply(expand.grid(x, x), 1, min))
  cl.prop.cocl.m <- matrix(apply(cl.prop.cocl, 1, sum), nrow(conf1), nrow(conf1),
                           dimnames = list(rownames(conf1), rownames(conf1)))
  return(list(conf = conf1, cocl = cl.prop.cocl.m))
}

cl.conf <- compare_cl(classFiltSeurat$subtype_recl, classFiltSeurat$seurat_clusters)
cocl.subset <- cl.conf$cocl[grepl("^mac_",    rownames(cl.conf$cocl)),
                            grepl("^5-HT",    colnames(cl.conf$cocl))]
cat("Overlap 矩阵:", nrow(cocl.subset), "x", ncol(cocl.subset), "\n")

row_max <- apply(cocl.subset, 1, max)
bar_colors <- ifelse(row_max > 0.3, "#D6251F", "#3778AE")
left_anno <- rowAnnotation("Max Overlap\nscore" = anno_barplot(
  row_max, gp = gpar(fill = bar_colors, col = NA), bar_width = 0.9,
  border = FALSE, axis_param = list(direction = "reverse")), width = unit(3, "cm"))
bar_legend <- Legend(title = "Type",
                     at = c("macaque-specific", "macaque/mouse-conserved"),
                     legend_gp = gpar(fill = c("#3778AE", "#D6251F")),
                     border = "black", labels_gp = gpar(fontsize = 10))

ht <- Heatmap(cocl.subset, name = "Overlap score",
              col = colorRampPalette(c("white", "grey70", "black"))(100),
              cluster_rows = FALSE, cluster_columns = FALSE,
              left_annotation = left_anno, row_names_side = "right",
              row_names_gp = gpar(fontsize = 9), column_names_gp = gpar(fontsize = 9),
              width = unit(6, "cm"), height = unit(8, "cm"))

png(paste0(save_dir, "overlap_cca_dims50_mac5HT.png"),
    width = 8.5, height = 8, units = "in", res = 300)
draw(ht, annotation_legend_side = "bottom", heatmap_legend_list = list(bar_legend))
decorate_annotation("Max Overlap\nscore", {
  n <- length(row_max)
  for (j in seq_len(n))
    grid.text(sprintf("%.2f", row_max[j]), x = unit(row_max[j], "native"),
              y = unit(1 - (j - 0.5)/n, "npc"), just = "right", gp = gpar(fontsize = 8))
})
decorate_annotation("Max Overlap\nscore", {
  grid.lines(x = unit(0.7, "native"), y = unit(c(0,1), "npc"), gp = gpar(lty = 2, col = "black"))
})
dev.off()

write.csv(cocl.subset, paste0(save_dir, "overlap_cca_dims50_mac5HT1.csv"))
cat("Overlap 热图已保存\n")

In [ ]:
## ---- 2.7 可视化（★ pt.size 已再调大） ----
library(ggplot2)
colorLeg  <- c(macaque = "#EB6368", mouse = "#82ADD0")
batchCols <- c("mouse1" = "#82ADD0", "mouse2" = "#6A93B8",
               "Mq246" = "#EB6368", "Mq277" = "#D14E5B",
               "NHP_published" = "#F5B971")

p1 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "species",
              cols = colorLeg, raster = FALSE,
              pt.size = 1.5
            ) +
  coord_fixed() + ggtitle("5-HT cross-species (own macaque, CCA dims50)")
print(p1)


In [ ]:
suppressMessages({
  library(tidyverse)
  library(Seurat)
  library(patchwork)
  library(Matrix)
})

set.seed(42)
out_dir <- "./data/"
save_dir <- "./data/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)
## =========================================================================
## 第一部分：读取已注释数据，只取 5-HT 细胞（★ 不重聚类，直接用 cell_type_new）
## =========================================================================

own <- readRDS("./data/all_cells_reclustered.rds")
own[["RNA"]] <- JoinLayers(own[["RNA"]])

# 只取 5-HT 细胞，直接沿用已有注释
mcq <- subset(own, subset = cell_type_new %in%
                grep("^5-HT_", unique(own$cell_type_new), value = TRUE))
mcq$renew_type <- as.character(mcq$cell_type_new)

cat("--- macaque 5-HT 亚型构成 ---\n")
print(table(mcq$renew_type, mcq$donor))

## =========================================================================
## 第二部分：跨物种整合（macaque × mouse，CCA dims 1:50，batch 抽平）
## =========================================================================

PAR <- list(
  variable_genes = 8000,    # ★ 8000 → 10000：逼近一对一同源基因上限
  npcs           = 100,
  cca_dims       = 1:80,     # ★ 60 → 80：锚点相关空间再放宽
  nb_dims        = 1:30,     # ★ 50 → 40：建图维度收紧，滤掉 CCA 尾部噪声
  resolution     = 0.1,      # 不变（再强调：res 不影响整合，只影响聚类粒度）
  min_cells      = 100,
  k_anchor       = 5        # ★ 新增：锚点 k 值，默认 5 → 10
)

## ---- 2.1 读取数据 + 清理 ----
macaque <- mcq
DefaultAssay(macaque) <- "RNA"
macaque <- DietSeurat(macaque, assays = "RNA", dimreducs = FALSE, graphs = FALSE)
macaque[["RNA"]] <- JoinLayers(macaque[["RNA"]])
macaque$donor <- as.character(macaque$donor)
macaque$maxPredClass <- "5-HT"
Idents(macaque) <- "renew_type"

mouse <- readRDS("./data/mouse_sero.combined_res0.5.rds")
mouse$maxPredClass <- "5-HT"
mouse$donor <- mouse$group
Idents(mouse) <- "renew_type"

## ---- 2.2 同源基因转换（不变）----
test <- read.table('./data/mart_export.txt',
                   sep = "\t", header = TRUE)
comGeneTb <- test[c("Gene.name", "Mouse.gene.name")]
colnames(comGeneTb) <- c('macaqueGene', 'mouseGene')
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$macaqueGene), ]
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$mouseGene), ]
comGeneTb <- comGeneTb[!duplicated(comGeneTb$macaqueGene), ]

prepareHomoSeurat <- function(seurat, originalGeneName, homoGeneName,
                              keepMetaCol, commonMetaCol,
                              speciesPrefix = "macaque_", clusterCol = "renew_type") {
  homoCountMx <- GetAssayData(seurat[["RNA"]], slot = "counts")[originalGeneName, ]
  rownames(homoCountMx) <- homoGeneName
  colnames(homoCountMx) <- paste0(speciesPrefix, colnames(homoCountMx))
  homoSeurat <- CreateSeuratObject(counts = homoCountMx)
  tmpDf <- seurat@meta.data[sub(paste0("^", speciesPrefix), "",
                                colnames(homoSeurat)), keepMetaCol]
  tmpDf$species <- sub("(_|-)", "", speciesPrefix)
  tmpDf$clusterName <- paste0(speciesPrefix, tmpDf[[clusterCol]])
  commonMetaCol <- c(commonMetaCol, "species", "clusterName")
  colnames(tmpDf) <- ifelse(colnames(tmpDf) %in% commonMetaCol,
                            colnames(tmpDf),
                            paste0(speciesPrefix, colnames(tmpDf)))
  rownames(tmpDf) <- paste0(speciesPrefix, rownames(tmpDf))
  homoSeurat <- AddMetaData(homoSeurat, metadata = tmpDf)
  return(homoSeurat)
}

comGeneTb$homoGeneSymbol <- comGeneTb$macaqueGene
comGeneTb <- subset(comGeneTb,
  macaqueGene %in% rownames(GetAssayData(macaque[["RNA"]], slot = "counts")) &
  mouseGene   %in% rownames(GetAssayData(mouse[["RNA"]], slot = "counts")))
message("一对一同源基因: ", nrow(comGeneTb))

macaqueHomoSeurat <- prepareHomoSeurat(
  macaque, comGeneTb$macaqueGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "macaque_"
)
mouseHomoSeurat <- prepareHomoSeurat(
  mouse, comGeneTb$mouseGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "group", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "mouse_"
)
mouseHomoSeurat$species   <- "mouse"
macaqueHomoSeurat$species <- "macaque"

## ---- 2.3 合并小 donor + 每 batch 抽平（不变）----
own_donors <- c("Mq246", "Mq277")
macaqueHomoSeurat$batch <- ifelse(macaqueHomoSeurat$donor %in% own_donors,
                                  macaqueHomoSeurat$donor, "NHP_published")
mouseHomoSeurat$batch <- mouseHomoSeurat$donor

target_per_batch <- min(c(table(macaqueHomoSeurat$batch),
                          table(mouseHomoSeurat$batch)))
message("每个 batch 抽平到 ", target_per_batch)

downsample_by_batch <- function(obj, target_per_batch) {
  ct <- obj$clusterName; bt <- obj$batch
  cells <- unlist(lapply(split(seq_len(ncol(obj)), bt), function(idx) {
    n_target <- min(length(idx), target_per_batch)
    tab <- table(ct[idx])
    n_per <- round(tab / sum(tab) * n_target)
    n_per <- pmin(n_per, tab)
    n_per[names(which.max(n_per))] <- n_per[names(which.max(n_per))] + (n_target - sum(n_per))
    unlist(lapply(names(n_per), function(x) {
      cells_x <- colnames(obj)[idx][ct[idx] == x]
      sample(cells_x, min(n_per[x], length(cells_x)))
    }))
  }))
  sample(cells, min(length(cells), sum(pmin(table(bt), target_per_batch))))
}
macaqueHomoSeurat <- subset(macaqueHomoSeurat,
                            cells = downsample_by_batch(macaqueHomoSeurat, target_per_batch))
mouseHomoSeurat   <- subset(mouseHomoSeurat,
                            cells = downsample_by_batch(mouseHomoSeurat, target_per_batch))
print(table(macaqueHomoSeurat$batch)); print(table(mouseHomoSeurat$batch))

## ---- 2.4 split + merge（不变）----
mouseHomoSeurat[["RNA"]]   <- split(mouseHomoSeurat[["RNA"]],   f = mouseHomoSeurat$batch)
macaqueHomoSeurat[["RNA"]] <- split(macaqueHomoSeurat[["RNA"]], f = macaqueHomoSeurat$batch)
classFiltSeurat <- merge(mouseHomoSeurat, macaqueHomoSeurat)
print(sapply(Layers(classFiltSeurat[["RNA"]]),
             function(l) ncol(LayerData(classFiltSeurat[["RNA"]], layer = l))))

## ---- 2.5 SCT + PCA + CCA（dims 1:50）（不变）----
classFiltSeurat <- SCTransform(classFiltSeurat, assay = "RNA",
                               ncells = ncol(classFiltSeurat[["RNA"]]),
                               variable.features.n = PAR$variable_genes,
                               method = "glmGamPoi")
classFiltSeurat <- RunPCA(classFiltSeurat, npcs = PAR$npcs)
classFiltSeurat <- IntegrateLayers(classFiltSeurat,
                                   method = CCAIntegration,
                                   orig.reduction = "pca",
                                   new.reduction = "integrated.cca",
                                   normalization.method = "SCT",
                                   dims = PAR$cca_dims, verbose = TRUE)

## ---- 2.6 聚类 + UMAP（不变）----
classFiltSeurat <- FindNeighbors(classFiltSeurat, reduction = "integrated.cca",
                                 dims = PAR$nb_dims)
classFiltSeurat <- FindClusters(classFiltSeurat, resolution = PAR$resolution)
classFiltSeurat <- RunUMAP(classFiltSeurat, reduction = "integrated.cca",
                           dims = PAR$nb_dims, reduction.name = "umap")

## ---- 2.7 可视化 ----
library(ggplot2)
colorLeg  <- c(macaque = "#EB6368", mouse = "#82ADD0")
batchCols <- c("mouse1" = "#82ADD0", "mouse2" = "#6A93B8",
               "Mq246" = "#EB6368", "Mq277" = "#D14E5B",
               "NHP_published" = "#F5B971")

p1 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "species",
              cols = colorLeg, raster = FALSE) +
  coord_fixed() + ggtitle("5-HT cross-species (own macaque, CCA dims50)")
print(p1)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_species.pdf"), p1, width = 6, height = 5)

p2 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "batch",
              cols = batchCols) +
  coord_fixed() + ggtitle("batch check")
print(p2)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_batch.pdf"), p2, width = 6.5, height = 5)

# ★ 直接用已注释标签：macaque = mac_5-HT_x（加前缀和 mouse 区分），mouse = 5-HT_x
mac_labels <- setNames(paste0("mac_", macaque$renew_type), colnames(macaque))
mac_cells_cf <- colnames(classFiltSeurat)[classFiltSeurat$species == "macaque"]
mac_stripped <- sub("^macaque_", "", mac_cells_cf)

classFiltSeurat$subtype_recl <- NA
classFiltSeurat$subtype_recl[classFiltSeurat$species == "macaque"] <- mac_labels[mac_stripped]
classFiltSeurat$subtype_recl[classFiltSeurat$species == "mouse"] <-
  sub("^mouse_", "", as.character(classFiltSeurat$clusterName[classFiltSeurat$species == "mouse"]))

n_missing <- sum(is.na(classFiltSeurat$subtype_recl))
cat("标签缺失细胞数:", n_missing, "/", ncol(classFiltSeurat), "\n")
classFiltSeurat$subtype_recl[is.na(classFiltSeurat$subtype_recl)] <- "Unassigned"
classFiltSeurat$subtype_recl <- factor(classFiltSeurat$subtype_recl)

print(table(classFiltSeurat$subtype_recl, classFiltSeurat$species))
write.csv(as.matrix(table(classFiltSeurat$subtype_recl, classFiltSeurat$species)),
          paste0(save_dir, "subtype_recl_vs_species.csv"))

all_types <- levels(classFiltSeurat$subtype_recl)
mac_types_lbl <- grep("^mac_", all_types, value = TRUE)
mus_types_lbl <- setdiff(all_types, mac_types_lbl)
subtype_cols <- c(
  setNames(colorRampPalette(c("#F4A0A4", "#B2182B"))(max(length(mac_types_lbl), 2))[seq_along(mac_types_lbl)], mac_types_lbl),
  setNames(colorRampPalette(c("#9ECAE1", "#08519C"))(max(length(mus_types_lbl), 2))[seq_along(mus_types_lbl)], mus_types_lbl)
)
legend_theme <- theme(legend.position = "right",
                      legend.title = element_text(size = 10),
                      legend.text = element_text(size = 8),
                      legend.key.size = unit(0.4, "cm"))
legend_guide <- guides(color = guide_legend(title = "Subtype", ncol = 1,
                                            override.aes = list(size = 3)))

p3 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              split.by = "species", label = TRUE, repel = TRUE,
              label.size = 3.5, ncol = 2, raster = FALSE) +
  coord_fixed() + scale_color_manual(values = subtype_cols) +
  ggtitle("subtypes split (macaque = annotated 5-HT)") +
  legend_theme + legend_guide
print(p3)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_subtype_split_legend.pdf"),
       p3, width = 15, height = 6.5)

p4 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              label = TRUE, repel = TRUE, label.size = 3, raster = FALSE) +
  coord_fixed() + scale_color_manual(values = subtype_cols) +
  legend_theme + legend_guide
print(p4)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_subtype_all_legend.pdf"),
       p4, width = 10, height = 6.5)

## ---- 2.8 Overlap 热图（★ 行 = mac_5-HT_x，列 = mouse 5-HT_x）----
suppressMessages({ library(ComplexHeatmap); library(circlize); library(grid) })

compare_cl <- function(cl, ref.cl) {
  conf1 <- table(cl, ref.cl)
  conf1 <- sweep(conf1, 1, rowSums(conf1), "/")
  cl.prop.cocl <- apply(conf1, 2, function(x) apply(expand.grid(x, x), 1, min))
  cl.prop.cocl.m <- matrix(apply(cl.prop.cocl, 1, sum), nrow(conf1), nrow(conf1),
                           dimnames = list(rownames(conf1), rownames(conf1)))
  return(list(conf = conf1, cocl = cl.prop.cocl.m))
}

cl.conf <- compare_cl(classFiltSeurat$subtype_recl, classFiltSeurat$seurat_clusters)
cocl.subset <- cl.conf$cocl[grepl("^mac_",    rownames(cl.conf$cocl)),
                            grepl("^5-HT",    colnames(cl.conf$cocl))]
cat("Overlap 矩阵:", nrow(cocl.subset), "x", ncol(cocl.subset), "\n")

row_max <- apply(cocl.subset, 1, max)
bar_colors <- ifelse(row_max > 0.3, "#D6251F", "#3778AE")
left_anno <- rowAnnotation("Max Overlap\nscore" = anno_barplot(
  row_max, gp = gpar(fill = bar_colors, col = NA), bar_width = 0.9,
  border = FALSE, axis_param = list(direction = "reverse")), width = unit(3, "cm"))
bar_legend <- Legend(title = "Type",
                     at = c("macaque-specific", "macaque/mouse-conserved"),
                     legend_gp = gpar(fill = c("#3778AE", "#D6251F")),
                     border = "black", labels_gp = gpar(fontsize = 10))

ht <- Heatmap(cocl.subset, name = "Overlap score",
              col = colorRampPalette(c("white", "grey70", "black"))(100),
              cluster_rows = FALSE, cluster_columns = FALSE,
              left_annotation = left_anno, row_names_side = "right",
              row_names_gp = gpar(fontsize = 9), column_names_gp = gpar(fontsize = 9),
              width = unit(6, "cm"), height = unit(8, "cm"))

png(paste0(out_dir, "overlap_cca_dims50_mac5HT.png"),
    width = 8.5, height = 8, units = "in", res = 300)
draw(ht, annotation_legend_side = "bottom", heatmap_legend_list = list(bar_legend))
decorate_annotation("Max Overlap\nscore", {
  n <- length(row_max)
  for (j in seq_len(n))
    grid.text(sprintf("%.2f", row_max[j]), x = unit(row_max[j], "native"),
              y = unit(1 - (j - 0.5)/n, "npc"), just = "right", gp = gpar(fontsize = 8))
})
decorate_annotation("Max Overlap\nscore", {
  grid.lines(x = unit(0.7, "native"), y = unit(c(0,1), "npc"), gp = gpar(lty = 2, col = "black"))
})
dev.off()

write.csv(cocl.subset, paste0(out_dir, "overlap_cca_dims50_mac5HT.csv"))
cat("Overlap 热图已保存\n")

In [ ]:
suppressMessages({
  library(tidyverse)
  library(Seurat)
  library(patchwork)
  library(Matrix)
})

set.seed(42)
out_dir <- "./data/"
save_dir <- "./data/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)
## =========================================================================
## 第一部分：读取已注释数据，只取 5-HT 细胞（★ 不重聚类，直接用 cell_type_new）
## =========================================================================

own <- readRDS("./data/all_cells_reclustered.rds")
own[["RNA"]] <- JoinLayers(own[["RNA"]])

# 只取 5-HT 细胞，直接沿用已有注释
mcq <- subset(own, subset = cell_type_new %in%
                grep("^5-HT_", unique(own$cell_type_new), value = TRUE))
mcq$renew_type <- as.character(mcq$cell_type_new)

cat("--- macaque 5-HT 亚型构成 ---\n")
print(table(mcq$renew_type, mcq$donor))

## =========================================================================
## 第二部分：跨物种整合（macaque × mouse，CCA dims 1:50，batch 抽平）
## =========================================================================

PAR <- list(
  variable_genes = 5000,
  npcs           = 100,
  cca_dims       = 1:50,
  nb_dims        = 1:50,
  resolution     = 0.1,
  min_cells      = 100
)

## ---- 2.1 读取数据 + 清理 ----
macaque <- mcq
DefaultAssay(macaque) <- "RNA"
macaque <- DietSeurat(macaque, assays = "RNA", dimreducs = FALSE, graphs = FALSE)
macaque[["RNA"]] <- JoinLayers(macaque[["RNA"]])
macaque$donor <- as.character(macaque$donor)
macaque$maxPredClass <- "5-HT"
Idents(macaque) <- "renew_type"

mouse <- readRDS("./data/mouse_sero.combined_res0.5.rds")
mouse$maxPredClass <- "5-HT"
mouse$donor <- mouse$group
Idents(mouse) <- "renew_type"

## ---- 2.2 同源基因转换（不变）----
test <- read.table('./data/mart_export.txt',
                   sep = "\t", header = TRUE)
comGeneTb <- test[c("Gene.name", "Mouse.gene.name")]
colnames(comGeneTb) <- c('macaqueGene', 'mouseGene')
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$macaqueGene), ]
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$mouseGene), ]
comGeneTb <- comGeneTb[!duplicated(comGeneTb$macaqueGene), ]

prepareHomoSeurat <- function(seurat, originalGeneName, homoGeneName,
                              keepMetaCol, commonMetaCol,
                              speciesPrefix = "macaque_", clusterCol = "renew_type") {
  homoCountMx <- GetAssayData(seurat[["RNA"]], slot = "counts")[originalGeneName, ]
  rownames(homoCountMx) <- homoGeneName
  colnames(homoCountMx) <- paste0(speciesPrefix, colnames(homoCountMx))
  homoSeurat <- CreateSeuratObject(counts = homoCountMx)
  tmpDf <- seurat@meta.data[sub(paste0("^", speciesPrefix), "",
                                colnames(homoSeurat)), keepMetaCol]
  tmpDf$species <- sub("(_|-)", "", speciesPrefix)
  tmpDf$clusterName <- paste0(speciesPrefix, tmpDf[[clusterCol]])
  commonMetaCol <- c(commonMetaCol, "species", "clusterName")
  colnames(tmpDf) <- ifelse(colnames(tmpDf) %in% commonMetaCol,
                            colnames(tmpDf),
                            paste0(speciesPrefix, colnames(tmpDf)))
  rownames(tmpDf) <- paste0(speciesPrefix, rownames(tmpDf))
  homoSeurat <- AddMetaData(homoSeurat, metadata = tmpDf)
  return(homoSeurat)
}

comGeneTb$homoGeneSymbol <- comGeneTb$macaqueGene
comGeneTb <- subset(comGeneTb,
  macaqueGene %in% rownames(GetAssayData(macaque[["RNA"]], slot = "counts")) &
  mouseGene   %in% rownames(GetAssayData(mouse[["RNA"]], slot = "counts")))
message("一对一同源基因: ", nrow(comGeneTb))

macaqueHomoSeurat <- prepareHomoSeurat(
  macaque, comGeneTb$macaqueGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "macaque_"
)
mouseHomoSeurat <- prepareHomoSeurat(
  mouse, comGeneTb$mouseGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "group", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "mouse_"
)
mouseHomoSeurat$species   <- "mouse"
macaqueHomoSeurat$species <- "macaque"

## ---- 2.3 合并小 donor + 每 batch 抽平（不变）----
own_donors <- c("Mq246", "Mq277")
macaqueHomoSeurat$batch <- ifelse(macaqueHomoSeurat$donor %in% own_donors,
                                  macaqueHomoSeurat$donor, "NHP_published")
mouseHomoSeurat$batch <- mouseHomoSeurat$donor

target_per_batch <- min(c(table(macaqueHomoSeurat$batch),
                          table(mouseHomoSeurat$batch)))
message("每个 batch 抽平到 ", target_per_batch)

downsample_by_batch <- function(obj, target_per_batch) {
  ct <- obj$clusterName; bt <- obj$batch
  cells <- unlist(lapply(split(seq_len(ncol(obj)), bt), function(idx) {
    n_target <- min(length(idx), target_per_batch)
    tab <- table(ct[idx])
    n_per <- round(tab / sum(tab) * n_target)
    n_per <- pmin(n_per, tab)
    n_per[names(which.max(n_per))] <- n_per[names(which.max(n_per))] + (n_target - sum(n_per))
    unlist(lapply(names(n_per), function(x) {
      cells_x <- colnames(obj)[idx][ct[idx] == x]
      sample(cells_x, min(n_per[x], length(cells_x)))
    }))
  }))
  sample(cells, min(length(cells), sum(pmin(table(bt), target_per_batch))))
}
macaqueHomoSeurat <- subset(macaqueHomoSeurat,
                            cells = downsample_by_batch(macaqueHomoSeurat, target_per_batch))
mouseHomoSeurat   <- subset(mouseHomoSeurat,
                            cells = downsample_by_batch(mouseHomoSeurat, target_per_batch))
print(table(macaqueHomoSeurat$batch)); print(table(mouseHomoSeurat$batch))

## ---- 2.4 split + merge（不变）----
mouseHomoSeurat[["RNA"]]   <- split(mouseHomoSeurat[["RNA"]],   f = mouseHomoSeurat$batch)
macaqueHomoSeurat[["RNA"]] <- split(macaqueHomoSeurat[["RNA"]], f = macaqueHomoSeurat$batch)
classFiltSeurat <- merge(mouseHomoSeurat, macaqueHomoSeurat)
print(sapply(Layers(classFiltSeurat[["RNA"]]),
             function(l) ncol(LayerData(classFiltSeurat[["RNA"]], layer = l))))

## ---- 2.5 SCT + PCA + CCA（dims 1:50）（不变）----
classFiltSeurat <- SCTransform(classFiltSeurat, assay = "RNA",
                               ncells = ncol(classFiltSeurat[["RNA"]]),
                               variable.features.n = PAR$variable_genes,
                               method = "glmGamPoi")
classFiltSeurat <- RunPCA(classFiltSeurat, npcs = PAR$npcs)
classFiltSeurat <- IntegrateLayers(classFiltSeurat,
                                   method = CCAIntegration,
                                   orig.reduction = "pca",
                                   new.reduction = "integrated.cca",
                                   normalization.method = "SCT",
                                   dims = PAR$cca_dims, verbose = TRUE)

## ---- 2.6 聚类 + UMAP（不变）----
classFiltSeurat <- FindNeighbors(classFiltSeurat, reduction = "integrated.cca",
                                 dims = PAR$nb_dims)
classFiltSeurat <- FindClusters(classFiltSeurat, resolution = PAR$resolution)
classFiltSeurat <- RunUMAP(classFiltSeurat, reduction = "integrated.cca",
                           dims = PAR$nb_dims, reduction.name = "umap")

## ---- 2.7 可视化 ----
library(ggplot2)
colorLeg  <- c(macaque = "#EB6368", mouse = "#82ADD0")
batchCols <- c("mouse1" = "#82ADD0", "mouse2" = "#6A93B8",
               "Mq246" = "#EB6368", "Mq277" = "#D14E5B",
               "NHP_published" = "#F5B971")

p1 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "species",
              cols = colorLeg, raster = FALSE) +
  coord_fixed() + ggtitle("5-HT cross-species (own macaque, CCA dims50)")
print(p1)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_species.pdf"), p1, width = 6, height = 5)

p2 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "batch",
              cols = batchCols) +
  coord_fixed() + ggtitle("batch check")
print(p2)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_batch.pdf"), p2, width = 6.5, height = 5)

# ★ 直接用已注释标签：macaque = mac_5-HT_x（加前缀和 mouse 区分），mouse = 5-HT_x
mac_labels <- setNames(paste0("mac_", macaque$renew_type), colnames(macaque))
mac_cells_cf <- colnames(classFiltSeurat)[classFiltSeurat$species == "macaque"]
mac_stripped <- sub("^macaque_", "", mac_cells_cf)

classFiltSeurat$subtype_recl <- NA
classFiltSeurat$subtype_recl[classFiltSeurat$species == "macaque"] <- mac_labels[mac_stripped]
classFiltSeurat$subtype_recl[classFiltSeurat$species == "mouse"] <-
  sub("^mouse_", "", as.character(classFiltSeurat$clusterName[classFiltSeurat$species == "mouse"]))

n_missing <- sum(is.na(classFiltSeurat$subtype_recl))
cat("标签缺失细胞数:", n_missing, "/", ncol(classFiltSeurat), "\n")
classFiltSeurat$subtype_recl[is.na(classFiltSeurat$subtype_recl)] <- "Unassigned"
classFiltSeurat$subtype_recl <- factor(classFiltSeurat$subtype_recl)

print(table(classFiltSeurat$subtype_recl, classFiltSeurat$species))
write.csv(as.matrix(table(classFiltSeurat$subtype_recl, classFiltSeurat$species)),
          paste0(save_dir, "subtype_recl_vs_species.csv"))

all_types <- levels(classFiltSeurat$subtype_recl)
mac_types_lbl <- grep("^mac_", all_types, value = TRUE)
mus_types_lbl <- setdiff(all_types, mac_types_lbl)
subtype_cols <- c(
  setNames(colorRampPalette(c("#F4A0A4", "#B2182B"))(max(length(mac_types_lbl), 2))[seq_along(mac_types_lbl)], mac_types_lbl),
  setNames(colorRampPalette(c("#9ECAE1", "#08519C"))(max(length(mus_types_lbl), 2))[seq_along(mus_types_lbl)], mus_types_lbl)
)
legend_theme <- theme(legend.position = "right",
                      legend.title = element_text(size = 10),
                      legend.text = element_text(size = 8),
                      legend.key.size = unit(0.4, "cm"))
legend_guide <- guides(color = guide_legend(title = "Subtype", ncol = 1,
                                            override.aes = list(size = 3)))

p3 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              split.by = "species", label = TRUE, repel = TRUE,
              label.size = 3.5, ncol = 2, raster = FALSE) +
  coord_fixed() + scale_color_manual(values = subtype_cols) +
  ggtitle("subtypes split (macaque = annotated 5-HT)") +
  legend_theme + legend_guide
print(p3)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_subtype_split_legend.pdf"),
       p3, width = 15, height = 6.5)

p4 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              label = TRUE, repel = TRUE, label.size = 3, raster = FALSE) +
  coord_fixed() + scale_color_manual(values = subtype_cols) +
  legend_theme + legend_guide
print(p4)
ggsave(paste0(save_dir, "UMAP_CCA_dims50_subtype_all_legend.pdf"),
       p4, width = 10, height = 6.5)

## ---- 2.8 Overlap 热图（★ 行 = mac_5-HT_x，列 = mouse 5-HT_x）----
suppressMessages({ library(ComplexHeatmap); library(circlize); library(grid) })

compare_cl <- function(cl, ref.cl) {
  conf1 <- table(cl, ref.cl)
  conf1 <- sweep(conf1, 1, rowSums(conf1), "/")
  cl.prop.cocl <- apply(conf1, 2, function(x) apply(expand.grid(x, x), 1, min))
  cl.prop.cocl.m <- matrix(apply(cl.prop.cocl, 1, sum), nrow(conf1), nrow(conf1),
                           dimnames = list(rownames(conf1), rownames(conf1)))
  return(list(conf = conf1, cocl = cl.prop.cocl.m))
}

cl.conf <- compare_cl(classFiltSeurat$subtype_recl, classFiltSeurat$seurat_clusters)
cocl.subset <- cl.conf$cocl[grepl("^mac_",    rownames(cl.conf$cocl)),
                            grepl("^5-HT",    colnames(cl.conf$cocl))]
cat("Overlap 矩阵:", nrow(cocl.subset), "x", ncol(cocl.subset), "\n")

row_max <- apply(cocl.subset, 1, max)
bar_colors <- ifelse(row_max > 0.3, "#D6251F", "#3778AE")
left_anno <- rowAnnotation("Max Overlap\nscore" = anno_barplot(
  row_max, gp = gpar(fill = bar_colors, col = NA), bar_width = 0.9,
  border = FALSE, axis_param = list(direction = "reverse")), width = unit(3, "cm"))
bar_legend <- Legend(title = "Type",
                     at = c("macaque-specific", "macaque/mouse-conserved"),
                     legend_gp = gpar(fill = c("#3778AE", "#D6251F")),
                     border = "black", labels_gp = gpar(fontsize = 10))

ht <- Heatmap(cocl.subset, name = "Overlap score",
              col = colorRampPalette(c("white", "grey70", "black"))(100),
              cluster_rows = FALSE, cluster_columns = FALSE,
              left_annotation = left_anno, row_names_side = "right",
              row_names_gp = gpar(fontsize = 9), column_names_gp = gpar(fontsize = 9),
              width = unit(6, "cm"), height = unit(8, "cm"))

png(paste0(out_dir, "overlap_cca_dims50_mac5HT.png"),
    width = 8.5, height = 8, units = "in", res = 300)
draw(ht, annotation_legend_side = "bottom", heatmap_legend_list = list(bar_legend))
decorate_annotation("Max Overlap\nscore", {
  n <- length(row_max)
  for (j in seq_len(n))
    grid.text(sprintf("%.2f", row_max[j]), x = unit(row_max[j], "native"),
              y = unit(1 - (j - 0.5)/n, "npc"), just = "right", gp = gpar(fontsize = 8))
})
decorate_annotation("Max Overlap\nscore", {
  grid.lines(x = unit(0.7, "native"), y = unit(c(0,1), "npc"), gp = gpar(lty = 2, col = "black"))
})
dev.off()

write.csv(cocl.subset, paste0(out_dir, "overlap_cca_dims50_mac5HT.csv"))
cat("Overlap 热图已保存\n")

# ## ---- 2.9 保存 ----
# saveRDS(classFiltSeurat, paste0(out_dir, "crossSpecies_CCA_dims50_mac5HT.rds"))

In [ ]:
## ---- 2.9 保存 ----
saveRDS(classFiltSeurat, paste0(save_dir, "crossSpecies_CCA_dims50_mac5HT.rds"))
writeLines(capture.output(sessionInfo()),
           paste0(save_dir, "sessionInfo_CCA_dims50_mac5HT.txt"))
message("全部完成")

In [ ]:
# 桑基图

In [ ]:
save_dir <- "./data/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

In [ ]:
## =========================================================================
## 2.9 桑基图：macaque 原始亚型 → (kNN 映射) → mouse 亚型
## =========================================================================
suppressMessages({ library(ggalluvial); library(RANN); library(reshape2) })

# ---- 1. 在 integrated.cca 空间里做 kNN 标签转移 ----
mac_cells <- colnames(classFiltSeurat)[classFiltSeurat$species == "macaque"]
mus_cells <- colnames(classFiltSeurat)[classFiltSeurat$species == "mouse"]

emb <- Embeddings(classFiltSeurat, "integrated.cca")[, 1:50]
emb_mac <- emb[mac_cells, , drop = FALSE]
emb_mus <- emb[mus_cells, , drop = FALSE]

mus_label <- as.character(classFiltSeurat$subtype_recl[mus_cells])

k <- 10
nn <- nn2(emb_mus, emb_mac, k = k)          # 每个猴细胞的 10 个最近鼠细胞
pred_mouse <- vapply(seq_len(nrow(nn$nn.idx)), function(i) {
  labs <- mus_label[nn$nn.idx[i, ]]
  names(sort(table(labs), decreasing = TRUE))[1]
}, character(1))

# 每个猴细胞：原始亚型 + 预测鼠亚型
mac_lab <- as.character(classFiltSeurat$subtype_recl[mac_cells])

# ---- 2. 汇总成 macaque × mouse 计数矩阵（模仿你 melt(NV) 的方式）----
NV <- table(mac_lab, pred_mouse)

# ★ 按末尾数字排序：mac_5-HT_0..10、5-HT_0..n（避免字典序 10 排到 2 前面）
sort_by_num <- function(x) {
  x[order(as.numeric(sub(".*_(\\d+)$", "\\1", x)))]
}
NV <- NV[sort_by_num(rownames(NV)), sort_by_num(colnames(NV))]

mac_types <- rownames(NV)
mus_types <- colnames(NV)

df <- melt(NV, varnames = c("macaque", "mouse"), value.name = "n")
df$macaque <- factor(df$macaque, levels = mac_types)
df$mouse   <- factor(df$mouse,   levels = mus_types)

# 映射一致性（顺便打印，可当补充表）
map_rate <- df %>%
  group_by(macaque) %>%
  summarise(top_pred = mouse[which.max(n)],
            frac = max(n) / sum(n), .groups = "drop")
print(map_rate)
write.csv(map_rate, paste0(save_dir, "mac_to_mouse_knn_mapping.csv"),
          row.names = FALSE)

# ---- 3. 全量桑基图（MetaNeighbor 同款样式）----
p_sankey <- ggplot(df, aes(axis1 = macaque, axis2 = mouse, y = n)) +
  geom_alluvium(aes(fill = macaque), alpha = 0.75, width = 1/10,
                knot.pos = 0.4, curve_type = "cubic") +
  geom_stratum(width = 1/10, fill = "grey95", color = "grey60") +
  geom_text(stat = "stratum", aes(label = after_stat(stratum)), size = 3) +
  scale_x_continuous(breaks = 1:2,
                     labels = c("macaque\n(annotated)", "mouse\n(kNN predicted)"),
                     expand = c(0.12, 0.12)) +
  theme_void(base_size = 13) +
  theme(legend.position = "none",
        axis.text.x = element_text(size = 13, face = "bold")) +
  ggtitle("macaque → mouse 5-HT subtype mapping (kNN in CCA space)")

print(p_sankey)
ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_all.pdf"),
       p_sankey, width = 14, height = 7)
ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_all.png"),
       p_sankey, width = 14, height = 7, dpi = 300)

# ---- 4. 只看强对应（映射细胞数占比 > 30%），图更清爽 ----
df_strong <- df %>%
  group_by(macaque) %>%
  mutate(frac = n / sum(n)) %>%
  ungroup() %>%
  subset(frac > 0.3)

if (nrow(df_strong) > 0) {
  p_sankey_strong <- ggplot(df_strong,
                            aes(axis1 = macaque, axis2 = mouse, y = n)) +
    geom_alluvium(aes(fill = macaque), alpha = 0.8, width = 1/10,
                  knot.pos = 0.4, curve_type = "cubic") +
    geom_stratum(width = 1/10, fill = "grey95", color = "grey60") +
    geom_text(stat = "stratum", aes(label = after_stat(stratum)), size = 3.5) +
    scale_x_continuous(breaks = 1:2,
                       labels = c("macaque\n(annotated)", "mouse\n(kNN predicted)"),
                       expand = c(0.12, 0.12)) +
    theme_void(base_size = 13) +
    theme(legend.position = "none",
          axis.text.x = element_text(size = 13, face = "bold")) +
    ggtitle("macaque → mouse (kNN mapping, >30% of subtype)")
  print(p_sankey_strong)
  ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_strong.pdf"),
         p_sankey_strong, width = 12, height = 7)
  ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_strong.png"),
         p_sankey_strong, width = 12, height = 7, dpi = 300)
}

cat("桑基图已保存\n")

In [ ]:
# ---- 3. 全量桑基图 ----
p_sankey <- ggplot(df, aes(axis1 = macaque, axis2 = mouse, y = n)) +
  geom_alluvium(aes(fill = macaque), alpha = 0.75, width = 1/10,
                knot.pos = 0.4, curve_type = "cubic") +
  geom_stratum(width = 1/10, fill = "grey95", color = "grey60") +
  scale_x_continuous(breaks = 1:2,
                     labels = c("macaque\n(annotated)", "mouse\n(kNN predicted)"),
                     expand = c(0.12, 0.12)) +
  theme_void(base_size = 13) +
  theme(legend.position = "none",
        axis.text.x = element_text(size = 13, face = "bold")) +
  ggtitle("macaque → mouse 5-HT subtype mapping (kNN in CCA space)")

print(p_sankey)
ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_all.pdf"),
       p_sankey, width = 14, height = 7)
ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_all.png"),
       p_sankey, width = 14, height = 7, dpi = 300)

# ---- 4. 强对应桑基图 ----
df_strong <- df %>%
  group_by(macaque) %>%
  mutate(frac = n / sum(n)) %>%
  ungroup() %>%
  subset(frac > 0.3)

if (nrow(df_strong) > 0) {
  p_sankey_strong <- ggplot(df_strong,
                            aes(axis1 = macaque, axis2 = mouse, y = n)) +
    geom_alluvium(aes(fill = macaque), alpha = 0.8, width = 1/10,
                  knot.pos = 0.4, curve_type = "cubic") +
    geom_stratum(width = 1/10, fill = "grey95", color = "grey60") +
    scale_x_continuous(breaks = 1:2,
                       labels = c("macaque\n(annotated)", "mouse\n(kNN predicted)"),
                       expand = c(0.12, 0.12)) +
    theme_void(base_size = 13) +
    theme(legend.position = "none",
          axis.text.x = element_text(size = 13, face = "bold")) +
    ggtitle("macaque → mouse (kNN mapping, >30% of subtype)")
  print(p_sankey_strong)
  ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_strong.pdf"),
         p_sankey_strong, width = 12, height = 7)
  ggsave(paste0(save_dir, "Sankey_mac_to_mouse_knn_strong.png"),
         p_sankey_strong, width = 12, height = 7, dpi = 300)
}

In [ ]:
# 基因图

In [ ]:
install.packages("ggvenn")

In [ ]:
suppressMessages({
  library(tidyverse); library(Seurat); library(ggvenn)
})

save_dir <- "./data/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

# ---------- 0. 数据准备（不变） ----------
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)
classFiltSeurat$sub_show <- sub("^mac_", "", as.character(classFiltSeurat$subtype_recl))

keep_pat <- "^5-HT_[0-9]+$"
is_bad <- !grepl(keep_pat, classFiltSeurat$sub_show)
if (any(is_bad)) {
  cat("剔除非编号亚型细胞数:", sum(is_bad), fill = TRUE)
  classFiltSeurat <- subset(classFiltSeurat, cells = colnames(classFiltSeurat)[!is_bad])
}

cap_first <- function(x) paste0(toupper(substr(x, 1, 1)), substr(x, 2, nchar(x)))

# ---------- 1. 每个物种内分别找亚型 marker（不变） ----------
find_mk <- function(obj) {
  Idents(obj) <- "sub_show"
  mk <- FindAllMarkers(obj, assay = "RNA", only.pos = TRUE,
                       min.pct = 0.1, logfc.threshold = 0.1,
                       test.use = "wilcox", verbose = FALSE)
  if (is.null(mk) || nrow(mk) == 0 || !"p_val_adj" %in% colnames(mk)) {
    mk <- FindAllMarkers(obj, assay = "RNA", only.pos = TRUE,
                         min.pct = 0.05, logfc.threshold = 0,
                         test.use = "wilcox", verbose = FALSE)
  }
  stopifnot(!is.null(mk), nrow(mk) > 0, "p_val_adj" %in% colnames(mk))
  mk %>% filter(p_val_adj < 0.05) %>%
    mutate(cluster = as.character(cluster))
}

species_levels <- sort(unique(as.character(classFiltSeurat$species)))
species_levels <- c(intersect(c("macaque", "marmoset", "mouse"), species_levels),
                    setdiff(species_levels, c("macaque", "marmoset", "mouse")))
n_sp <- length(species_levels)

obj_list <- lapply(species_levels, function(sp)
  subset(classFiltSeurat, subset = species == sp))
names(obj_list) <- species_levels
mk_list <- lapply(obj_list, find_mk)



# ---------- 2. 韦恩图：各物种 marker 基因集合 ----------
gene_sets <- lapply(mk_list, function(mk) unique(mk$gene))
names(gene_sets) <- cap_first(species_levels)

all_genes <- unique(unlist(gene_sets))          # ★ 先定义全集
venn_df <- as.data.frame(                       # 再构建 0/1 矩阵
  sapply(gene_sets, function(g) all_genes %in% g)
)
rownames(venn_df) <- all_genes

## 计算每个区域的基因列表
region_genes <- list()
for (i in seq_len(nrow(venn_df))) {
  reg <- paste(names(venn_df)[as.logical(venn_df[i, ])], collapse = "&")
  region_genes[[reg]] <- c(region_genes[[reg]], rownames(venn_df)[i])
}

## 韦恩图主体
p_venn <- ggplot(venn_df) +
  geom_venn(aes(A = .data[[names(venn_df)[1]]],
                B = .data[[names(venn_df)[2]]],
                C = if (ncol(venn_df) >= 3) .data[[names(venn_df)[3]]] else NULL),
            fill_color = c("#EB6368", "#E8A33D", "#82ADD0")[seq_len(ncol(venn_df))],
            fill_alpha = 0.45, stroke_color = "white", stroke_size = 0.5,
            text_size = 5, set_name_size = 5) +
  coord_fixed() +
  theme_void()

ggsave(paste0(save_dir, "Fig_markerVennyouzi.pdf"), p_venn, width = 8, height = 8)
ggsave(paste0(save_dir, "Fig_markerVennyouzi.png"), p_venn, width = 8, height = 8, dpi = 300)

# ---------- 3. 导出各区域基因列表 ----------
region_summary <- data.frame(
  region = names(region_genes),
  n_genes = lengths(region_genes)
)
write.csv(region_summary, paste0(save_dir, "Table_markerVenn_regionsyouzi.csv"),
          row.names = FALSE)
for (reg in names(region_genes)) {
  safe_reg <- gsub("[^A-Za-z0-9]+", "_", reg)
  write.csv(data.frame(gene = region_genes[[reg]]),
            paste0(save_dir, "Genes_", safe_reg, ".csv"), row.names = FALSE)
}

cat("韦恩图已保存到", save_dir, fill = TRUE)
print(region_summary)

In [ ]:
suppressMessages({
  library(tidyverse); library(Seurat); library(ggvenn)
})

save_dir <- "./data/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

# ---------- 0. 数据准备 ----------
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)
classFiltSeurat$sub_show <- sub("^mac_", "", as.character(classFiltSeurat$subtype_recl))

## 只保留编号亚型（5-HT_0、5-HT_1 ...），去掉 5-HT_filtered 等
keep_pat <- "^5-HT_[0-9]+$"
is_bad <- !grepl(keep_pat, classFiltSeurat$sub_show)
if (any(is_bad)) {
  cat("剔除非编号亚型细胞数:", sum(is_bad), fill = TRUE)
  classFiltSeurat <- subset(classFiltSeurat, cells = colnames(classFiltSeurat)[!is_bad])
}

cap_first <- function(x) paste0(toupper(substr(x, 1, 1)), substr(x, 2, nchar(x)))

# ---------- 1. 每个物种内分别找亚型 marker ----------
find_mk <- function(obj) {
  Idents(obj) <- "sub_show"
  mk <- FindAllMarkers(obj, assay = "RNA", only.pos = TRUE,
                       min.pct = 0.1, logfc.threshold = 0.1,
                       test.use = "wilcox", verbose = FALSE)
  if (is.null(mk) || nrow(mk) == 0 || !"p_val_adj" %in% colnames(mk)) {
    mk <- FindAllMarkers(obj, assay = "RNA", only.pos = TRUE,
                         min.pct = 0.05, logfc.threshold = 0,
                         test.use = "wilcox", verbose = FALSE)
  }
  stopifnot(!is.null(mk), nrow(mk) > 0, "p_val_adj" %in% colnames(mk))
  mk %>% filter(p_val_adj < 0.05) %>%
    mutate(cluster = as.character(cluster))
}

species_levels <- sort(unique(as.character(classFiltSeurat$species)))
species_levels <- c(intersect(c("macaque", "marmoset", "mouse"), species_levels),
                    setdiff(species_levels, c("macaque", "marmoset", "mouse")))
n_sp <- length(species_levels)

obj_list <- lapply(species_levels, function(sp)
  subset(classFiltSeurat, subset = species == sp))
names(obj_list) <- species_levels
mk_list <- lapply(obj_list, find_mk)

# ---------- 2. 韦恩图：各物种 marker 基因集合 ----------
gene_sets <- lapply(mk_list, function(mk) unique(mk$gene))
names(gene_sets) <- cap_first(species_levels)

all_genes <- unique(unlist(gene_sets))          # 先定义全集
venn_df <- as.data.frame(                       # 再构建 0/1 矩阵
  sapply(gene_sets, function(g) all_genes %in% g)
)
rownames(venn_df) <- all_genes

## 计算每个区域的基因列表
region_genes <- list()
for (i in seq_len(nrow(venn_df))) {
  reg <- paste(names(venn_df)[as.logical(venn_df[i, ])], collapse = "&")
  region_genes[[reg]] <- c(region_genes[[reg]], rownames(venn_df)[i])
}

## 韦恩图主体（不显示区域数字，保留集合名）
p_venn <- ggplot(venn_df) +
  geom_venn(aes(A = .data[[names(venn_df)[1]]],
                B = .data[[names(venn_df)[2]]],
                C = if (ncol(venn_df) >= 3) .data[[names(venn_df)[3]]] else NULL),
            fill_color = c("#EB6368", "#E8A33D", "#82ADD0")[seq_len(ncol(venn_df))],
            fill_alpha = 0.45,
            stroke_color = "white", stroke_size = 0.5,
            text_size = 0,           # ★ 不显示区域内数字
            set_name_size = 5) +     # ★ 保留集合名（Macaque / Mouse）
  coord_fixed() +
  theme_void()

ggsave(paste0(save_dir, "Fig_markerVenn.pdf"), p_venn, width = 8, height = 8)
ggsave(paste0(save_dir, "Fig_markerVenn.png"), p_venn, width = 8, height = 8, dpi = 300)

# ---------- 3. 导出各区域基因列表 ----------
region_summary <- data.frame(
  region = names(region_genes),
  n_genes = lengths(region_genes)
)
write.csv(region_summary, paste0(save_dir, "Table_markerVenn_regions.csv"),
          row.names = FALSE)
for (reg in names(region_genes)) {
  safe_reg <- gsub("[^A-Za-z0-9]+", "_", reg)
  write.csv(data.frame(gene = region_genes[[reg]]),
            paste0(save_dir, "Genes_", safe_reg, ".csv"), row.names = FALSE)
}

cat("韦恩图已保存到", save_dir, fill = TRUE)
print(region_summary)

In [ ]:
suppressMessages({
  library(tidyverse); library(Seurat); library(patchwork)
})

save_dir <- "./data/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

# ---------- 0. 数据准备 ----------
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)
classFiltSeurat$sub_show <- sub("^mac_", "", as.character(classFiltSeurat$subtype_recl))

# ---------- 1. 跨物种 DEG（macaque vs mouse，全部 5-HT 细胞） ----------
cells_5ht <- colnames(classFiltSeurat)[grepl("^5-HT_", classFiltSeurat$sub_show)]
obj_5ht   <- subset(classFiltSeurat, cells = cells_5ht)
Idents(obj_5ht) <- "species"

deg <- FindMarkers(obj_5ht, ident.1 = "macaque", ident.2 = "mouse",
                   assay = "RNA", min.pct = 0.1, logfc.threshold = 0,
                   test.use = "wilcox", verbose = FALSE)

# ---------- 2. top30 上调 / 下调 ----------
top_up <- deg %>% filter(avg_log2FC > 0) %>%
  arrange(desc(avg_log2FC)) %>% head(200) %>% rownames()
top_dn <- deg %>% filter(avg_log2FC < 0) %>%
  arrange(avg_log2FC) %>% head(200) %>% rownames()

# ---------- 3. 取真实表达（每物种抽 3000 细胞加速画图） ----------
set.seed(1)
expr <- FetchData(obj_5ht, vars = c(top_up, top_dn, "species"))
expr_s <- expr %>%
  group_by(species) %>%
  slice_sample(n = 3000) %>%
  ungroup()

to_long <- function(genes) {
  expr_s %>%
    select(species, all_of(genes)) %>%
    pivot_longer(-species, names_to = "gene", values_to = "expr") %>%
    mutate(gene   = factor(gene, levels = genes),
           species = factor(species, levels = c("mouse", "macaque")))  # 左 mouse 右 macaque
}
long_up <- to_long(top_up)
long_dn <- to_long(top_dn)

# ---------- 4. 画图 ----------
sp_fill <- c("mouse" = "#82ADD0", "macaque" = "#EB6368")

vln_panel <- function(df, title) {
  ggplot(df, aes(species, expr, fill = species)) +
    geom_violin(scale = "width", linewidth = 0.1, color = NA, na.rm = TRUE) +
    geom_boxplot(width = 0.12, outlier.shape = NA, alpha = 0.6,
                 linewidth = 0.2, color = "grey20") +
    scale_fill_manual(values = sp_fill) +
    scale_x_discrete(drop = FALSE) +
    facet_wrap(~ gene, scales = "free_y", ncol = 6) +
    labs(title = title, x = NULL, y = "Normalized expression") +
    theme_bw(base_size = 9) +
    theme(legend.position = "none",
          strip.text = element_text(face = "italic", size = 8),
          axis.text.x  = element_text(size = 8),
          plot.title   = element_text(face = "bold", size = 12))
}

p_up <- vln_panel(long_up, "Top 200 up in macaque (vs mouse)")
p_dn <- vln_panel(long_dn, "Top 200 down in macaque (up in mouse)")

pdf(paste0(save_dir, "Fig_5HT_top50_up_down_violin.pdf"), width = 12, height = 18)
print(p_up / p_dn + plot_layout(heights = c(1, 1)))
dev.off()

png(paste0(save_dir, "Fig_5HT_top200_up_down_violin.png"),
    width = 12, height = 18, units = "in", res = 300)
print(p_up / p_dn + plot_layout(heights = c(1, 1)))
dev.off()

cat("小提琴图已保存到", save_dir, fill = TRUE)

In [ ]:
suppressMessages({
  library(tidyverse); library(Seurat); library(patchwork)
})

save_dir <- "./data/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

GENE <- "NXPH2"     # 换基因改这里

# ---------- 0. 数据准备 ----------
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)
classFiltSeurat$sub_show <- sub("^mac_", "", as.character(classFiltSeurat$subtype_recl))

cells_5ht <- colnames(classFiltSeurat)[grepl("^5-HT_", classFiltSeurat$sub_show)]
obj_5ht   <- subset(classFiltSeurat, cells = cells_5ht)
stopifnot(GENE %in% rownames(obj_5ht))

sort_by_num <- function(x) {
  n <- suppressWarnings(as.numeric(sub(".*_(\\d+)$", "\\1", x)))
  x[order(n, na.last = TRUE)]
}

# ---------- 1. 取表达量（data = log-normalized）和真实 count ----------
df <- FetchData(obj_5ht, vars = c(GENE, "species", "sub_show"))
colnames(df)[1] <- "expr"
cnt <- FetchData(obj_5ht, vars = GENE, layer = "counts")
df$count <- cnt[rownames(df), 1]
df$species  <- factor(as.character(df$species), levels = c("macaque", "mouse"))
df$sub_show <- factor(as.character(df$sub_show),
                      levels = sort_by_num(unique(as.character(df$sub_show))))
df$express  <- df$count > 0

# ---------- 2. 小提琴图：每个物种各自的小类 ----------
sp_fill <- c("macaque" = "#EB6368", "mouse" = "#82ADD0")

p_vln <- ggplot(df, aes(sub_show, expr, fill = species)) +
  geom_violin(scale = "width", linewidth = 0.1, color = NA) +
  scale_fill_manual(values = sp_fill) +
  facet_wrap(~ species, nrow = 1, scales = "free") +
  labs(title = paste0(GENE, " expression across 5-HT subclasses"),
       subtitle = "Raw UMI counts", x = NULL, y = "Raw count") +
  theme_classic(base_size = 11) +
  theme(legend.position = "none",
        strip.text = element_text(face = "bold", size = 12),
        axis.text.x = element_text(angle = 45, hjust = 1))

# ---------- 3. 表达比例：每个小类内表达细胞占比（物种间比较） ----------
pct_df <- df %>%
  group_by(species, sub_show) %>%
  summarise(pct = mean(express), n = n(), .groups = "drop")

## 每个共有小类做 Fisher 精确检验
subs_mac  <- sort_by_num(unique(as.character(df$sub_show[df$species == "macaque"])))
subs_mus  <- sort_by_num(unique(as.character(df$sub_show[df$species == "mouse"])))
subs_both <- intersect(subs_mac, subs_mus)

sub_stat <- lapply(subs_both, function(s) {
  d <- df[as.character(df$sub_show) == s, ]
  if (n_distinct(d$species) < 2) return(NULL)
  tb <- table(d$species, d$express)
  if (nrow(tb) < 2 || ncol(tb) < 2) return(NULL)
  ft <- fisher.test(tb)
  data.frame(sub_show = s,
             pct_mac = mean(d$express[d$species == "macaque"]),
             pct_mus = mean(d$express[d$species == "mouse"]),
             p_val = ft$p.value)
})
sub_stat <- bind_rows(sub_stat)
if (nrow(sub_stat) > 0) {
  sub_stat$p_adj <- p.adjust(sub_stat$p_val, method = "BH")
  sub_stat$star <- cut(sub_stat$p_adj,
                       breaks = c(-1, 0.001, 0.01, 0.05, 1),
                       labels = c("***", "**", "*", "ns"))
}

lab_df <- pct_df %>%
  left_join(sub_stat %>% select(sub_show, star), by = "sub_show") %>%
  mutate(lab = paste0(round(pct * 100), "%"))

p_pct <- ggplot(pct_df, aes(sub_show, pct, fill = species)) +
  geom_col(position = position_dodge(0.85), width = 0.75, color = NA) +
  geom_text(data = lab_df,
            aes(label = lab), position = position_dodge(0.85),
            vjust = -0.4, size = 3) +
  geom_text(data = sub_stat,
            aes(x = sub_show, y = max(pct_mac, pct_mus) + 0.12,
                label = star), inherit.aes = FALSE, size = 4) +
  scale_fill_manual(values = sp_fill) +
  scale_y_continuous(labels = scales::percent, expand = expansion(mult = c(0, 0.15))) +
  labs(title = paste0(GENE, ": fraction of expressing cells"),
       subtitle = "Fisher exact test, BH-adjusted", x = NULL, y = "% of cells") +
  theme_classic(base_size = 11) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))

# ---------- 4. 总体比例：全部 5-HT 细胞层面物种间比较 ----------
overall <- df %>%
  group_by(species) %>%
  summarise(pct = mean(express), n = n(), .groups = "drop")
tab_all <- table(df$species, df$express)
ft_all  <- fisher.test(tab_all)
p_all   <- chisq.test(tab_all)$p.value

p_ov <- ggplot(overall, aes(species, pct, fill = species)) +
  geom_col(width = 0.6, color = NA) +
  geom_text(aes(label = paste0(round(pct * 100, 1), "%\n(n=", n, ")")),
            vjust = -0.3, size = 3.5) +
  scale_fill_manual(values = sp_fill) +
  scale_y_continuous(labels = scales::percent, expand = expansion(mult = c(0, 0.15))) +
  labs(title = paste0(GENE, ": overall expression in 5-HT cells"),
       subtitle = paste0("Fisher p = ", signif(ft_all$p.value, 3),
                         " | chi-sq p = ", signif(p_all, 3)),
       x = NULL, y = "% of 5-HT cells") +
  theme_classic(base_size = 11) +
  theme(legend.position = "none")

# ---------- 5. 出图 + 统计表 ----------
p_all_fig <- (p_vln / p_pct) | p_ov
p_all_fig <- p_all_fig + plot_layout(widths = c(3, 1))

ggsave(paste0(save_dir, "Fig_", GENE, "_subclass_violin_pct.pdf"),
       p_all_fig, width = 14, height = 10)
ggsave(paste0(save_dir, "Fig_", GENE, "_subclass_violin_pct.png"),
       p_all_fig, width = 14, height = 10, dpi = 300)

write.csv(sub_stat, paste0(save_dir, "Table_", GENE, "_pct_by_subclass.csv"),
          row.names = FALSE)
write.csv(overall, paste0(save_dir, "Table_", GENE, "_pct_overall.csv"),
          row.names = FALSE)

cat("==== ", GENE, " 总体表达比例 ====", fill = TRUE)
print(overall)
cat("Fisher p =", ft_all$p.value, " | chi-sq p =", p_all, fill = TRUE)
cat("图和表已保存到", save_dir, fill = TRUE)

In [ ]:
suppressMessages({
  library(Seurat); library(dplyr)
  library(ggplot2); library(ggrepel)
})

save_dir <- "./data/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

# ---- 0. 数据准备 ----
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)
classFiltSeurat$sub_show <- sub("^mac_", "", as.character(classFiltSeurat$subtype_recl))

## 只保留 5-HT 细胞（不限亚型，全部合并）
## ★ 改用 cells = 子集化，绕开 subset 表达式的 FetchData 解析问题
cells_5ht <- colnames(classFiltSeurat)[grepl("^5-HT_", classFiltSeurat$sub_show)]
obj_5ht <- subset(classFiltSeurat, cells = cells_5ht)

## 检查细胞数和物种分布
print(table(obj_5ht$species, obj_5ht$sub_show))

# ---- 1. 所有 5-HT 细胞：macaque vs mouse ----
## avg_log2FC > 0 = 在 macaque 中上调；< 0 = 在 mouse 中上调
Idents(obj_5ht) <- "species"
deg <- FindMarkers(obj_5ht, ident.1 = "macaque", ident.2 = "mouse",
                   assay = "RNA", min.pct = 0.1, logfc.threshold = 0,
                   test.use = "wilcox", verbose = FALSE)
deg$gene <- rownames(deg)

# ---- 2. 火山图 ----
pick_spread <- function(sub, n = 5) {
  sub <- sub %>% arrange(p_val_adj)
  bind_rows(
    slice_head(sub, n = 2),
    sub %>% filter(!(gene %in% slice_head(sub, n = 2)$gene)) %>%
      mutate(rank_fc = rank(-abs(avg_log2FC))) %>%
      arrange(rank_fc) %>% slice_head(n = n - 2)
  )
}

df <- deg %>%
  mutate(y_raw = -log10(pmax(p_val_adj, 1e-300)),  # 防止 p=0 产生 Inf
         sig = case_when(y_raw > 50 & avg_log2FC >  1 ~ "up",
                         y_raw > 50 & avg_log2FC < -1 ~ "down",
                         TRUE ~ "ns"),
         sig = factor(sig, levels = c("down", "ns", "up")))

lab <- bind_rows(
  df %>% filter(sig == "up")   %>% pick_spread(5),
  df %>% filter(sig == "down") %>% pick_spread(5)
)

n_up <- sum(df$sig == "up"); n_dn <- sum(df$sig == "down"); n_ns <- sum(df$sig == "ns")

p <- ggplot(df, aes(avg_log2FC, y_raw, color = sig)) +
  geom_point(size = 0.7, alpha = 0.6) +
  scale_color_manual(values = c(down = "#3778AE", ns = "grey85", up = "#D6251F")) +
  geom_vline(xintercept = c(-1, 1), linetype = "dashed", color = "grey50") +
  geom_hline(yintercept = 50, linetype = "dashed", color = "grey50") +
  geom_text_repel(
    data = lab, aes(label = gene),
    size = 3.5, color = "black", fontface = "italic",
    box.padding = 0.8, point.padding = 0.5,
    force = 5, force_pull = 0.2, direction = "both",
    max.overlaps = Inf, min.segment.length = 0, seed = 42,
    segment.size = 0.3, segment.alpha = 0.6
  ) +
  coord_cartesian(clip = "off") +
  labs(title = paste0("DEG: macaque 5-HT vs mouse 5-HT\n",
                      "down(", n_dn, ")    ns(", n_ns, ")    up(", n_up, ")"),
       x = "Log2 FoldChange (macaque / mouse)", y = "-Log10 P.adj", color = NULL) +
  theme_classic(base_size = 13) +
  theme(plot.margin = margin(20, 40, 20, 20))

# ---- 3. 出图 + 导出 ----
ggsave(paste0(save_dir, "Fig_5HT_volcano_macaque_vs_mouse.pdf"),
       p, width = 7, height = 6)
ggsave(paste0(save_dir, "Fig_5HT_volcano_macaque_vs_mouse.png"),
       p, width = 7, height = 6, dpi = 300)

## ★ 用 dplyr::select 防止 AnnotationDbi 抢占
deg_out <- deg %>% dplyr::select(gene, avg_log2FC, p_val, p_val_adj, pct.1, pct.2)
write.csv(deg_out, paste0(save_dir, "Table_5HT_volcano_macaque_vs_mouse.csv"),
          row.names = FALSE)

print(p)
cat("火山图已保存到", save_dir, "\n")

In [ ]:
suppressMessages({
  library(Seurat); library(dplyr)
  library(ggplot2); library(ggrepel)
})

save_dir <- "./data/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

# ---- 0. 数据准备 ----
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)
classFiltSeurat$sub_show <- sub("^mac_", "", as.character(classFiltSeurat$subtype_recl))

## 只保留 5-HT 细胞（不限亚型，全部合并）
cells_5ht <- colnames(classFiltSeurat)[grepl("^5-HT_", classFiltSeurat$sub_show)]
obj_5ht <- subset(classFiltSeurat, cells = cells_5ht)

## 检查细胞数和物种分布
print(table(obj_5ht$species, obj_5ht$sub_show))

# ---- 1. 所有 5-HT 细胞：macaque vs mouse ----
## avg_log2FC > 0 = 在 macaque 中上调；< 0 = 在 mouse 中上调
Idents(obj_5ht) <- "species"
deg <- FindMarkers(obj_5ht, ident.1 = "macaque", ident.2 = "mouse",
                   assay = "RNA", min.pct = 0.1, logfc.threshold = 0,
                   test.use = "wilcox", verbose = FALSE)
deg$gene <- rownames(deg)

# ---- 2. 火山图 ----
pick_spread <- function(sub, n = 5) {
  sub <- sub %>% arrange(p_val_adj)
  bind_rows(
    slice_head(sub, n = 2),
    sub %>% filter(!(gene %in% slice_head(sub, n = 2)$gene)) %>%
      mutate(rank_fc = rank(-abs(avg_log2FC))) %>%
      arrange(rank_fc) %>% slice_head(n = n - 2)
  )
}

df <- deg %>%
  mutate(y_raw = -log10(pmax(p_val_adj, 1e-300)),  # 防止 p=0 产生 Inf
         sig = case_when(y_raw > 50 & avg_log2FC >  1 ~ "up",
                         y_raw > 50 & avg_log2FC < -1 ~ "down",
                         TRUE ~ "ns"),
         sig = factor(sig, levels = c("down", "ns", "up")))

## ★ 标签：每侧 top5 + 强制标出 NXPH2
lab <- bind_rows(
  df %>% filter(sig == "up")   %>% pick_spread(5),
  df %>% filter(sig == "down") %>% pick_spread(5)
)

nxph2_lab <- df %>% filter(gene == "NXPH2")
if (nrow(nxph2_lab) > 0) {
  lab <- bind_rows(lab, nxph2_lab) %>% distinct(gene, .keep_all = TRUE)
  message("NXPH2: log2FC = ", round(nxph2_lab$avg_log2FC, 3),
          ", p.adj = ", format(nxph2_lab$p_val_adj, digits = 3))
} else {
  message("注意: NXPH2 不在本次 DEG 结果中（可能被 min.pct = 0.1 过滤掉了）")
}

n_up <- sum(df$sig == "up"); n_dn <- sum(df$sig == "down"); n_ns <- sum(df$sig == "ns")

p <- ggplot(df, aes(avg_log2FC, y_raw, color = sig)) +
  geom_point(size = 0.7, alpha = 0.6) +
  ## ★ NXPH2 用紫色大点高亮
  geom_point(data = df %>% filter(gene == "NXPH2"),
             color = "#7B2D8E", size = 2.5, shape = 16) +
  scale_color_manual(values = c(down = "#3778AE", ns = "grey85", up = "#D6251F")) +
  geom_vline(xintercept = c(-1, 1), linetype = "dashed", color = "grey50") +
  geom_hline(yintercept = 50, linetype = "dashed", color = "grey50") +
  geom_text_repel(
    data = lab, aes(label = gene),
    size = 3.5, color = "black", fontface = "italic",
    box.padding = 0.8, point.padding = 0.5,
    force = 5, force_pull = 0.2, direction = "both",
    max.overlaps = Inf, min.segment.length = 0, seed = 42,
    segment.size = 0.3, segment.alpha = 0.6
  ) +
  coord_cartesian(clip = "off") +
  labs(title = paste0("DEG: macaque 5-HT vs mouse 5-HT\n",
                      "down(", n_dn, ")    ns(", n_ns, ")    up(", n_up, ")"),
       x = "Log2 FoldChange (macaque / mouse)", y = "-Log10 P.adj", color = NULL) +
  theme_classic(base_size = 13) +
  theme(plot.margin = margin(20, 40, 20, 20))

# ---- 3. 出图 + 导出 ----
ggsave(paste0(save_dir, "Fig_5HT_volcano_macaque_vs_mouse.pdf"),
       p, width = 7, height = 6)
ggsave(paste0(save_dir, "Fig_5HT_volcano_macaque_vs_mouse.png"),
       p, width = 7, height = 6, dpi = 300)

deg_out <- deg %>% dplyr::select(gene, avg_log2FC, p_val, p_val_adj, pct.1, pct.2)
write.csv(deg_out, paste0(save_dir, "Table_5HT_volcano_macaque_vs_mouse.csv"),
          row.names = FALSE)

print(p)
cat("火山图已保存到", save_dir, "\n")

In [ ]:
suppressMessages({
  library(Seurat); library(dplyr)
  library(ggplot2); library(ggrepel)
})

save_dir <- "./data/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

## ★ 检查系统里有没有 Arial（Linux 常常没有）
## 如果返回 0 行，说明没装，需要改用 "Liberation Sans"
system("fc-list | grep -i arial")

# ---- 0. 数据准备 ----
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)
classFiltSeurat$sub_show <- sub("^mac_", "", as.character(classFiltSeurat$subtype_recl))

## 只保留 5-HT 细胞（不限亚型，全部合并）
cells_5ht <- colnames(classFiltSeurat)[grepl("^5-HT_", classFiltSeurat$sub_show)]
obj_5ht <- subset(classFiltSeurat, cells = cells_5ht)

print(table(obj_5ht$species, obj_5ht$sub_show))

# ---- 1. 所有 5-HT 细胞：macaque vs mouse ----
Idents(obj_5ht) <- "species"
deg <- FindMarkers(obj_5ht, ident.1 = "macaque", ident.2 = "mouse",
                   assay = "RNA", min.pct = 0.1, logfc.threshold = 0,
                   test.use = "wilcox", verbose = FALSE)
deg$gene <- rownames(deg)

# ---- 2. 火山图 ----
pick_spread <- function(sub, n = 5) {
  sub <- sub %>% arrange(p_val_adj)
  bind_rows(
    slice_head(sub, n = 2),
    sub %>% filter(!(gene %in% slice_head(sub, n = 2)$gene)) %>%
      mutate(rank_fc = rank(-abs(avg_log2FC))) %>%
      arrange(rank_fc) %>% slice_head(n = n - 2)
  )
}

df <- deg %>%
  mutate(y_raw = -log10(pmax(p_val_adj, 1e-300)),
         sig = case_when(y_raw > 50 & avg_log2FC >  1 ~ "up",
                         y_raw > 50 & avg_log2FC < -1 ~ "down",
                         TRUE ~ "ns"),
         sig = factor(sig, levels = c("down", "ns", "up")))

## ★ 标签：每侧 top5 + 强制标出 NXPH2
## ★ 标签：每侧 top6（2 个 p 值最小 + 4 个 |FC| 最大）+ 强制标出 NXPH2
lab <- bind_rows(
  df %>% filter(sig == "up")   %>% pick_spread(6),
  df %>% filter(sig == "down") %>% pick_spread(6)
)
nxph2_lab <- df %>% filter(gene == "NXPH2")
if (nrow(nxph2_lab) > 0) {
  lab <- bind_rows(lab, nxph2_lab) %>% distinct(gene, .keep_all = TRUE)
  message("NXPH2: log2FC = ", round(nxph2_lab$avg_log2FC, 3),
          ", p.adj = ", format(nxph2_lab$p_val_adj, digits = 3))
} else {
  message("注意: NXPH2 不在本次 DEG 结果中（可能被 min.pct = 0.1 过滤掉了）")
}

n_up <- sum(df$sig == "up"); n_dn <- sum(df$sig == "down"); n_ns <- sum(df$sig == "ns")

## ★ 字体设置：Arial 8pt
ft <- "Arial"   # 如果 fc-list 没找到 Arial，改成 "Liberation Sans"

p <- ggplot(df, aes(avg_log2FC, y_raw, color = sig)) +
  geom_point(size = 0.7, alpha = 0.6) +
  ## ★ NXPH2 用紫色大点高亮
  geom_point(data = df %>% filter(gene == "NXPH2"),
             color = "#7B2D8E", size = 2.5, shape = 16) +
  scale_color_manual(values = c(down = "#3778AE", ns = "grey85", up = "#D6251F")) +
  geom_vline(xintercept = c(-1, 1), linetype = "dashed", color = "grey50") +
  geom_hline(yintercept = 50, linetype = "dashed", color = "grey50") +
  geom_text_repel(
    data = lab, aes(label = gene),
    ## ★ 标签：Arial 8pt（ggplot 文本 size 单位是 mm，8pt ≈ 2.82mm）
    family = ft, size = 8 / ggplot2::.pt, color = "black", fontface = "italic",
    box.padding = 0.8, point.padding = 0.5,
    force = 5, force_pull = 0.2, direction = "both",
    max.overlaps = Inf, min.segment.length = 0, seed = 42,
    segment.size = 0.3, segment.alpha = 0.6
  ) +
  coord_cartesian(clip = "off") +
  labs(title = paste0("DEG: macaque 5-HT vs mouse 5-HT\n",
                      "down(", n_dn, ")    ns(", n_ns, ")    up(", n_up, ")"),
       x = "Log2 FoldChange (macaque / mouse)", y = "-Log10 P.adj", color = NULL) +
  ## ★ 主题：Arial 8pt
  theme_classic(base_size = 8, base_family = ft) +
  theme(
    text            = element_text(family = ft, size = 8),
    plot.title      = element_text(family = ft, size = 8),
    axis.title      = element_text(family = ft, size = 8),
    axis.text       = element_text(family = ft, size = 8),
    legend.text     = element_text(family = ft, size = 8),
    legend.title    = element_text(family = ft, size = 8),
    plot.margin     = margin(20, 40, 20, 20)
  )

# ---- 3. 出图 + 导出 ----
## ★ PDF 用 cairo_pdf 设备，字体嵌入更可靠
ggsave(paste0(save_dir, "Fig_5HT_volcano_macaque_vs_mouse.pdf"),
       p, width = 7, height = 6, device = cairo_pdf)
ggsave(paste0(save_dir, "Fig_5HT_volcano_macaque_vs_mouse.png"),
       p, width = 7, height = 6, dpi = 300)

deg_out <- deg %>% dplyr::select(gene, avg_log2FC, p_val, p_val_adj, pct.1, pct.2)
write.csv(deg_out, paste0(save_dir, "Table_5HT_volcano_macaque_vs_mouse.csv"),
          row.names = FALSE)

print(p)
cat("火山图已保存到", save_dir, "\n")

In [ ]:
suppressMessages({
  library(Seurat); library(dplyr)
  library(ggplot2); library(ggrepel)
})

save_dir <- "./data/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

## 检查系统里有没有 Arial（Linux 常常没有）
## 如果返回 0 行，说明没装，把下面的 ft 改成 "Liberation Sans"
system("fc-list | grep -i arial")

# ---- 0. 数据准备 ----
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)
classFiltSeurat$sub_show <- sub("^mac_", "", as.character(classFiltSeurat$subtype_recl))

## 只保留 5-HT 细胞（不限亚型，全部合并）
cells_5ht <- colnames(classFiltSeurat)[grepl("^5-HT_", classFiltSeurat$sub_show)]
obj_5ht <- subset(classFiltSeurat, cells = cells_5ht)

print(table(obj_5ht$species, obj_5ht$sub_show))

# ---- 1. 所有 5-HT 细胞：macaque vs mouse ----
Idents(obj_5ht) <- "species"
deg <- FindMarkers(obj_5ht, ident.1 = "macaque", ident.2 = "mouse",
                   assay = "RNA", min.pct = 0.1, logfc.threshold = 0,
                   test.use = "wilcox", verbose = FALSE)
deg$gene <- rownames(deg)

# ---- 2. 火山图 ----
pick_spread <- function(sub, n = 6) {
  sub <- sub %>% arrange(p_val_adj)
  bind_rows(
    slice_head(sub, n = 2),
    sub %>% filter(!(gene %in% slice_head(sub, n = 2)$gene)) %>%
      mutate(rank_fc = rank(-abs(avg_log2FC))) %>%
      arrange(rank_fc) %>% slice_head(n = n - 2)
  )
}

df <- deg %>%
  mutate(y_raw = -log10(pmax(p_val_adj, 1e-300)),  # 防止 p=0 产生 Inf
         sig = case_when(y_raw > 50 & avg_log2FC >  1 ~ "up",
                         y_raw > 50 & avg_log2FC < -1 ~ "down",
                         TRUE ~ "ns"),
         sig = factor(sig, levels = c("down", "ns", "up")))

## 标签：每侧 top6（2 个 p 值最小 + 4 个 |FC| 最大）+ 强制标出 NXPH2
## ★ 标签：每侧只标 top1（p.adj 最小的 1 个）+ 强制标出 NXPH2
## ★ 标签：每侧只标 top1（p.adj 最小的 1 个），其他基因一律不标
## ★ 标签：每侧只标 top1，p 值相同时取 |FC| 最大的那个，并列不保留
lab <- bind_rows(
  df %>% filter(sig == "up") %>%
    arrange(p_val_adj, desc(abs(avg_log2FC))) %>%
    slice_head(n = 1),
  df %>% filter(sig == "down") %>%
    arrange(p_val_adj, desc(abs(avg_log2FC))) %>%
    slice_head(n = 1)
)

nxph2_lab <- df %>% filter(gene == "NXPH2")
if (nrow(nxph2_lab) > 0) {
  lab <- bind_rows(lab, nxph2_lab) %>% distinct(gene, .keep_all = TRUE)
  message("NXPH2: log2FC = ", round(nxph2_lab$avg_log2FC, 3),
          ", p.adj = ", format(nxph2_lab$p_val_adj, digits = 3))
} else {
  message("注意: NXPH2 不在本次 DEG 结果中（可能被 min.pct = 0.1 过滤掉了）")
}

n_up <- sum(df$sig == "up"); n_dn <- sum(df$sig == "down"); n_ns <- sum(df$sig == "ns")

## 字体设置：Arial 8pt
ft <- "Arial"   # 如果 fc-list 没找到 Arial，改成 "Liberation Sans"

p <- ggplot(df, aes(avg_log2FC, y_raw, color = sig)) +
  geom_point(size = 0.35, alpha = 0.6) +
  ## NXPH2 用紫色大点高亮
  geom_point(data = df %>% filter(gene == "NXPH2"),
             color = "#7B2D8E", size = 1.6, shape = 16) +
  scale_color_manual(values = c(down = "#3778AE", ns = "grey85", up = "#D6251F")) +
  geom_vline(xintercept = c(-1, 1), linetype = "dashed", color = "grey50") +
  geom_hline(yintercept = 50, linetype = "dashed", color = "grey50") +
    geom_text_repel(
    data = lab, aes(label = gene),
    family = ft, size = 8 / ggplot2::.pt, color = "black", fontface = "italic",
    ## ★ 进一步收短：回拉更强、排斥更弱、间距更小
    box.padding = 0.4,
    point.padding = 0.3,
    force = 1.5, force_pull = 3,     # ★ 原 force = 3, force_pull = 2
    direction = "both",
    max.overlaps = Inf,
    min.segment.length = 0,
    seed = 42,
    max.time = 3, max.iter = 1000000,
    segment.size = 0.3, segment.alpha = 0.7,   # ★ 线也细一点（原 0.35）
    segment.curvature = -0.1, segment.ncp = 3, segment.angle = 20
  ) +
  ## ★ 出血空间加大
  coord_cartesian(clip = "off", expand = TRUE) +
  labs(title = paste0("DEG: macaque 5-HT vs mouse 5-HT\n",
                      "down(", n_dn, ")    ns(", n_ns, ")    up(", n_up, ")"),
       x = "Log2 FoldChange (macaque / mouse)", y = "-Log10 P.adj", color = NULL) +
  ## 主题：Arial 8pt
  theme_classic(base_size = 8, base_family = ft) +
  theme(
    text            = element_text(family = ft, size = 8),
    plot.title      = element_text(family = ft, size = 8),
    axis.title      = element_text(family = ft, size = 8),
    axis.text       = element_text(family = ft, size = 8),
    legend.text     = element_text(family = ft, size = 8),
    legend.title    = element_text(family = ft, size = 8),
    ## ★ 四周边距加宽，给错开的标签留位置
        ## ★ 小图必须配小边距（6.5cm 图容不下原来的 margin(40, 80, 40, 40)）
    plot.margin     = margin(6, 10, 6, 6)
  )

# ---- 3. 出图 + 导出 ----
## ★ 高度 6 → 8，顶部密集区纵向空间更大
# ---- 3. 出图 + 导出 ----
## ★ 宽 6.5、高 6
# ---- 3. 出图 + 导出 ----
## ★ 高 6cm × 宽 6.5cm（期刊单栏图标准尺寸）
ggsave(paste0(save_dir, "Fig_5HT_volcano_macaque_vs_mouse.pdf"),
       p, width = 6.5, height = 6, units = "cm", device = cairo_pdf)
ggsave(paste0(save_dir, "Fig_5HT_volcano_macaque_vs_mouse.png"),
       p, width = 6.5, height = 6, units = "cm", dpi = 600)  # ★ 小图建议 dpi 提到 600
deg_out <- deg %>% dplyr::select(gene, avg_log2FC, p_val, p_val_adj, pct.1, pct.2)
write.csv(deg_out, paste0(save_dir, "Table_5HT_volcano_macaque_vs_mouse.csv"),
          row.names = FALSE)

print(p)
cat("火山图已保存到", save_dir, "\n")

In [ ]:
suppressMessages({
  library(Seurat); library(dplyr)
  library(ggplot2); library(ggrepel)
})

save_dir <- "./data/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

## 检查系统里有没有 Arial（Linux 常常没有）
## 如果返回 0 行，说明没装，把下面的 ft 改成 "Liberation Sans"
system("fc-list | grep -i arial")

# ---- 0. 数据准备 ----
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)
classFiltSeurat$sub_show <- sub("^mac_", "", as.character(classFiltSeurat$subtype_recl))

## 只保留 5-HT 细胞（不限亚型，全部合并）
cells_5ht <- colnames(classFiltSeurat)[grepl("^5-HT_", classFiltSeurat$sub_show)]
obj_5ht <- subset(classFiltSeurat, cells = cells_5ht)

print(table(obj_5ht$species, obj_5ht$sub_show))

# ---- 1. 所有 5-HT 细胞：macaque vs mouse ----
Idents(obj_5ht) <- "species"
deg <- FindMarkers(obj_5ht, ident.1 = "macaque", ident.2 = "mouse",
                   assay = "RNA", min.pct = 0.1, logfc.threshold = 0,
                   test.use = "wilcox", verbose = FALSE)
deg$gene <- rownames(deg)

# ---- 2. 火山图 ----
pick_spread <- function(sub, n = 6) {
  sub <- sub %>% arrange(p_val_adj)
  bind_rows(
    slice_head(sub, n = 2),
    sub %>% filter(!(gene %in% slice_head(sub, n = 2)$gene)) %>%
      mutate(rank_fc = rank(-abs(avg_log2FC))) %>%
      arrange(rank_fc) %>% slice_head(n = n - 2)
  )
}

df <- deg %>%
  mutate(y_raw = -log10(pmax(p_val_adj, 1e-300)),
         sig = case_when(y_raw > 50 & avg_log2FC >  3 ~ "up",    # ★ 1 → 3
                         y_raw > 50 & avg_log2FC < -3 ~ "down",  # ★ -1 → -3
                         TRUE ~ "ns"),
         sig = factor(sig, levels = c("down", "ns", "up")))

## 标签：每侧 top6（2 个 p 值最小 + 4 个 |FC| 最大）+ 强制标出 NXPH2
## ★ 标签：每侧只标 top1（p.adj 最小的 1 个）+ 强制标出 NXPH2
## ★ 标签：每侧只标 top1（p.adj 最小的 1 个），其他基因一律不标
## ★ 标签：每侧只标 top1，p 值相同时取 |FC| 最大的那个，并列不保留
lab <- bind_rows(
  df %>% filter(sig == "up") %>%
    arrange(p_val_adj, desc(abs(avg_log2FC))) %>%
    slice_head(n = 1),
  df %>% filter(sig == "down") %>%
    arrange(p_val_adj, desc(abs(avg_log2FC))) %>%
    slice_head(n = 1)
)

nxph2_lab <- df %>% filter(gene == "NXPH2")
if (nrow(nxph2_lab) > 0) {
  lab <- bind_rows(lab, nxph2_lab) %>% distinct(gene, .keep_all = TRUE)
  message("NXPH2: log2FC = ", round(nxph2_lab$avg_log2FC, 3),
          ", p.adj = ", format(nxph2_lab$p_val_adj, digits = 3))
} else {
  message("注意: NXPH2 不在本次 DEG 结果中（可能被 min.pct = 0.1 过滤掉了）")
}

n_up <- sum(df$sig == "up"); n_dn <- sum(df$sig == "down"); n_ns <- sum(df$sig == "ns")

## 字体设置：Arial 8pt
ft <- "Arial"   # 如果 fc-list 没找到 Arial，改成 "Liberation Sans"

p <- ggplot(df, aes(avg_log2FC, y_raw, color = sig)) +
  geom_point(size = 0.35, alpha = 0.6) +
  ## NXPH2 用紫色大点高亮
  geom_point(data = df %>% filter(gene == "NXPH2"),
             color = "#7B2D8E", size = 1.6, shape = 16) +
  scale_color_manual(values = c(down = "#3778AE", ns = "grey85", up = "#D6251F")) +
  geom_vline(xintercept = c(-3, 3), linetype = "dashed", color = "grey50") +  # ★ ±1 → ±3
  geom_hline(yintercept = 50, linetype = "dashed", color = "grey50") +
    geom_text_repel(
    data = lab, aes(label = gene),
    family = ft, size = 8 / ggplot2::.pt, color = "black", fontface = "italic",
    ## ★ 进一步收短：回拉更强、排斥更弱、间距更小
    box.padding = 0.4,
    point.padding = 0.3,
    force = 1.5, force_pull = 3,     # ★ 原 force = 3, force_pull = 2
    direction = "both",
    max.overlaps = Inf,
    min.segment.length = 0,
    seed = 42,
    max.time = 3, max.iter = 1000000,
    segment.size = 0.3, segment.alpha = 0.7,   # ★ 线也细一点（原 0.35）
    segment.curvature = -0.1, segment.ncp = 3, segment.angle = 20
  ) +
  ## ★ 出血空间加大
  coord_cartesian(clip = "off", expand = TRUE) +
  labs(title = paste0("DEG: macaque 5-HT vs mouse 5-HT\n",
                      "down(", n_dn, ")    ns(", n_ns, ")    up(", n_up, ")"),
       x = "Log2 FoldChange (macaque / mouse)", y = "-Log10 P.adj", color = NULL) +
  ## 主题：Arial 8pt
  theme_classic(base_size = 8, base_family = ft) +
  theme(
    text            = element_text(family = ft, size = 8),
    plot.title      = element_text(family = ft, size = 8),
    axis.title      = element_text(family = ft, size = 8),
    axis.text       = element_text(family = ft, size = 8),
    legend.text     = element_text(family = ft, size = 8),
    legend.title    = element_text(family = ft, size = 8),
    ## ★ 四周边距加宽，给错开的标签留位置
        ## ★ 小图必须配小边距（6.5cm 图容不下原来的 margin(40, 80, 40, 40)）
    plot.margin     = margin(6, 10, 6, 6)
  )

# ---- 3. 出图 + 导出 ----
## ★ 高度 6 → 8，顶部密集区纵向空间更大
# ---- 3. 出图 + 导出 ----
## ★ 宽 6.5、高 6
# ---- 3. 出图 + 导出 ----
## ★ 高 6cm × 宽 6.5cm（期刊单栏图标准尺寸）
ggsave(paste0(save_dir, "Fig_5HT_volcano_macaque_vs_mouse2.pdf"),
       p, width = 6.5, height = 6, units = "cm", device = cairo_pdf)
ggsave(paste0(save_dir, "Fig_5HT_volcano_macaque_vs_mouse2.png"),
       p, width = 6.5, height = 6, units = "cm", dpi = 600)  # ★ 小图建议 dpi 提到 600
deg_out <- deg %>% dplyr::select(gene, avg_log2FC, p_val, p_val_adj, pct.1, pct.2)
write.csv(deg_out, paste0(save_dir, "Table_5HT_volcano_macaque_vs_mouse2.csv"),
          row.names = FALSE)

print(p)
cat("火山图已保存到", save_dir, "\n")

In [ ]:
# ---- 4. NXPH2 表达小提琴图：macaque vs mouse ----
## 取出 NXPH2 表达数据
df_nxph2 <- FetchData(obj_5ht, vars = c("NXPH2", "species"))
df_nxph2$species <- factor(df_nxph2$species, levels = c("macaque", "mouse"))

p_nx <- ggplot(df_nxph2, aes(x = species, y = NXPH2, fill = species)) +
  ## 纯小提琴：不加箱线、不加中位线、不加散点
  geom_violin(scale = "width", trim = TRUE, adjust = 1,
              linewidth = 0.3, color = "black") +
  scale_fill_manual(values = c(macaque = "#EB6368", mouse = "#82ADD0")) +
  labs(x = NULL, y = "NXPH2 expression", fill = NULL) +
  theme_classic(base_size = 8, base_family = ft) +
  theme(
    text        = element_text(family = ft, size = 8),
    axis.text   = element_text(family = ft, size = 8),
    axis.title  = element_text(family = ft, size = 8),
    legend.text = element_text(family = ft, size = 8),
    legend.position = "none",          # 颜色已对应 x 轴，不用图例
    plot.margin = margin(6, 10, 6, 6)
  )

ggsave(paste0(save_dir, "Fig_NXPH2_violin_macaque_vs_mouse.pdf"),
       p_nx, width = 4, height = 4.5, units = "cm", device = cairo_pdf)
ggsave(paste0(save_dir, "Fig_NXPH2_violin_macaque_vs_mouse.png"),
       p_nx, width = 4, height = 4.5, units = "cm", dpi = 600)

print(p_nx)

In [ ]:
# ---- 4. NXPH2 表达小提琴图：macaque vs mouse ----
## 取出 NXPH2 表达数据
df_nxph2 <- FetchData(obj_5ht, vars = c("NXPH2", "species"))
df_nxph2$species <- factor(df_nxph2$species, levels = c("macaque", "mouse"))

p_nx <- ggplot(df_nxph2, aes(x = species, y = NXPH2, fill = species)) +
  ## 纯小提琴：不加箱线、不加中位线、不加散点
  geom_violin(scale = "width", trim = TRUE, adjust = 1,
              linewidth = 0.3, color = "black") +
  scale_fill_manual(values = c(macaque = "#EB6368", mouse = "#82ADD0")) +
  labs(x = NULL, y = "NXPH2 expression", fill = NULL) +
  theme_classic(base_size = 8, base_family = ft) +
  theme(
    text        = element_text(family = ft, size = 8),
    axis.text   = element_text(family = ft, size = 8),
    axis.title  = element_text(family = ft, size = 8),
    legend.text = element_text(family = ft, size = 8),
    legend.position = "none",          # 颜色已对应 x 轴，不用图例
    plot.margin = margin(6, 10, 6, 6)
  )

ggsave(paste0(save_dir, "Fig_NXPH2_violin_macaque_vs_mouse.pdf"),
       p_nx, width = 4, height = 4.5, units = "cm", device = cairo_pdf)
ggsave(paste0(save_dir, "Fig_NXPH2_violin_macaque_vs_mouse.png"),
       p_nx, width = 4, height = 4.5, units = "cm", dpi = 600)

print(p_nx)

In [ ]:
## 1) NXPH2 本身的信息
deg %>% filter(gene == "NXPH2") %>%
  mutate(y_raw = -log10(pmax(p_val_adj, 1e-300)))

## 2) 在"显著上调"基因（y>50 且 FC>1）中按 p.adj 排名
deg %>%
  filter(p_val_adj < 1e-50, avg_log2FC > 1) %>%
  arrange(p_val_adj) %>%
  mutate(rank_by_p = row_number(),
         rank_by_fc = rank(-avg_log2FC)) %>%
  filter(gene == "NXPH2")

## 3) 在"显著下调"基因（y>50 且 FC<-1）中按 p.adj 排名
deg %>%
  filter(p_val_adj < 1e-50, avg_log2FC < -1) %>%
  arrange(p_val_adj) %>%
  mutate(rank_by_p = row_number(),
         rank_by_fc = rank(avg_log2FC)) %>%
  filter(gene == "NXPH2")

## 4) 不分方向，在整个 DEG 表里按 p.adj 的总排名
deg %>% arrange(p_val_adj) %>%
  mutate(rank_total = row_number()) %>%
  filter(gene == "NXPH2")

In [ ]:
deg %>% filter(gene == "NXPH2") %>%
  mutate(y_raw = -log10(pmax(p_val_adj, 1e-300)))

In [ ]:
## NXPH2 在显著上调基因中的名次（不依赖 select）
res <- deg %>%
  filter(p_val_adj < 1e-50, avg_log2FC > 1) %>%
  arrange(p_val_adj) %>%
  mutate(rank_by_p  = row_number(),           # 按 p.adj 从小到大排
         rank_by_fc = rank(-avg_log2FC)) %>%  # 按 FC 从大到小排
  filter(gene == "NXPH2")

print(res[, c("gene", "avg_log2FC", "p_val_adj", "rank_by_p", "rank_by_fc")])

In [ ]:
suppressMessages({
  library(tidyverse); library(Seurat)
  library(ComplexHeatmap); library(circlize); library(grid); library(dplyr)
})

save_dir <- "./data/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)

# ---- 1. macaque vs mouse 一次比较（logfc.threshold = 0，保留所有基因）----
Idents(classFiltSeurat) <- "species"
mk <- FindMarkers(classFiltSeurat, assay = "RNA",
                  ident.1 = "macaque", ident.2 = "mouse",
                  min.pct = 0.05, logfc.threshold = 0,
                  test.use = "wilcox", verbose = FALSE)

sig <- mk$p_val_adj < 0.05
genes_mac <- rownames(mk)[sig & mk$avg_log2FC >  0.5]   # macaque 富集
genes_mus <- rownames(mk)[sig & mk$avg_log2FC < -0.5]   # mouse 富集
# ★ 保守共有：不显著差异 + 两个物种都表达
shared <- rownames(mk)[!sig & mk$pct.1 > 0.25 & mk$pct.2 > 0.25]

cat("Shared:", length(shared), " macaque-specific:", length(genes_mac),
    " mouse-specific:", length(genes_mus), "\n")

# ---- 2. 三块分类 ----
genes_use <- c(shared, genes_mac, genes_mus)
gene2stat <- setNames(c(rep("Shared", length(shared)),
                        rep("Macaque-specific", length(genes_mac)),
                        rep("Mouse-specific", length(genes_mus))),
                      genes_use)
n_stat <- c("Shared" = length(shared),
            "Macaque-specific" = length(genes_mac),
            "Mouse-specific" = length(genes_mus))

# ---- 3. 表达矩阵（基因 × 细胞 → z-score）----
mat <- as.matrix(GetAssayData(classFiltSeurat, assay = "RNA",
                              slot = "data")[genes_use, , drop = FALSE])
z <- t(scale(t(mat)))
z[is.na(z)] <- 0
z[z > 2] <- 2; z[z < -2] <- -2

cell_sp <- as.character(classFiltSeurat$species)

# ---- 4. 热图：行 = 细胞（上 mouse 下 macaque），列 = 三块基因 ----
stat_cols <- c("Shared" = "grey35",
               "Macaque-specific" = "#EB6368",
               "Mouse-specific"   = "#82ADD0")
col_stat <- factor(gene2stat[genes_use], levels = names(stat_cols))
col_titles <- sprintf("%s\nn=%d", names(n_stat), n_stat)

ht <- Heatmap(t(z),                              # ★ 转置：行=细胞，列=基因
  name = "z-score",
  col = colorRamp2(c(-2, 0, 2), c("#2166AC", "white", "#B2182B")),
  row_split = factor(cell_sp, levels = c("mouse", "macaque")),
  row_title = c("mouse", "macaque"),
  row_title_gp = gpar(fontsize = 12, fontface = "bold"),
  cluster_row_slices = TRUE, cluster_rows = TRUE,
  show_row_dend = FALSE, show_row_names = FALSE,
  column_split = col_stat,
  cluster_column_slices = FALSE, cluster_columns = FALSE,
  show_column_names = FALSE,
  column_title = col_titles,
  column_title_gp = gpar(fontsize = 11, fontface = "bold"),
  border = FALSE, rect_gp = gpar(col = NA),
  right_annotation = rowAnnotation(
    species = cell_sp,
    col = list(species = c(macaque = "#EB6368", mouse = "#82ADD0")),
    show_annotation_name = FALSE,
    annotation_legend_param = list(species = list(title = "species"))),
  top_annotation = columnAnnotation(
    Conservation = col_stat,
    col = list(Conservation = stat_cols),
    show_annotation_name = FALSE,
    annotation_legend_param = list(Conservation = list(title = "Conservation"))),
  use_raster = TRUE, raster_quality = 3,
  width = unit(16, "cm"), height = unit(14, "cm")
)

pdf(paste0(save_dir, "Fig_5HT_shared_specific_genes_cells.pdf"),
    width = 12, height = 10)
draw(ht)
dev.off()

png(paste0(save_dir, "Fig_5HT_shared_specific_genes_cells.png"),
    width = 12, height = 10, units = "in", res = 300)
draw(ht)
dev.off()

# ---- 5. 输出基因个数和列表 ----
gene_table <- data.frame(gene = genes_use,
                         conservation = gene2stat[genes_use],
                         row.names = NULL)
write.csv(gene_table, paste0(save_dir, "Table_5HT_shared_specific_genes.csv"),
          row.names = FALSE)
write.csv(data.frame(category = names(n_stat), n = as.integer(n_stat)),
          paste0(save_dir, "Table_5HT_shared_specific_geneCounts.csv"),
          row.names = FALSE)
cat("图和基因表已保存到", save_dir, "\n")

In [ ]:
# go分析

In [ ]:
library(dplyr)
library(ggplot2)
save_dir <- "./data/"
## ============================================================
## 诊断 1：不同阈值组合下，每个亚型通过过滤的基因数
## ============================================================
threshold_grid <- expand.grid(
  FC  = c(0.25, 0.5, 0.75, 1.0),
  PCT = c(0.1, 0.2, 0.3, 0.5),
  stringsAsFactors = FALSE
)

diag_pass <- lapply(split(threshold_grid, seq_len(nrow(threshold_grid))), function(th) {
  mk_mac %>%
    filter(p_val_adj < 0.05, avg_log2FC > th$FC, pct.1 > th$PCT) %>%
    count(cluster, name = "n_pass") %>%
    mutate(FC = th$FC, PCT = th$PCT)
}) %>% bind_rows()

# 宽表：行 = 亚型，列 = 阈值组合
diag_wide <- diag_pass %>%
  mutate(combo = paste0("FC>", FC, "_pct>", PCT)) %>%
  select(cluster, combo, n_pass) %>%
  pivot_wider(names_from = combo, values_from = n_pass, values_fill = 0) %>%
  arrange(cluster)

print(diag_wide, n = Inf)
write.csv(diag_wide, paste0(save_dir, "Table_threshold_diagnostic.csv"),
          row.names = FALSE)

## ============================================================
## 诊断 2：每个亚型 avg_log2FC 的分布（决定 FC 阈值切哪）
## ============================================================
fc_dist <- mk_mac %>%
  filter(p_val_adj < 0.05) %>%
  group_by(cluster) %>%
  summarise(
    n_sig      = n(),
    fc_median  = round(median(avg_log2FC), 2),
    fc_top10   = round(sort(avg_log2FC, decreasing = TRUE)[min(10, n())], 2),
    fc_top30   = round(sort(avg_log2FC, decreasing = TRUE)[min(30, n())], 2),
    fc_top50   = round(sort(avg_log2FC, decreasing = TRUE)[min(50, n())], 2),
    .groups = "drop"
  ) %>%
  arrange(cluster)

print(fc_dist, n = Inf)
# fc_top10/30/50 = 该亚型 FC 排名第 10/30/50 的基因折叠是多少
# 如果 fc_top50 只有 0.4，说明该亚型"前50"里很多基因 FC 不到 0.5

## ============================================================
## 诊断 3：TOP_N 取多少合适？看 FC>0.5 且 pct>0.3 后的累计分布
## ============================================================
n_after <- mk_mac %>%
  filter(p_val_adj < 0.05, avg_log2FC > 0.5, pct.1 > 0.3) %>%
  count(cluster, name = "n_after_filter") %>%
  arrange(cluster)

print(n_after, n = Inf)
hist(n_after$n_after_filter, breaks = 20,
     main = "每亚型通过 FC>0.5 & pct>0.3 的基因数分布",
     xlab = "基因数")

## ============================================================
## 诊断 4（快速决定法）：找到"既不太严也不太松"的阈值
## ============================================================
# 目标：绝大多数亚型过滤后剩 30~100 个基因（GO 富集的甜区）
target_check <- diag_pass %>%
  group_by(FC, PCT) %>%
  summarise(
    n_subtype     = n(),
    min_genes     = min(n_pass),
    median_genes  = round(median(n_pass)),
    n_below_30    = sum(n_pass < 30),    # 剩太少、GO 跑不稳的亚型数
    n_above_100   = sum(n_pass > 100),   # 剩太多、TOP_N 截断浪费的亚型数
    .groups = "drop"
  ) %>%
  arrange(desc(n_below_30 == 0), median_genes)   # 优先看"没有亚型低于30"的组合

print(target_check, n = Inf)

In [ ]:
if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")
BiocManager::install("org.Mm.eg.db")

In [ ]:
suppressMessages({
  library(clusterProfiler); library(org.Hs.eg.db); library(org.Mm.eg.db)
  library(enrichplot); library(ggplot2); library(dplyr)
})

save_dir <- "./data/"

## ============================================================
## 0. 定义两个分析单元（改这里就是换基因集）
## ============================================================
analyses <- list(
  list(label    = "macaque_specific",
       genes    = panel %>% filter(status == "Macaque-specific") %>% pull(gene),
       bg       = unique(mk_mac$gene),        # macaque FindAllMarkers 检验过的全部基因
       OrgDb    = org.Hs.eg.db),
  list(label    = "mouse_specific",
       genes    = panel %>% filter(status == "Mouse-specific") %>% pull(gene),
       bg       = unique(mk_mus$gene),        # mouse 检验过的基因
       OrgDb    = org.Mm.eg.db)
)

## ============================================================
## 1. 富集 + 出图函数
## ============================================================
run_one <- function(gene_set, bg_genes, OrgDb, label) {
  cat("\n========== ", label, " ==========\n")
  cat("基因集大小:", length(gene_set), "  背景集:", length(bg_genes), "\n")

  # ★ 关键：基因集和背景集用同一个 bitr 转换，映射率才一致可比
  gene_map <- bitr(gene_set, fromType = "SYMBOL", toType = "ENTREZID", OrgDb = OrgDb)
  bg_map   <- bitr(bg_genes, fromType = "SYMBOL", toType = "ENTREZID", OrgDb = OrgDb)

  cat("基因集映射率:", nrow(gene_map), "/", length(gene_set),
      " (", round(100 * nrow(gene_map) / length(gene_set), 1), "%)\n", sep = "")
  cat("背景集映射率:", nrow(bg_map), "/", length(bg_genes),
      " (", round(100 * nrow(bg_map) / length(bg_genes), 1), "%)\n", sep = "")

  ego_bp <- enrichGO(gene          = gene_map$ENTREZID,
                     universe      = bg_map$ENTREZID,
                     OrgDb         = OrgDb,
                     keyType       = "ENTREZID",
                     ont           = "BP",
                     pAdjustMethod = "BH",
                     pvalueCutoff  = 0.05,
                     qvalueCutoff  = 0.2,
                     readable      = TRUE)

  df <- as.data.frame(ego_bp)
  cat("富集到 ", nrow(df), " 条 BP 通路\n")

  write.csv(df, paste0(save_dir, "GO_BP_", label, ".csv"), row.names = FALSE)

  if (nrow(df) > 0) {
    p_dot <- dotplot(ego_bp, showCategory = 20, font.size = 9) +
      ggtitle(paste0("GO BP: ", label))
    ggsave(paste0(save_dir, "GO_BP_", label, "_dotplot.pdf"),
           p_dot, width = 8, height = 7)
    ggsave(paste0(save_dir, "GO_BP_", label, "_dotplot.png"),
           p_dot, width = 8, height = 7, dpi = 300)

    p_bar <- barplot(ego_bp, showCategory = 20, font.size = 9) +
      ggtitle(paste0("GO BP: ", label))
    ggsave(paste0(save_dir, "GO_BP_", label, "_barplot.pdf"),
           p_bar, width = 8, height = 7)

    p_net <- cnetplot(ego_bp, showCategory = 8, circular = FALSE,
                      colorEdge = TRUE, node_label = "gene",
                      cex_label_category = 0.9, cex_label_gene = 0.7)
    ggsave(paste0(save_dir, "GO_BP_", label, "_cnetplot.pdf"),
           p_net, width = 11, height = 9)

    ego_s <- pairwise_termsim(ego_bp)
    p_tree <- treeplot(ego_s, showCategory = 20)
    ggsave(paste0(save_dir, "GO_BP_", label, "_treeplot.pdf"),
           p_tree, width = 9, height = 7)

    print(p_dot)
  }
  invisible(df)
}

## ============================================================
## 2. 跑两个物种特异集
## ============================================================
go_results <- lapply(analyses, function(a)
  run_one(a$genes, a$bg, a$OrgDb, a$label))
names(go_results) <- sapply(analyses, `[[`, "label")

In [ ]:
suppressMessages({ library(ggplot2); library(dplyr) })

save_dir <- "./data/"

## ============================================================
## 读结果 + 过滤（和之前定的非 raphe 条目黑名单一致）
## ============================================================
go_mac <- read.csv(paste0(save_dir, "GO_BP_macaque_specific.csv"))

drop_terms <- c("ossification", "osteoblast differentiation",
                "epithelial cell proliferation",
                "positive regulation of epithelial cell proliferation",
                "morphogenesis of a branching epithelium",
                "muscle cell proliferation",
                "response to lipopolysaccharide",
                "response to molecule of bacterial origin")

go_show <- go_mac %>% filter(!Description %in% drop_terms)

TOP_N <- 10

plot_df <- go_show %>%
  slice_min(order_by = p.adjust, n = min(TOP_N, nrow(go_show)), with_ties = FALSE) %>%
  mutate(neglogP = -log10(p.adjust),
         label   = paste0(Description, " (", ID, ")"),
         # 最显著的条在最上面
         label   = factor(label, levels = rev(label)))

## ============================================================
## 条形图
## ============================================================
p_bar <- ggplot(plot_df, aes(x = neglogP, y = label, fill = neglogP)) +
  geom_bar(stat = "identity", width = 0.7, color = NA) +
  scale_fill_gradient2(low = "#2166AC", mid = "white", high = "#B2182B",
                       midpoint = max(plot_df$neglogP) / 2,
                       name = "-log10(Padj)") +
  scale_x_continuous(expand = expansion(mult = c(0, 0.1))) +
  labs(title = "Top GO Biological Processes",
       x = "-log10(Padj)", y = NULL) +
  theme_classic(base_size = 10) +
  theme(plot.title  = element_text(hjust = 0.5, face = "bold", size = 12),
        axis.text.y = element_text(size = 9, color = "black"),
        axis.text.x = element_text(size = 9, color = "black"),
        axis.line.y = element_blank(),
        axis.ticks.y = element_blank(),
        legend.position = "right",
        legend.title = element_text(size = 9),
        legend.text  = element_text(size = 8),
        legend.key.size = unit(0.4, "cm"))

print(p_bar)

ggsave(paste0(save_dir, "GO_BP_macaque_specific_barplot_custom.png"),
       p_bar, width = 8, height = 5.5, dpi = 300)
ggsave(paste0(save_dir, "GO_BP_macaque_specific_barplot_custom.pdf"),
       p_bar, width = 8, height = 5.5)

In [ ]:
suppressMessages({
  library(clusterProfiler); library(org.Hs.eg.db)
  library(enrichplot); library(ggplot2); library(dplyr)
})
select <- dplyr::select

save_dir <- "./data/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

## ============================================================
## 0. 准备：macaque-specific 基因集 + 按 FC 排名
## ============================================================
mac_specific <- panel %>% filter(status == "Macaque-specific") %>% pull(gene)

# 用 mk_mac 里的最佳 avg_log2FC 给每个基因排名（一个基因可能出现在多个亚型，取最大值）
fc_rank <- mk_mac %>%
  filter(gene %in% mac_specific) %>%
  group_by(gene) %>%
  summarise(max_fc = max(avg_log2FC), .groups = "drop") %>%
  arrange(desc(max_fc))

bg_genes <- unique(mk_mac$gene)

cat("macaque-specific 基因总数:", nrow(fc_rank), "\n")

## ============================================================
## 1. GO 函数（阈值和之前一致）
## ============================================================
run_go <- function(gene_set, bg_genes, top_n) {
  gene_map <- bitr(gene_set, fromType = "SYMBOL", toType = "ENTREZID",
                   OrgDb = org.Hs.eg.db)
  bg_map   <- bitr(bg_genes, fromType = "SYMBOL", toType = "ENTREZID",
                   OrgDb = org.Hs.eg.db)

  ego <- enrichGO(gene          = gene_map$ENTREZID,
                  universe      = bg_map$ENTREZID,
                  OrgDb         = org.Hs.eg.db,
                  keyType       = "ENTREZID",
                  ont           = "BP",
                  pAdjustMethod = "BH",
                  pvalueCutoff  = 0.05,
                  qvalueCutoff  = 0.2,
                  readable      = TRUE)

  df <- as.data.frame(ego) %>% mutate(top_n = top_n)
  cat(sprintf("Top%-4d | 基因 %4d (映射%d) | 富集 %d 条 BP\n",
              top_n, length(gene_set), nrow(gene_map), nrow(df)))
  df
}

## ============================================================
## 2. 循环 Top 10,20,...,100 分别跑
## ============================================================
top_n_vec <- seq(10, 100, by = 10)

go_list <- lapply(top_n_vec, function(nn) {
  genes_n <- fc_rank %>% slice_head(n = min(nn, nrow(.))) %>% pull(gene)
  run_go(genes_n, bg_genes, nn)
})
names(go_list) <- paste0("Top", top_n_vec)

# 汇总表：每个 N 富集到多少条通路
summary_df <- data.frame(
  top_n = top_n_vec,
  n_genes = top_n_vec,
  n_BP = sapply(go_list, nrow)
)
print(summary_df)
write.csv(summary_df, paste0(save_dir, "GO_topN_summary.csv"), row.names = FALSE)

# 每个 N 的完整结果分别存
for (nm in names(go_list)) {
  write.csv(go_list[[nm]],
            paste0(save_dir, "GO_BP_macaque_specific_", nm, ".csv"),
            row.names = FALSE)
}

## ============================================================
## 3. 快速对比：富集通路数 vs N（看饱和点）
## ============================================================
p_sat <- ggplot(summary_df, aes(top_n, n_BP)) +
  geom_line(color = "#B2182B", linewidth = 0.8) +
  geom_point(size = 2, color = "#B2182B") +
  scale_x_continuous(breaks = top_n_vec) +
  labs(x = "Top N genes (ranked by avg_log2FC)",
       y = "Number of enriched GO BP terms",
       title = "GO enrichment saturation across top-N thresholds") +
  theme_bw(base_size = 10) +
  theme(panel.grid.minor = element_blank(),
        plot.title = element_text(hjust = 0.5, face = "bold", size = 11))

print(p_sat)
ggsave(paste0(save_dir, "GO_topN_saturation.png"), p_sat,
       width = 7, height = 4.5, dpi = 300)

# 把 GPCR 条目在各 N 中的显著性轨迹拉出来（你的故事线核心条目）
gpcr_track <- bind_rows(go_list) %>%
  filter(Description == "G protein-coupled receptor signaling pathway") %>%
  select(top_n, Description, Count, GeneRatio, p.adjust)

print(gpcr_track)
write.csv(gpcr_track, paste0(save_dir, "GO_topN_GPCR_track.csv"),
          row.names = FALSE)

In [ ]:
suppressMessages({
  library(clusterProfiler); library(org.Hs.eg.db)
  library(enrichplot); library(ggplot2); library(dplyr)
})

save_dir <- "./data/"

# ---- 1. 基因集和背景集 ----
mac_genes <- gene_table %>% filter(conservation == "Macaque-specific") %>% pull(gene)
bg_genes  <- rownames(mk)          # FindMarkers 检验过的全部基因做背景，比全基因组更严谨
cat("macaque-specific:", length(mac_genes), " background:", length(bg_genes), "\n")

# ---- 2. GO 富集（BP 为主，三个本体可都跑）----
run_go <- function(ont) {
  enrichGO(gene          = mac_genes,
           universe      = bg_genes,
           OrgDb         = org.Hs.eg.db,
           keyType       = "SYMBOL",
           ont           = ont,
           pAdjustMethod = "BH",
           pvalueCutoff  = 0.05,
           qvalueCutoff  = 0.2,
           readable      = TRUE)
}

ego_bp <- run_go("BP")
ego_cc <- run_go("CC")
ego_mf <- run_go("MF")

# 看看映射率（猴符号 → 人源 Entrez 会丢一部分，属正常）
cat("BP 富集到 ", nrow(as.data.frame(ego_bp)), " 条通路\n")

# ---- 3. 保存结果表 ----
write.csv(as.data.frame(ego_bp), paste0(save_dir, "GO_BP_macaque_specific.csv"), row.names = FALSE)
write.csv(as.data.frame(ego_cc), paste0(save_dir, "GO_CC_macaque_specific.csv"), row.names = FALSE)
write.csv(as.data.frame(ego_mf), paste0(save_dir, "GO_MF_macaque_specific.csv"), row.names = FALSE)

# ---- 4. 出图 ----
# 条形图
p_bar <- barplot(ego_bp, showCategory = 20, font.size = 9) +
  ggtitle("GO BP: macaque-specific 5-HT genes")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_barplot.pdf"), p_bar, width = 8, height = 7)

# 气泡图
p_dot <- dotplot(ego_bp, showCategory = 20, font.size = 9) +
  ggtitle("GO BP: macaque-specific 5-HT genes")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_dotplot.pdf"), p_dot, width = 8, height = 7)

# 基因-通路网络图（最直观，哪些基因推富集）
p_net <- cnetplot(ego_bp, showCategory = 8, circular = FALSE,
                  colorEdge = TRUE, node_label = "gene",
                  cex_label_category = 0.9, cex_label_gene = 0.7)
ggsave(paste0(save_dir, "GO_BP_macaque_specific_cnetplot.pdf"), p_net, width = 11, height = 9)

# 聚类树状图（简化冗余通路）
ego_bp_s <- pairwise_termsim(ego_bp)
p_tree <- treeplot(ego_bp_s, showCategory = 20)
ggsave(paste0(save_dir, "GO_BP_macaque_specific_treeplot.pdf"), p_tree, width = 9, height = 7)

print(p_dot)

In [ ]:
save_dir <- "./data/"
# ---- 4. 出图（先剔除非 raphe/污染相关术语）----
# 要删的术语关键词：纤毛、感觉/视觉、精子、附肢/肢体、免疫/炎症
drop_kw <- paste(c(
  "cilium", "flagell", "sperm",
  "sensory perception", "visual perception", "detection of",
  "phototransduction", "appendage", "limb",
  "immune", "inflammatory", "leukocyte", "lymphocyte",
  "defense response", "complement", "cytokine"
), collapse = "|")

res <- ego_bp@result
cat("剔除前:", nrow(res), "条\n")
keep <- !grepl(drop_kw, res$Description, ignore.case = TRUE)
ego_bp_filt <- ego_bp
ego_bp_filt@result <- res[keep, ]
cat("剔除后:", nrow(ego_bp_filt@result), "条\n")
print(ego_bp_filt@result$Description)

# 保存清洗后的表
write.csv(as.data.frame(ego_bp_filt),
          paste0(save_dir, "GO_BP_macaque_specific_clean.csv"),
          row.names = FALSE)

# 气泡图（清洗版）
p_dot <- dotplot(ego_bp_filt, showCategory = 20, font.size = 9) +
  ggtitle("GO BP: macaque-specific 5-HT genes")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_dotplot_clean.pdf"),
       p_dot, width = 8, height = 6)
print(p_dot)

# 条形图（清洗版）
p_bar <- barplot(ego_bp_filt, showCategory = 15, font.size = 9) +
  ggtitle("GO BP: macaque-specific 5-HT genes")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_barplot_clean.pdf"),
       p_bar, width = 8, height = 6)

# 树状图（清洗版，看主题聚类）
ego_filt_s <- pairwise_termsim(ego_bp_filt)
p_tree <- treeplot(ego_filt_s, showCategory = 15)
ggsave(paste0(save_dir, "GO_BP_macaque_specific_treeplot_clean.pdf"),
       p_tree, width = 9, height = 6)

In [ ]:
# 只保留黏附/ECM/细胞连接/基质相关术语，其余全部不显示
keep_kw <- paste(c(
  "adhesion", "extracellular matrix", "extracellular structure",
  "encapsulating structure", "cell junction", "cell-cell adhesion",
  "integrin", "focal adhesion", "substrate adhesion"
), collapse = "|")

res <- ego_bp@result
ego_bp_filt <- ego_bp
ego_bp_filt@result <- res[grepl(keep_kw, res$Description, ignore.case = TRUE), ]
cat("保留:", nrow(ego_bp_filt@result), "条\n")
print(ego_bp_filt@result$Description)

write.csv(as.data.frame(ego_bp_filt),
          paste0(save_dir, "GO_BP_macaque_specific_clean.csv"), row.names = FALSE)

p_dot <- dotplot(ego_bp_filt, showCategory = 20, font.size = 10) +
  ggtitle("GO BP: macaque-specific 5-HT genes")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_dotplot_clean.pdf"),
       p_dot, width = 8, height = 5)
print(p_dot)

In [ ]:
# ---- 4. 出图（先剔除非 raphe/污染相关术语）----
# 要删的术语关键词：纤毛、感觉/视觉、精子、附肢/肢体、免疫/炎症
drop_kw <- paste(c(
  "cilium", "flagell", "sperm",
  "sensory perception", "visual perception", "detection of",
  "phototransduction", "appendage", "limb",
  "immune", "inflammatory", "leukocyte", "lymphocyte",
  "defense response", "complement", "cytokine"
), collapse = "|")

res <- ego_bp@result
cat("剔除前:", nrow(res), "条\n")
keep <- !grepl(drop_kw, res$Description, ignore.case = TRUE)
ego_bp_filt <- ego_bp
ego_bp_filt@result <- res[keep, ]
cat("剔除后:", nrow(ego_bp_filt@result), "条\n")
print(ego_bp_filt@result$Description)

# 保存清洗后的表
write.csv(as.data.frame(ego_bp_filt),
          paste0(save_dir, "GO_BP_macaque_specific_clean.csv"),
          row.names = FALSE)

# 气泡图（清洗版）
p_dot <- dotplot(ego_bp_filt, showCategory = 20, font.size = 9) +
  ggtitle("GO BP: macaque-specific 5-HT genes")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_dotplot_clean.pdf"),
       p_dot, width = 8, height = 6)
print(p_dot)

# 条形图（清洗版）
p_bar <- barplot(ego_bp_filt, showCategory = 15, font.size = 9) +
  ggtitle("GO BP: macaque-specific 5-HT genes")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_barplot_clean.pdf"),
       p_bar, width = 8, height = 6)

# 树状图（清洗版，看主题聚类）
ego_filt_s <- pairwise_termsim(ego_bp_filt)
p_tree <- treeplot(ego_filt_s, showCategory = 15)
ggsave(paste0(save_dir, "GO_BP_macaque_specific_treeplot_clean.pdf"),
       p_tree, width = 9, height = 6)

In [ ]:
suppressMessages({
  library(tidyverse); library(Seurat); library(dplyr)
  library(ggplot2); library(clusterProfiler); library(org.Hs.eg.db)
})

save_dir <- "./data/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

# ---- 0. 全部 macaque 差异基因（不做 top 截取）----
sig <- mk$p_val_adj < 0.05
genes_mac <- rownames(mk)[sig & mk$avg_log2FC > 0.5]   # 全部，一个不留
bg_genes  <- rownames(mk)                               # 背景 = 全部检验基因
cat("macaque-specific DEGs:", length(genes_mac), "\n")

# ---- 可选：剔除疑似污染基因（建议先 FeaturePlot 确认后启用）----
# contam_genes <- c("DNAH9","CFAP157","FOXJ1","RHO","PDE6B","CNGB1",   # 纤毛/光感受
#                   "ACTG2","TAGLN","MYH11","RGS5","KCNJ8","ABCC9")    # 平滑肌/周细胞
# genes_mac <- setdiff(genes_mac, contam_genes)
# cat("去污染后:", length(genes_mac), "\n")

pal_fill <- scale_fill_gradientn(colors = c("red", "orange", "purple", "blue"),
                                 values = c(0, 0.33, 0.66, 1))
pal_col  <- scale_color_gradientn(colors = c("red", "orange", "purple", "blue"),
                                  values = c(0, 0.33, 0.66, 1))

## =========================================================================
## 1. GO BP（全部 DEG，enrichGO）
## =========================================================================
ego <- enrichGO(gene = genes_mac, universe = bg_genes,
                OrgDb = org.Hs.eg.db, keyType = "SYMBOL",
                ont = "BP", pAdjustMethod = "BH",
                pvalueCutoff = 0.05, qvalueCutoff = 0.2,
                readable = TRUE)

go_tab <- as.data.frame(ego) %>%
  mutate(neglogP = -log10(pvalue),
         Description = str_wrap(Description, 40)) %>%
  arrange(desc(neglogP))
write.csv(go_tab, paste0(save_dir, "GO_BP_macaque_specific_allDEG.csv"),
          row.names = FALSE)
cat("GO BP 显著通路数:", nrow(go_tab), "\n")

# Top 10 条形图
top10 <- head(go_tab, 10) %>%
  mutate(Description = factor(Description, levels = rev(Description)))
p_go <- ggplot(top10, aes(neglogP, Description, fill = neglogP)) +
  geom_col() + pal_fill +
  labs(x = "-log10(P-value)", y = NULL, fill = "-log10(P-value)",
       title = "Top GO Biological Processes (macaque-specific)") +
  theme_classic(base_size = 13) +
  theme(plot.title = element_text(face = "bold", size = 12))
ggsave(paste0(save_dir, "GO_BP_macaque_specific_allDEG_top10.pdf"),
       p_go, width = 7.5, height = 5)
print(p_go)

# dotplot 版本（20 条）
p_dot <- dotplot(ego, showCategory = 20, font.size = 9) +
  ggtitle("GO BP: macaque-specific 5-HT genes (all DEGs)")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_allDEG_dotplot.pdf"),
       p_dot, width = 7.5, height = 8)

## =========================================================================
## 2. KEGG（全部 DEG，符号→Entrez 后 enrichKEGG）
## =========================================================================
df_g <- bitr(genes_mac, fromType = "SYMBOL", toType = "ENTREZID",
             OrgDb = org.Hs.eg.db)
df_b <- bitr(bg_genes, fromType = "SYMBOL", toType = "ENTREZID",
             OrgDb = org.Hs.eg.db)
cat("Entrez 映射:", nrow(df_g), "/", length(genes_mac), "\n")

ekegg <- enrichKEGG(gene = df_g$ENTREZID, universe = df_b$ENTREZID,
                    organism = "hsa", keyType = "kegg",
                    pAdjustMethod = "BH",
                    pvalueCutoff = 0.05, qvalueCutoff = 0.2)

kegg_tab <- as.data.frame(ekegg) %>%
  mutate(neglogP = -log10(pvalue)) %>%
  arrange(desc(neglogP))
write.csv(kegg_tab, paste0(save_dir, "KEGG_macaque_specific_allDEG.csv"),
          row.names = FALSE)
cat("KEGG 显著通路数:", nrow(kegg_tab), "\n")

# 神经/突触相关单独筛出（重点看 Serotonergic synapse）
neuro <- kegg_tab[grep("seroton|synap|dopamin|glutamat|calcium|neuroactive|long-term|ECM|focal adhesion",
                       kegg_tab$Description, ignore.case = TRUE), ]
write.csv(neuro, paste0(save_dir, "KEGG_macaque_specific_neuro_related.csv"),
          row.names = FALSE)
if (nrow(neuro) > 0) {
  cat("神经/突触/基质相关 KEGG:\n")
  print(neuro[, c("Description", "pvalue", "Count")])
}

# Top 5 气泡图
if (nrow(kegg_tab) > 0) {
  top5 <- head(kegg_tab, 5) %>%
    mutate(Description = str_to_upper(Description),
           Description = factor(Description, levels = rev(Description)))
  p_kegg <- ggplot(top5, aes(neglogP, Description, size = Count)) +
    geom_point(aes(color = neglogP)) + pal_col +
    scale_size_continuous(range = c(2, 8), name = "Gene Counts") +
    labs(x = "-log10(P-value)", y = "KEGG Terms", color = "-log10(P-value)",
         title = "Top 5 KEGG (macaque-specific)") +
    theme_classic(base_size = 13) +
    theme(plot.title = element_text(face = "bold", size = 12))
  ggsave(paste0(save_dir, "KEGG_macaque_specific_allDEG_top5.pdf"),
         p_kegg, width = 6.5, height = 4.5)
  print(p_kegg)
}

cat("全部完成，结果在", save_dir, "\n")

In [ ]:
contam_markers <- c("DNAH9","CFAP157","FOXJ1",      # 纤毛/室管膜
                    "RHO","PDE6B","CNGB1",          # 光感受
                    "ACTG2","TAGLN","MYH11","RGS5") # 平滑肌/周细胞
p_check <- FeaturePlot(classFiltSeurat, features = contam_markers,
                       reduction = "umap", ncol = 4, order = TRUE)
print(p_check)
ggsave(paste0(save_dir, "Check_contamination_featureplot.pdf"),
       p_check, width = 16, height = 8)

In [ ]:
# 1. 先定量看看有多少细胞受影响（心里有数）
contam_genes_detect <- intersect(contam_markers, rownames(classFiltSeurat))
n_pos <- colSums(GetAssayData(classFiltSeurat, slot = "data")[contam_genes_detect, , drop = FALSE] > 0)
cat("携带≥1个污染标志的细胞:", sum(n_pos > 0), "/", ncol(classFiltSeurat), "\n")

# 2. 从 DEG 列表里剔除这批基因（富集分析专用）
contam_genes <- c(contam_markers, "KCNJ8","ABCC9",        # 周细胞补全
                  "MYL9","DES","PLN")                     # 平滑肌/肌细胞补充
genes_mac_clean <- setdiff(genes_mac, contam_genes)
cat("剔除前:", length(genes_mac), " 剔除后:", length(genes_mac_clean), "\n")

# 3. 重跑 GO / KEGG（用 genes_mac_clean 替换 genes_mac，其余不变）

In [ ]:
## =========================================================================
## 按 GO 术语数据驱动地剔除污染基因
## =========================================================================
# 1. 从上次 GO 结果里捞出所有可疑术语（关键词匹配）
bad_kw <- "cilium|flagell|sperm motility|sensory perception|visual perception|detection of stimulus|detection of abiotic|detection of external|detection of chemical|appendage|limb|phototransduction|visual system"
bad_rows <- go_tab[grep(bad_kw, go_tab$Description, ignore.case = TRUE), ]
cat("可疑 GO 术语:", nrow(bad_rows), "条\n")

# 2. 这些术语命中的全部基因（readable=TRUE 时 geneID 是符号，用 / 分隔）
bad_genes <- unique(unlist(strsplit(bad_rows$geneID, "/")))
cat("涉及基因:", length(bad_genes), "个\n")

# 3. 从 DEG 里剔除
genes_mac_clean <- setdiff(genes_mac, bad_genes)
cat("剔除前:", length(genes_mac), " → 剔除后:", length(genes_mac_clean), "\n")

# 4. 重跑 GO / KEGG（用 genes_mac_clean 替换 genes_mac，背景集不变）

In [ ]:
## =========================================================================
## 4. 用 genes_mac_clean 重跑 GO BP
## =========================================================================
ego_clean <- enrichGO(gene = genes_mac_clean, universe = bg_genes,
                      OrgDb = org.Hs.eg.db, keyType = "SYMBOL",
                      ont = "BP", pAdjustMethod = "BH",
                      pvalueCutoff = 0.05, qvalueCutoff = 0.2,
                      readable = TRUE)

go_clean <- as.data.frame(ego_clean) %>%
  mutate(neglogP = -log10(pvalue),
         Description = str_wrap(Description, 40)) %>%
  arrange(desc(neglogP))
write.csv(go_clean, paste0(save_dir, "GO_BP_macaque_specific_clean.csv"),
          row.names = FALSE)
cat("去污染后 GO BP 显著通路数:", nrow(go_clean), "\n")

# Top 10 条形图
top10 <- head(go_clean, 10) %>%
  mutate(Description = factor(Description, levels = rev(Description)))
p_go <- ggplot(top10, aes(neglogP, Description, fill = neglogP)) +
  geom_col() +
  scale_fill_gradientn(colors = c("red", "orange", "purple", "blue"),
                       values = c(0, 0.33, 0.66, 1)) +
  labs(x = "-log10(P-value)", y = NULL, fill = "-log10(P-value)",
       title = "Top GO Biological Processes (macaque-specific, cleaned)") +
  theme_classic(base_size = 13) +
  theme(plot.title = element_text(face = "bold", size = 12))
ggsave(paste0(save_dir, "GO_BP_macaque_specific_clean_top10.pdf"),
       p_go, width = 7.5, height = 5)
print(p_go)

# Top 20 气泡图
p_dot <- dotplot(ego_clean, showCategory = 20, font.size = 9) +
  ggtitle("GO BP: macaque-specific 5-HT genes (cleaned)")
ggsave(paste0(save_dir, "GO_BP_macaque_specific_clean_dotplot.pdf"),
       p_dot, width = 7.5, height = 8)

## =========================================================================
## 5. 用 genes_mac_clean 重跑 KEGG
## =========================================================================
df_g <- bitr(genes_mac_clean, fromType = "SYMBOL", toType = "ENTREZID",
             OrgDb = org.Hs.eg.db)
df_b <- bitr(bg_genes, fromType = "SYMBOL", toType = "ENTREZID",
             OrgDb = org.Hs.eg.db)
cat("Entrez 映射:", nrow(df_g), "/", length(genes_mac_clean), "\n")

ekegg_clean <- enrichKEGG(gene = df_g$ENTREZID, universe = df_b$ENTREZID,
                          organism = "hsa", keyType = "kegg",
                          pAdjustMethod = "BH",
                          pvalueCutoff = 0.05, qvalueCutoff = 0.2)

kegg_clean <- as.data.frame(ekegg_clean) %>%
  mutate(neglogP = -log10(pvalue)) %>%
  arrange(desc(neglogP))
write.csv(kegg_clean, paste0(save_dir, "KEGG_macaque_specific_clean.csv"),
          row.names = FALSE)
cat("去污染后 KEGG 显著通路数:", nrow(kegg_clean), "\n")

# 神经/突触/基质相关单独筛出
neuro <- kegg_clean[grep("seroton|synap|dopamin|glutamat|calcium|neuroactive|long-term|ECM|focal adhesion|adherens|tight junction|gap junction",
                         kegg_clean$Description, ignore.case = TRUE), ]
write.csv(neuro, paste0(save_dir, "KEGG_macaque_specific_clean_neuro_related.csv"),
          row.names = FALSE)
if (nrow(neuro) > 0) {
  cat("神经/突触/基质相关 KEGG:\n")
  print(neuro[, c("Description", "pvalue", "Count")])
} else {
  cat("没有命中神经/突触相关 KEGG 通路\n")
}

# Top 5 气泡图
if (nrow(kegg_clean) > 0) {
  top5 <- head(kegg_clean, 5) %>%
    mutate(Description = str_to_upper(Description),
           Description = factor(Description, levels = rev(Description)))
  p_kegg <- ggplot(top5, aes(neglogP, Description, size = Count)) +
    geom_point(aes(color = neglogP)) +
    scale_color_gradientn(colors = c("red", "orange", "purple", "blue"),
                          values = c(0, 0.33, 0.66, 1)) +
    scale_size_continuous(range = c(2, 8), name = "Gene Counts") +
    labs(x = "-log10(P-value)", y = "KEGG Terms", color = "-log10(P-value)",
         title = "Top 5 KEGG (macaque-specific, cleaned)") +
    theme_classic(base_size = 13) +
    theme(plot.title = element_text(face = "bold", size = 12))
  ggsave(paste0(save_dir, "KEGG_macaque_specific_clean_top5.pdf"),
         p_kegg, width = 6.5, height = 4.5)
  print(p_kegg)
}

## =========================================================================
## 6. 清洗前后的对照表（给正文/补充材料用）
## =========================================================================
compare_tab <- data.frame(
  category = names(n_stat)[2],                    # Macaque-specific
  n_before = length(genes_mac),
  n_after  = length(genes_mac_clean),
  n_removed = length(genes_mac) - length(genes_mac_clean)
)
write.csv(compare_tab, paste0(save_dir, "Table_DEG_clean_summary.csv"),
          row.names = FALSE)
cat("全部完成，结果在", save_dir, "\n")

In [ ]:
ekegg_relax <- enrichKEGG(gene = df_g$ENTREZID, universe = df_b$ENTREZID,
                          organism = "hsa", keyType = "kegg",
                          pAdjustMethod = "BH",
                          pvalueCutoff = 0.2, qvalueCutoff = 0.5)  # 放宽
tab <- as.data.frame(ekegg_relax)
# 手动看 ECM / 突触相关条目，哪怕 p_adj 在 0.05~0.2 之间也可以报告
neuro <- tab[grep("seroton|synap|ECM|focal adhesion|calcium|neuroactive",
                  tab$Description, ignore.case = TRUE), ]
print(neuro[, c("Description", "pvalue", "p.adjust", "Count")])

In [ ]:
# 用清洗前的完整排序（mk_clean 或 mk），只把污染基因从排序里删掉
geneList <- mk$avg_log2FC
names(geneList) <- mk$gene
geneList <- geneList[!names(geneList) %in% bad_genes]   # bad_genes 是上一步的可疑基因
geneList <- sort(geneList, decreasing = TRUE)

# 符号 → Entrez
df_all <- bitr(names(geneList), fromType = "SYMBOL",
               toType = "ENTREZID", OrgDb = org.Hs.eg.db)
geneList <- geneList[df_all$SYMBOL]
names(geneList) <- df_all$ENTREZID
geneList <- sort(geneList, decreasing = TRUE)

gse <- gseKEGG(geneList = geneList, organism = "hsa",
               keyType = "kegg", pvalueCutoff = 0.05,
               pAdjustMethod = "BH", verbose = FALSE)
gse_tab <- as.data.frame(gse)
write.csv(gse_tab, paste0(save_dir, "GSEA_KEGG_macaque_vs_mouse_clean.csv"),
          row.names = FALSE)

# 看 NES>0（macaque 侧上调）的突触/ECM 通路
neuro_gse <- gse_tab[grep("seroton|synap|ECM|focal adhesion|calcium|neuroactive|long-term",
                          gse_tab$Description, ignore.case = TRUE) & gse_tab$NES > 0, ]
print(neuro_gse[, c("Description", "NES", "p.adjust")])

In [ ]:
ribo <- grep("^RPL|^RPS|^MRPL|^MRPS", rownames(mk), value = TRUE)
ribo_entrez <- bitr(ribo, fromType = "SYMBOL", toType = "ENTREZID",
                    OrgDb = org.Hs.eg.db)$ENTREZID
test_kegg <- enrichKEGG(gene = ribo_entrez, organism = "hsa",
                        pvalueCutoff = 0.05)
print(head(as.data.frame(test_kegg)))

In [ ]:
# metaneighbor

In [ ]:
formals(MetaNeighborUS)

In [ ]:
suppressMessages(library(MetaNeighbor))
suppressMessages(library(SummarizedExperiment))

## ---- 1. 准备输入 ----
expr_mat <- GetAssayData(classFiltSeurat[["SCT"]], slot = "data")
use_genes <- intersect(VariableFeatures(classFiltSeurat), rownames(expr_mat))
expr_mat <- as.matrix(expr_mat[use_genes, ])

keep_cells <- colnames(classFiltSeurat)[classFiltSeurat$subtype_recl != "Unassigned"]
meta_nn <- data.frame(
  cell_id    = keep_cells,
  study_id   = as.character(classFiltSeurat$species[keep_cells]),
  cell_label = as.character(classFiltSeurat$subtype_recl[keep_cells]),
  row.names  = keep_cells,
  stringsAsFactors = FALSE
)

# ★ 关键修改：包成 SummarizedExperiment
se_nn <- SummarizedExperiment(
  assays  = list(counts = expr_mat[, keep_cells]),
  colData = meta_nn[keep_cells, ]
)

## ---- 2. 选基因集 ----
var_genes <- variableGenes(dat = se_nn, exp_labels = se_nn$study_id)
message("MetaNeighbor 用基因数: ", length(var_genes))

## ---- 3. 跑 MetaNeighborUS（genesets 要 list；dat 传 se_nn）----
set.seed(42)
celltype_NV <- MetaNeighborUS(
  var_genes  = var_genes,          # 直接传基因向量，不用 list()
  dat        = se_nn,
  study_id   = se_nn$study_id,
  cell_type  = se_nn$cell_label
  # fast_version = TRUE 可加速（内存够的话建议打开）
)

In [ ]:
dim(celltype_NV)
head(rownames(celltype_NV))
head(colnames(celltype_NV))

In [ ]:
## ---- 3. 整理方向：行 = macaque，列 = mouse ----
parse_name <- function(x) {
  parts <- do.call(rbind, strsplit(x, "\\|"))
  data.frame(study = parts[, 1], label = parts[, 2],
             stringsAsFactors = FALSE)
}

rn <- parse_name(rownames(celltype_NV))
cn <- parse_name(colnames(celltype_NV))

mac_rows <- rownames(celltype_NV)[rn$study == "macaque"]
mus_cols <- colnames(celltype_NV)[cn$study == "mouse"]

NV <- celltype_NV[mac_rows, mus_cols, drop = FALSE]
rownames(NV) <- sub("^macaque\\|", "", mac_rows)
colnames(NV) <- sub("^mouse\\|",   "", mus_cols)

saveRDS(NV, paste0(out_dir, "celltype_NV_MetaNeighbor.rds"))

## ---- 4. 热图 + best match 表（这部分你原来的逻辑不用动）----
best_match_nn <- data.frame(
  macaque_subtype = rownames(NV),
  best_mouse      = colnames(NV)[apply(NV, 1, which.max)],
  AUROC           = round(apply(NV, 1, max), 3)
)
print(best_match_nn)
write.csv(best_match_nn, paste0(out_dir, "MetaNeighbor_best_match.csv"),
          row.names = FALSE)

In [ ]:
# ---- 4. 热图（行 macaque / 列 mouse）----
suppressMessages({ library(ComplexHeatmap); library(circlize); library(grid) })

ht <- Heatmap(
  NV,
  name = "AUROC",
  col  = colorRamp2(c(0.4, 0.5, 0.75, 1),
                    c("#3778AE", "#F7F7F7", "#F4A582", "#B2182B")),
  cluster_rows = TRUE,  cluster_columns = TRUE,
  clustering_method_rows = "ward.D2", clustering_method_columns = "ward.D2",
  row_names_side = "left",
  row_names_gp = gpar(fontsize = 9), column_names_gp = gpar(fontsize = 9),
  width = unit(7, "cm"), height = unit(8, "cm"),
  column_title = "mouse", row_title = "macaque"
)

png(paste0(out_dir, "MetaNeighbor_macaqueVSmouse.png"),
    width = 9, height = 8.5, units = "in", res = 300)
draw(ht, heatmap_legend_side = "right")
dev.off()

# 每行的最佳匹配表（AUROC 解读：>0.75 为高度保守）
best_match_nn <- data.frame(
  macaque_subtype = sub("^macaque\\|", "", rownames(NV)),
  best_mouse      = sub("^mouse\\|", "", colnames(NV)[apply(NV, 1, which.max)]),
  AUROC           = round(apply(NV, 1, max), 3)
)
print(best_match_nn)
write.csv(best_match_nn, paste0(out_dir, "MetaNeighbor_best_match.csv"),
          row.names = FALSE)

In [ ]:
ht <- Heatmap(
  NV,
  name = "AUROC",
  col  = colorRamp2(c(0.4, 0.5, 0.75, 1),
                    c("#3778AE", "#F7F7F7", "#F4A582", "#B2182B")),
  cluster_rows = TRUE,  cluster_columns = TRUE,
  clustering_method_rows = "ward.D2", clustering_method_columns = "ward.D2",
  row_names_side = "left",
  row_names_gp = gpar(fontsize = 9), column_names_gp = gpar(fontsize = 9),
  width = unit(7, "cm"), height = unit(8, "cm"),
  column_title = "mouse", row_title = "macaque",

  # ★ 把数值写进格子
  cell_fun = function(j, i, x, y, width, height, fill) {
    v <- NV[i, j]
    grid.text(sprintf("%.2f", v), x, y,
              gp = gpar(fontsize = 6.5,
                        # 深色底用白字，浅色底用黑字，保证可读
                        col = ifelse(v > 0.8, "white", "black")))
  }
)

png(paste0(out_dir, "MetaNeighbor_macaqueVSmouse.png"),
    width = 9, height = 8.5, units = "in", res = 300)
draw(ht, heatmap_legend_side = "right")
dev.off()

In [ ]:
# 桑基图

In [ ]:
library(reshape2)
library(ggalluvial)
library(ggplot2)

# 全量配对，不过滤；行 macaque、列 mouse
mac_types <- rownames(NV)   # macaque|mac5HT_x
mus_types <- colnames(NV)   # mouse|5-HT_x

df <- melt(NV, varnames = c("macaque", "mouse"), value.name = "nv")
df$macaque <- factor(df$macaque, levels = mac_types)
df$mouse   <- factor(df$mouse,   levels = mus_types)

p_sankey <- ggplot(df, aes(axis1 = macaque, axis2 = mouse, y = nv)) +
  geom_alluvium(aes(fill = macaque), alpha = 0.75, width = 1/10,
                knot.pos = 0.4, curve_type = "cubic") +
  geom_stratum(width = 1/10, fill = "grey95", color = "grey60") +
  geom_text(stat = "stratum", aes(label = after_stat(stratum)), size = 3) +
  scale_x_continuous(breaks = 1:2, labels = c("macaque", "mouse"),
                     expand = c(0.12, 0.12)) +
  theme_void(base_size = 13) +
  theme(legend.position = "none",
        axis.text.x = element_text(size = 13, face = "bold")) +
  ggtitle("macaque ↔ mouse 5-HT subtype mapping (MetaNeighbor)")

print(p_sankey)
ggsave(paste0(out_dir, "sankey_MetaNeighbor_all.pdf"),
       p_sankey, width = 14, height = 7)

# 只看强对应（AUROC > 0.7），图更清爽
df_strong <- subset(df, nv > 0.7)
if (nrow(df_strong) > 0) {
  p_sankey_strong <- ggplot(df_strong,
                            aes(axis1 = macaque, axis2 = mouse, y = nv)) +
    geom_alluvium(aes(fill = macaque), alpha = 0.8, width = 1/10,
                  knot.pos = 0.4, curve_type = "cubic") +
    geom_stratum(width = 1/10, fill = "grey95", color = "grey60") +
    geom_text(stat = "stratum", aes(label = after_stat(stratum)), size = 3.5) +
    scale_x_continuous(breaks = 1:2, labels = c("macaque", "mouse"),
                       expand = c(0.12, 0.12)) +
    theme_void(base_size = 13) +
    theme(legend.position = "none",
          axis.text.x = element_text(size = 13, face = "bold")) +
    ggtitle("macaque ↔ mouse (MetaNeighbor, AUROC > 0.7)")
  print(p_sankey_strong)
  ggsave(paste0(out_dir, "sankey_MetaNeighbor_strong.pdf"),
         p_sankey_strong, width = 12, height = 7)
}

In [ ]:
# 大于0.6

In [ ]:
library(reshape2)
library(ggalluvial)
library(ggplot2)

# 全量配对，不过滤；行 macaque、列 mouse
mac_types <- rownames(NV)   # macaque|mac5HT_x
mus_types <- colnames(NV)   # mouse|5-HT_x

df <- melt(NV, varnames = c("macaque", "mouse"), value.name = "nv")
df$macaque <- factor(df$macaque, levels = mac_types)
df$mouse   <- factor(df$mouse,   levels = mus_types)

p_sankey <- ggplot(df, aes(axis1 = macaque, axis2 = mouse, y = nv)) +
  geom_alluvium(aes(fill = macaque), alpha = 0.75, width = 1/10,
                knot.pos = 0.4, curve_type = "cubic") +
  geom_stratum(width = 1/10, fill = "grey95", color = "grey60") +
  geom_text(stat = "stratum", aes(label = after_stat(stratum)), size = 3) +
  scale_x_continuous(breaks = 1:2, labels = c("macaque", "mouse"),
                     expand = c(0.12, 0.12)) +
  theme_void(base_size = 13) +
  theme(legend.position = "none",
        axis.text.x = element_text(size = 13, face = "bold")) +
  ggtitle("macaque ↔ mouse 5-HT subtype mapping (MetaNeighbor)")

print(p_sankey)
ggsave(paste0(out_dir, "sankey_MetaNeighbor_all.pdf"),
       p_sankey, width = 14, height = 7)

# 只看强对应（AUROC > 0.6）
df_strong <- subset(df, nv > 0.6)          # ← 0.7 改成 0.6
if (nrow(df_strong) > 0) {
  p_sankey_strong <- ggplot(df_strong,
                            aes(axis1 = macaque, axis2 = mouse, y = nv)) +
    geom_alluvium(aes(fill = macaque), alpha = 0.8, width = 1/10,
                  knot.pos = 0.4, curve_type = "cubic") +
    geom_stratum(width = 1/10, fill = "grey95", color = "grey60") +
    geom_text(stat = "stratum", aes(label = after_stat(stratum)), size = 3.5) +
    scale_x_continuous(breaks = 1:2, labels = c("macaque", "mouse"),
                       expand = c(0.12, 0.12)) +
    theme_void(base_size = 13) +
    theme(legend.position = "none",
          axis.text.x = element_text(size = 13, face = "bold")) +
    ggtitle("macaque ↔ mouse (MetaNeighbor, AUROC > 0.6)")   # ← 标题同步改
  print(p_sankey_strong)
  ggsave(paste0(out_dir, "sankey_MetaNeighbor_strong06.pdf"),  # ← 文件名区分一下
         p_sankey_strong, width = 12, height = 7)
}

In [ ]:
# DEG

In [ ]:
suppressMessages(library(tidyverse))
library(ggrepel)
library(Seurat)

# classFiltSeurat <- readRDS("/mnt/90-connectome/Personal/XQ/a-scientific data投稿数据+代码/crossSpecies_final_5layers.rds")

## ===== 1. 差异分析：macaque vs mouse（所有 5-HT 细胞）=====
# RNA 有 5 个 layer，合并后整体归一化
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])

DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, normalization.method = "LogNormalize",
                                 scale.factor = 10000, verbose = FALSE)

Idents(classFiltSeurat) <- "species"

deg <- FindMarkers(classFiltSeurat,
                   ident.1 = "macaque", ident.2 = "mouse",
                   assay = "RNA", slot = "data",
                   logfc.threshold = 0.1, min.pct = 0.1,
                   test.use = "wilcox")
saveRDS(deg, "./data/DEG_macaque_vs_mouse.rds")

## ===== 2. 整理 + 火山图 =====
df <- deg %>%
  mutate(gene = rownames(deg),
         sig = case_when(p_val_adj < 0.05 & avg_log2FC >  1 ~ "up",
                         p_val_adj < 0.05 & avg_log2FC < -1 ~ "down",
                         TRUE ~ "ns"),
         sig = factor(sig, levels = c("down", "ns", "up")))

# 上下各标 5 个最显著的基因（图上带名字的那种）
lab <- bind_rows(
  df %>% filter(sig == "up")   %>% slice_min(p_val_adj, n = 5),
  df %>% filter(sig == "down") %>% slice_min(p_val_adj, n = 5)
)

n_up <- sum(df$sig == "up"); n_dn <- sum(df$sig == "down"); n_ns <- sum(df$sig == "ns")

p <- ggplot(df, aes(avg_log2FC, -log10(p_val_adj), color = sig)) +
  geom_point(size = 0.7, alpha = 0.6) +
  scale_color_manual(values = c(down = "#3778AE", ns = "grey85", up = "#D6251F")) +
  geom_vline(xintercept = c(-1, 1), linetype = "dashed", color = "grey50") +
  geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "grey50") +
  geom_text_repel(data = lab, aes(label = gene), size = 3.5,
                  color = "black", max.overlaps = 30, segment.size = 0.3) +
  labs(title = paste0("DEG: macaque 5-HT vs mouse 5-HT\n",
                      "down(", n_dn, ")    ns(", n_ns, ")    up(", n_up, ")"),
       x = "Log2 FoldChange", y = "-Log10 P.adj", color = NULL) +
  theme_classic(base_size = 13)

print(p)
ggsave("./data/volcano_macaque_vs_mouse1.pdf",
       p, width = 7, height = 6)

In [ ]:
library(ggplot2)
library(dplyr)
library(ggrepel)

df <- deg %>%
  mutate(gene  = rownames(deg),
         y_raw = -log10(p_val_adj),
         ## ★ y ≤ 50 的一律算 ns（不管 FC 多大），>50 才分 up/down
         sig = case_when(y_raw > 50 & avg_log2FC >  1 ~ "up",
                         y_raw > 50 & avg_log2FC < -1 ~ "down",
                         TRUE ~ "ns"),
         sig = factor(sig, levels = c("down", "ns", "up")))

## ★ 标签基因：恢复每侧 top5（在 y>50 的显著基因里选）
pick_spread <- function(sub, n = 5) {
  sub <- sub %>% arrange(p_val_adj)
  bind_rows(
    slice_head(sub, n = 2),
    sub %>% filter(!(gene %in% slice_head(sub, n = 2)$gene)) %>%
      mutate(rank_fc = rank(-abs(avg_log2FC))) %>%
      arrange(rank_fc) %>% slice_head(n = n - 2)
  )
}
lab <- bind_rows(
  df %>% filter(sig == "up")   %>% pick_spread(5),
  df %>% filter(sig == "down") %>% pick_spread(5)
)

n_up <- sum(df$sig == "up"); n_dn <- sum(df$sig == "down"); n_ns <- sum(df$sig == "ns")

p <- ggplot(df, aes(avg_log2FC, y_raw, color = sig)) +
  geom_point(size = 0.7, alpha = 0.6) +
  scale_color_manual(values = c(down = "#3778AE", ns = "grey85", up = "#D6251F")) +
  geom_vline(xintercept = c(-1, 1), linetype = "dashed", color = "grey50") +
  ## ★ 水平虚线：从 -log10(0.05) 挪到 y = 50
  geom_hline(yintercept = 50, linetype = "dashed", color = "grey50") +
  geom_text_repel(
    data = lab, aes(label = gene),
    size = 3.5, color = "black", fontface = "italic",
    box.padding = 0.8,
    point.padding = 0.5,
    force = 5, force_pull = 0.2,
    direction = "both",
    max.overlaps = Inf,
    min.segment.length = 0,
    seed = 42,
    segment.size = 0.3, segment.alpha = 0.6
  ) +
  coord_cartesian(clip = "off") +
  labs(title = paste0("DEG: macaque 5-HT vs mouse 5-HT\n",
                      "down(", n_dn, ")    ns(", n_ns, ")    up(", n_up, ")"),
       x = "Log2 FoldChange", y = "-Log10 P.adj", color = NULL) +
  theme_classic(base_size = 13) +
  theme(plot.margin = margin(20, 40, 20, 20))

print(p)
ggsave("./data/volcano_macaque_vs_mouse-50.pdf",
       p, width = 8, height = 7)

In [ ]:
# 某个基因的两个物种表达量小提琴图

In [ ]:
suppressMessages(library(tidyverse))
library(Seurat)

# classFiltSeurat <- readRDS("/mnt/90-connectome/Personal/XQ/a-scientific data投稿数据+代码/crossSpecies_final_5layers.rds")
classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)

## ===== 1. 指定基因 =====
top_genes <- c("PENK", "TAC1", "NXPH2", "RIMS2", "ROBO2","CAPS2","RUFY3"
)

## 检查一下这三个基因是否都在对象里（跨物种数据可能有基因名匹配问题）
missing_genes <- setdiff(top_genes, rownames(classFiltSeurat))
if (length(missing_genes) > 0) {
  cat("⚠️ 这些基因不在对象中:", paste(missing_genes, collapse = ", "), "\n")
}

cat("Genes:", paste(top_genes, collapse = ", "), "\n")

## ===== 2. 一次提取 + 转长表 =====
expr <- FetchData(classFiltSeurat, vars = c("species", top_genes))
expr_long <- expr %>%
  pivot_longer(cols = -species, names_to = "gene", values_to = "Expression") %>%
  mutate(gene = factor(gene, levels = top_genes),   # 按你给的顺序排
         species = factor(species, levels = c("macaque", "mouse")))

## ===== 3. 出图 =====
p <- ggplot(expr_long, aes(x = species, y = Expression, fill = species)) +
  geom_violin(scale = "width", trim = TRUE) +
  geom_boxplot(width = 0.15, outlier.size = 0.15, alpha = 0.7, color = "grey30") +
  facet_wrap(~ gene, nrow = 1, scales = "free_y") +
  scale_fill_manual(values = c("macaque" = "#EB6368", "mouse" = "#82ADD0")) +
  labs(title = "PENK / TAC1 / NXPH2: macaque vs mouse",
       x = NULL, y = "Normalized Expression") +
  theme_bw(base_size = 13) +
  theme(legend.position = "none",
        strip.text = element_text(face = "bold", size = 12),
        axis.text.x = element_text(angle = 45, hjust = 1))

print(p)
ggsave("./data/VlnPlot_PENK_TAC1_NXPH2_macaque_vs_mouse.png",
       p, width = 10, height = 5, dpi = 300)

In [ ]:
# 1. 看目标目录下到底有什么 rds（注意大小写、空格、扩展名）
list.files("./data/",
           pattern = "\\.rds$")

# 2. 如果上面是空字符向量，往上一层找
list.files("./data/",
           pattern = "crossSpecies.*\\.rds$", recursive = TRUE)

# 3. 全盘兜底（如果挂载点路径有变）
list.files("/mnt/90-connectome", pattern = "crossSpecies_CCA_dims50_mac5HT\\.rds",
           recursive = TRUE, full.names = TRUE)

In [ ]:
suppressMessages(library(tidyverse))
library(Seurat)

classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
DefaultAssay(classFiltSeurat) <- "RNA"
classFiltSeurat <- NormalizeData(classFiltSeurat, verbose = FALSE)

## ===== 1. 指定基因 =====
top_genes <- c("PENK", "TAC1", "NXPH2", "RIMS2", "ROBO2", "CAPS2", "RUFY3")

missing_genes <- setdiff(top_genes, rownames(classFiltSeurat))
if (length(missing_genes) > 0) {
  cat("⚠️ 这些基因不在对象中:", paste(missing_genes, collapse = ", "), "\n")
}

## ===== 2. 一次提取 + 转长表（多取 donor 和细胞类型两列）=====
expr <- FetchData(classFiltSeurat,
                  vars = c("species", "donor", "subtype_recl", top_genes))

# 看看 donor 和细胞类型实际有哪些值（顺便检查 NA）
table(expr$donor, useNA = "ifany")
table(expr$subtype_recl, useNA = "ifany")

expr_long <- expr %>%
  pivot_longer(cols = all_of(top_genes),
               names_to = "gene", values_to = "Expression") %>%
  mutate(gene   = factor(gene, levels = top_genes),
         donor  = factor(donor),
         celltype = factor(subtype_recl))

## ===== 3. 出图：x = 细胞类型，fill = donor，分面 = 基因 =====
p <- ggplot(expr_long, aes(x = celltype, y = Expression, fill = donor)) +
  geom_violin(scale = "width", trim = TRUE, position = position_dodge(width = 0.9)) +
  geom_boxplot(width = 0.12, outlier.size = 0, alpha = 0.6, color = "grey30",
               position = position_dodge(width = 0.9)) +
  facet_wrap(~ gene, nrow = 2, scales = "free_y") +
  labs(title = "Expression across cell types and donors",
       x = NULL, y = "Normalized Expression", fill = "Donor") +
  theme_bw(base_size = 12) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        strip.text = element_text(face = "bold", size = 11),
        legend.position = "right")

print(p)
ggsave("./data/VlnPlot_topGenes_byDonor_celltype.png",
       p, width = 16, height = 9, dpi = 300)

In [ ]:
suppressMessages({
  library(tidyverse)
  library(Seurat)
  library(patchwork)
  library(Matrix)
  library(harmony)
})

## =========================================================================
## 可调参数区
## =========================================================================
PAR <- list(
  batch_downsample = TRUE,   # 每个 batch 抽平到最小 batch 细胞数
  variable_genes   = 5000,
  npcs             = 100,
  cca_dims         = 1:30,
  nb_dims          = 1:30,
  resolution       = 0.1,
  min_cells        = 100
)

set.seed(42)
out_dir <- "./data/"

## =========================================================================
## 1. 读取数据 + 清理
## =========================================================================
macaque <- readRDS(paste0(out_dir, "macaque_5HT_integrated.rds"))
table(macaque$renew_type)

DefaultAssay(macaque) <- "RNA"
macaque <- DietSeurat(macaque, assays = "RNA", dimreducs = FALSE, graphs = FALSE)
macaque[["RNA"]] <- JoinLayers(macaque[["RNA"]])

macaque$donor <- as.character(macaque$donor)
macaque$donor[is.na(macaque$donor)] <- as.character(macaque$Individual[is.na(macaque$donor)])
macaque$maxPredClass <- "5-HT"
Idents(macaque) <- "renew_type"

mouse <- readRDS("./data/mouse_sero.combined_res0.5.rds")
mouse$maxPredClass <- "5-HT"
mouse$donor <- mouse$group
Idents(mouse) <- "renew_type"

## =========================================================================
## 2. 同源基因转换
## =========================================================================
test <- read.table('./data/mart_export.txt',
                   sep = "\t", header = TRUE)
comGeneTb <- test[c("Gene.name", "Mouse.gene.name")]
colnames(comGeneTb) <- c('macaqueGene', 'mouseGene')
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$macaqueGene), ]
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$mouseGene), ]
comGeneTb <- comGeneTb[!duplicated(comGeneTb$macaqueGene), ]

prepareHomoSeurat <- function(seurat, originalGeneName, homoGeneName,
                              keepMetaCol, commonMetaCol,
                              speciesPrefix = "macaque_", clusterCol = "renew_type") {
  homoCountMx <- GetAssayData(seurat[["RNA"]], slot = "counts")[originalGeneName, ]
  rownames(homoCountMx) <- homoGeneName
  colnames(homoCountMx) <- paste0(speciesPrefix, colnames(homoCountMx))
  homoSeurat <- CreateSeuratObject(counts = homoCountMx)
  tmpDf <- seurat@meta.data[sub(paste0("^", speciesPrefix), "",
                                colnames(homoSeurat)), keepMetaCol]
  tmpDf$species <- sub("(_|-)", "", speciesPrefix)
  tmpDf$clusterName <- paste0(speciesPrefix, tmpDf[[clusterCol]])
  commonMetaCol <- c(commonMetaCol, "species", "clusterName")
  colnames(tmpDf) <- ifelse(colnames(tmpDf) %in% commonMetaCol,
                            colnames(tmpDf),
                            paste0(speciesPrefix, colnames(tmpDf)))
  rownames(tmpDf) <- paste0(speciesPrefix, rownames(tmpDf))
  homoSeurat <- AddMetaData(homoSeurat, metadata = tmpDf)
  return(homoSeurat)
}

comGeneTb$homoGeneSymbol <- comGeneTb$macaqueGene
comGeneTb <- subset(comGeneTb,
  macaqueGene %in% rownames(GetAssayData(macaque[["RNA"]], slot = "counts")) &
  mouseGene   %in% rownames(GetAssayData(mouse[["RNA"]], slot = "counts")))
message("一对一同源基因: ", nrow(comGeneTb))

macaqueHomoSeurat <- prepareHomoSeurat(
  macaque, comGeneTb$macaqueGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "macaque_"
)

mouseHomoSeurat <- prepareHomoSeurat(
  mouse, comGeneTb$mouseGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "group", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "mouse_"
)

mouseHomoSeurat$species   <- "mouse"
macaqueHomoSeurat$species <- "macaque"

## =========================================================================
## 3. 合并小 donor + ★ 每个 batch 分层抽平到最小 batch
## =========================================================================
own_donors <- c("Mq246", "Mq277")

macaqueHomoSeurat$batch <- ifelse(
  macaqueHomoSeurat$donor %in% own_donors,
  macaqueHomoSeurat$donor,
  "NHP_published"
)
mouseHomoSeurat$batch <- mouseHomoSeurat$donor

cat("--- 分层后各 batch 细胞数（抽平前）---\n")
print(table(macaqueHomoSeurat$batch))
print(table(mouseHomoSeurat$batch))

if (PAR$batch_downsample) {
  # 最小 batch 细胞数 = 抽平目标（当前是 mouse2 = 913）
  target_per_batch <- min(c(table(macaqueHomoSeurat$batch),
                            table(mouseHomoSeurat$batch)))
  message("每个 batch 抽平到 ", target_per_batch, " 细胞（batch 内按亚型分层）")
  
  # 单个对象内：每个 batch 独立分层抽样
  downsample_by_batch <- function(obj, target_per_batch) {
    ct <- obj$clusterName
    bt <- obj$batch
    cells <- unlist(lapply(split(seq_len(ncol(obj)), bt), function(idx) {
      n_target <- min(length(idx), target_per_batch)
      tab <- table(ct[idx])
      n_per <- round(tab / sum(tab) * n_target)
      n_per <- pmin(n_per, tab)
      n_per[names(which.max(n_per))] <- n_per[names(which.max(n_per))] +
        (n_target - sum(n_per))
      unlist(lapply(names(n_per), function(x) {
        cells_x <- colnames(obj)[idx][ct[idx] == x]
        sample(cells_x, min(n_per[x], length(cells_x)))
      }))
    }))
    sample(cells, min(length(cells), sum(pmin(table(bt), target_per_batch))))
  }
  
  macaqueHomoSeurat <- subset(macaqueHomoSeurat,
                              cells = downsample_by_batch(macaqueHomoSeurat, target_per_batch))
  mouseHomoSeurat   <- subset(mouseHomoSeurat,
                              cells = downsample_by_batch(mouseHomoSeurat, target_per_batch))
  
  cat("--- 抽平后各 batch 细胞数 ---\n")
  print(table(macaqueHomoSeurat$batch))
  print(table(mouseHomoSeurat$batch))
}

## =========================================================================
## 4. split + merge（预期 5 层，每层 913）
## =========================================================================
mouseHomoSeurat[["RNA"]]   <- split(mouseHomoSeurat[["RNA"]],
                                    f = mouseHomoSeurat$batch)
macaqueHomoSeurat[["RNA"]] <- split(macaqueHomoSeurat[["RNA"]],
                                    f = macaqueHomoSeurat$batch)

classFiltSeurat <- merge(mouseHomoSeurat, macaqueHomoSeurat)

cat("--- 合并后各层细胞数 ---\n")
print(sapply(Layers(classFiltSeurat[["RNA"]]),
             function(l) ncol(LayerData(classFiltSeurat[["RNA"]], layer = l))))

layer_cells <- sapply(Layers(classFiltSeurat[["RNA"]]),
                      function(l) ncol(LayerData(classFiltSeurat[["RNA"]], layer = l)))
if (any(layer_cells < PAR$min_cells)) {
  message("发现小层，执行过滤...")
  keep_layers <- names(layer_cells)[layer_cells >= PAR$min_cells]
  cells_keep <- unlist(lapply(keep_layers, function(l)
    colnames(LayerData(classFiltSeurat[["RNA"]], layer = l))))
  classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
  classFiltSeurat <- subset(classFiltSeurat, cells = cells_keep)
  classFiltSeurat[["RNA"]] <- split(classFiltSeurat[["RNA"]],
                                    f = classFiltSeurat$batch)
}

## =========================================================================
## 5. SCT + PCA
## =========================================================================
classFiltSeurat <- SCTransform(
  classFiltSeurat, assay = "RNA",
  ncells = ncol(classFiltSeurat[["RNA"]]),
  variable.features.n = PAR$variable_genes,
  method = "glmGamPoi"
)
classFiltSeurat <- RunPCA(classFiltSeurat, npcs = PAR$npcs)
print(ElbowPlot(classFiltSeurat, ndims = PAR$npcs))

## =========================================================================
## 6. CCA 整合
## =========================================================================
classFiltSeurat <- IntegrateLayers(
  object = classFiltSeurat,
  method = CCAIntegration,
  orig.reduction = "pca",
  new.reduction = "integrated.cca",
  normalization.method = "SCT",
  dims = PAR$cca_dims,
  verbose = TRUE
)

## =========================================================================
## 7. 聚类 + UMAP
## =========================================================================
classFiltSeurat <- FindNeighbors(classFiltSeurat,
                                 reduction = "integrated.cca",
                                 dims = PAR$nb_dims)
classFiltSeurat <- FindClusters(classFiltSeurat,
                                resolution = PAR$resolution)
classFiltSeurat <- RunUMAP(classFiltSeurat,
                           reduction = "integrated.cca",
                           dims = PAR$nb_dims,
                           reduction.name = "umap")

## =========================================================================
## 8. 可视化
## =========================================================================
colorLeg  <- c(macaque = "#EB6368", mouse = "#82ADD0")
batchCols <- c("mouse1" = "#82ADD0", "mouse2" = "#6A93B8",
               "Mq246" = "#EB6368", "Mq277" = "#D14E5B",
               "NHP_published" = "#F5B971")

p1 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "species",
              cols = colorLeg, raster = FALSE) +
  coord_fixed() + ggtitle("5-HT cross-species (CCA, batch-balanced)")
print(p1)
ggsave(paste0(out_dir, "UMAP_CCA_species_batchbal.pdf"), p1, width = 6, height = 5)

p2 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "batch",
              cols = batchCols) +
  coord_fixed() + ggtitle("batch check")
print(p2)
ggsave(paste0(out_dir, "UMAP_CCA_batch_batchbal.pdf"), p2, width = 6.5, height = 5)

classFiltSeurat$subtype <- sub("^(macaque|mouse)_", "",
                               classFiltSeurat$clusterName)

p3 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype",
              split.by = "species", label = TRUE, repel = TRUE,
              label.size = 3.5, ncol = 2, raster = FALSE) +
  coord_fixed() + NoLegend() + ggtitle("subtypes split by species")
print(p3)
ggsave(paste0(out_dir, "UMAP_CCA_subtype_split_batchbal.pdf"), p3, width = 12, height = 6)

# ## =========================================================================
# ## 9. 保存
# ## =========================================================================
# saveRDS(classFiltSeurat, paste0(out_dir, "crossSpecies_CCA_5layers_batch913.rds"))
# writeLines(capture.output(sessionInfo()),
#            paste0(out_dir, "sessionInfo_CCA_batch913.txt"))
# message("完成，共 ", ncol(classFiltSeurat), " 细胞（5 batch × 913）")

In [ ]:
suppressMessages({
  library(tidyverse)
  library(Seurat)
  library(patchwork)
  library(Matrix)
  library(harmony)
})

## =========================================================================
## 可调参数区（★ PC 统一 50，batch 抽平 913）
## =========================================================================
PAR <- list(
  batch_downsample = TRUE,
  variable_genes   = 5000,
  npcs             = 100,
  cca_dims         = 1:50,
  nb_dims          = 1:50,
  resolution       = 0.1,
  min_cells        = 100
)

set.seed(42)
out_dir <- "./data/"

## =========================================================================
## 1. 读取数据 + 清理
## =========================================================================
macaque <- readRDS(paste0(out_dir, "macaque_5HT_integrated.rds"))
table(macaque$renew_type)

DefaultAssay(macaque) <- "RNA"
macaque <- DietSeurat(macaque, assays = "RNA", dimreducs = FALSE, graphs = FALSE)
macaque[["RNA"]] <- JoinLayers(macaque[["RNA"]])

macaque$donor <- as.character(macaque$donor)
macaque$donor[is.na(macaque$donor)] <- as.character(macaque$Individual[is.na(macaque$donor)])
macaque$maxPredClass <- "5-HT"
Idents(macaque) <- "renew_type"

mouse <- readRDS("./data/mouse_sero.combined_res0.5.rds")
mouse$maxPredClass <- "5-HT"
mouse$donor <- mouse$group
Idents(mouse) <- "renew_type"

## =========================================================================
## 2. 同源基因转换
## =========================================================================
test <- read.table('./data/mart_export.txt',
                   sep = "\t", header = TRUE)
comGeneTb <- test[c("Gene.name", "Mouse.gene.name")]
colnames(comGeneTb) <- c('macaqueGene', 'mouseGene')
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$macaqueGene), ]
comGeneTb <- comGeneTb[grep('[A-Za-z]', comGeneTb$mouseGene), ]
comGeneTb <- comGeneTb[!duplicated(comGeneTb$macaqueGene), ]

prepareHomoSeurat <- function(seurat, originalGeneName, homoGeneName,
                              keepMetaCol, commonMetaCol,
                              speciesPrefix = "macaque_", clusterCol = "renew_type") {
  homoCountMx <- GetAssayData(seurat[["RNA"]], slot = "counts")[originalGeneName, ]
  rownames(homoCountMx) <- homoGeneName
  colnames(homoCountMx) <- paste0(speciesPrefix, colnames(homoCountMx))
  homoSeurat <- CreateSeuratObject(counts = homoCountMx)
  tmpDf <- seurat@meta.data[sub(paste0("^", speciesPrefix), "",
                                colnames(homoSeurat)), keepMetaCol]
  tmpDf$species <- sub("(_|-)", "", speciesPrefix)
  tmpDf$clusterName <- paste0(speciesPrefix, tmpDf[[clusterCol]])
  commonMetaCol <- c(commonMetaCol, "species", "clusterName")
  colnames(tmpDf) <- ifelse(colnames(tmpDf) %in% commonMetaCol,
                            colnames(tmpDf),
                            paste0(speciesPrefix, colnames(tmpDf)))
  rownames(tmpDf) <- paste0(speciesPrefix, rownames(tmpDf))
  homoSeurat <- AddMetaData(homoSeurat, metadata = tmpDf)
  return(homoSeurat)
}

comGeneTb$homoGeneSymbol <- comGeneTb$macaqueGene
comGeneTb <- subset(comGeneTb,
  macaqueGene %in% rownames(GetAssayData(macaque[["RNA"]], slot = "counts")) &
  mouseGene   %in% rownames(GetAssayData(mouse[["RNA"]], slot = "counts")))
message("一对一同源基因: ", nrow(comGeneTb))

macaqueHomoSeurat <- prepareHomoSeurat(
  macaque, comGeneTb$macaqueGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "macaque_"
)

mouseHomoSeurat <- prepareHomoSeurat(
  mouse, comGeneTb$mouseGene, comGeneTb$homoGeneSymbol,
  keepMetaCol = c("nCount_RNA", "nFeature_RNA", "maxPredClass",
                  "renew_type", "group", "donor"),
  commonMetaCol = c("nCount_RNA", "nFeature_RNA", "donor"),
  speciesPrefix = "mouse_"
)

mouseHomoSeurat$species   <- "mouse"
macaqueHomoSeurat$species <- "macaque"

## =========================================================================
## 3. 合并小 donor + 每个 batch 分层抽平到最小 batch
## =========================================================================
own_donors <- c("Mq246", "Mq277")

macaqueHomoSeurat$batch <- ifelse(
  macaqueHomoSeurat$donor %in% own_donors,
  macaqueHomoSeurat$donor,
  "NHP_published"
)
mouseHomoSeurat$batch <- mouseHomoSeurat$donor

cat("--- 分层后各 batch 细胞数（抽平前）---\n")
print(table(macaqueHomoSeurat$batch))
print(table(mouseHomoSeurat$batch))

if (PAR$batch_downsample) {
  target_per_batch <- min(c(table(macaqueHomoSeurat$batch),
                            table(mouseHomoSeurat$batch)))
  message("每个 batch 抽平到 ", target_per_batch, " 细胞（batch 内按亚型分层）")
  
  downsample_by_batch <- function(obj, target_per_batch) {
    ct <- obj$clusterName
    bt <- obj$batch
    cells <- unlist(lapply(split(seq_len(ncol(obj)), bt), function(idx) {
      n_target <- min(length(idx), target_per_batch)
      tab <- table(ct[idx])
      n_per <- round(tab / sum(tab) * n_target)
      n_per <- pmin(n_per, tab)
      n_per[names(which.max(n_per))] <- n_per[names(which.max(n_per))] +
        (n_target - sum(n_per))
      unlist(lapply(names(n_per), function(x) {
        cells_x <- colnames(obj)[idx][ct[idx] == x]
        sample(cells_x, min(n_per[x], length(cells_x)))
      }))
    }))
    sample(cells, min(length(cells), sum(pmin(table(bt), target_per_batch))))
  }
  
  macaqueHomoSeurat <- subset(macaqueHomoSeurat,
                              cells = downsample_by_batch(macaqueHomoSeurat, target_per_batch))
  mouseHomoSeurat   <- subset(mouseHomoSeurat,
                              cells = downsample_by_batch(mouseHomoSeurat, target_per_batch))
  
  cat("--- 抽平后各 batch 细胞数 ---\n")
  print(table(macaqueHomoSeurat$batch))
  print(table(mouseHomoSeurat$batch))
}

## =========================================================================
## 4. split + merge（预期 5 层，每层 913）
## =========================================================================
mouseHomoSeurat[["RNA"]]   <- split(mouseHomoSeurat[["RNA"]],
                                    f = mouseHomoSeurat$batch)
macaqueHomoSeurat[["RNA"]] <- split(macaqueHomoSeurat[["RNA"]],
                                    f = macaqueHomoSeurat$batch)

classFiltSeurat <- merge(mouseHomoSeurat, macaqueHomoSeurat)

cat("--- 合并后各层细胞数 ---\n")
print(sapply(Layers(classFiltSeurat[["RNA"]]),
             function(l) ncol(LayerData(classFiltSeurat[["RNA"]], layer = l))))

layer_cells <- sapply(Layers(classFiltSeurat[["RNA"]]),
                      function(l) ncol(LayerData(classFiltSeurat[["RNA"]], layer = l)))
if (any(layer_cells < PAR$min_cells)) {
  message("发现小层，执行过滤...")
  keep_layers <- names(layer_cells)[layer_cells >= PAR$min_cells]
  cells_keep <- unlist(lapply(keep_layers, function(l)
    colnames(LayerData(classFiltSeurat[["RNA"]], layer = l))))
  classFiltSeurat[["RNA"]] <- JoinLayers(classFiltSeurat[["RNA"]])
  classFiltSeurat <- subset(classFiltSeurat, cells = cells_keep)
  classFiltSeurat[["RNA"]] <- split(classFiltSeurat[["RNA"]],
                                    f = classFiltSeurat$batch)
}

## =========================================================================
## 5. SCT + PCA
## =========================================================================
classFiltSeurat <- SCTransform(
  classFiltSeurat, assay = "RNA",
  ncells = ncol(classFiltSeurat[["RNA"]]),
  variable.features.n = PAR$variable_genes,
  method = "glmGamPoi"
)
classFiltSeurat <- RunPCA(classFiltSeurat, npcs = PAR$npcs)
print(ElbowPlot(classFiltSeurat, ndims = PAR$npcs))

## =========================================================================
## 6. CCA 整合（dims = 1:50）
## =========================================================================
classFiltSeurat <- IntegrateLayers(
  object = classFiltSeurat,
  method = CCAIntegration,
  orig.reduction = "pca",
  new.reduction = "integrated.cca",
  normalization.method = "SCT",
  dims = PAR$cca_dims,
  verbose = TRUE
)

## =========================================================================
## 7. 聚类 + UMAP（dims = 1:50）
## =========================================================================
classFiltSeurat <- FindNeighbors(classFiltSeurat,
                                 reduction = "integrated.cca",
                                 dims = PAR$nb_dims)
classFiltSeurat <- FindClusters(classFiltSeurat,
                                resolution = PAR$resolution)
classFiltSeurat <- RunUMAP(classFiltSeurat,
                           reduction = "integrated.cca",
                           dims = PAR$nb_dims,
                           reduction.name = "umap")

## =========================================================================
## 8. 可视化（macaque 用合并重聚类后的 mac_subtype）
## =========================================================================
colorLeg  <- c(macaque = "#EB6368", mouse = "#82ADD0")
batchCols <- c("mouse1" = "#82ADD0", "mouse2" = "#6A93B8",
               "Mq246" = "#EB6368", "Mq277" = "#D14E5B",
               "NHP_published" = "#F5B971")

p1 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "species",
              cols = colorLeg, raster = FALSE) +
  coord_fixed() + ggtitle("5-HT cross-species (CCA dims50, batch-balanced)")
print(p1)
ggsave(paste0(out_dir, "UMAP_CCA_dims50_species.pdf"), p1, width = 6, height = 5)

p2 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "batch",
              cols = batchCols) +
  coord_fixed() + ggtitle("batch check")
print(p2)
ggsave(paste0(out_dir, "UMAP_CCA_dims50_batch.pdf"), p2, width = 6.5, height = 5)

# ---- macaque：读入合并重聚类结果，映射 mac_subtype ----
mac_re <- readRDS(paste0(out_dir, "macaque_5HT_merged_reclustered.rds"))
mac_labels <- setNames(as.character(mac_re$mac_subtype), colnames(mac_re))

mac_cells_cf <- colnames(classFiltSeurat)[classFiltSeurat$species == "macaque"]
mac_stripped <- sub("^macaque_", "", mac_cells_cf)

classFiltSeurat$subtype_recl <- NA
classFiltSeurat$subtype_recl[classFiltSeurat$species == "macaque"] <-
  mac_labels[mac_stripped]
classFiltSeurat$subtype_recl[classFiltSeurat$species == "mouse"] <-
  sub("^mouse_", "",
      as.character(classFiltSeurat$clusterName[classFiltSeurat$species == "mouse"]))

# 映射质检（必须看输出）
n_missing <- sum(is.na(classFiltSeurat$subtype_recl))
cat("标签缺失细胞数:", n_missing, "/", ncol(classFiltSeurat), "\n")
classFiltSeurat$subtype_recl[is.na(classFiltSeurat$subtype_recl)] <- "Unassigned"
classFiltSeurat$subtype_recl <- factor(classFiltSeurat$subtype_recl)

print(table(classFiltSeurat$subtype_recl, classFiltSeurat$species))
write.csv(as.matrix(table(classFiltSeurat$subtype_recl, classFiltSeurat$species)),
          paste0(out_dir, "subtype_recl_vs_species.csv"))

# split 图：macaque 重聚类亚型 vs mouse 原标签
p3 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              split.by = "species", label = TRUE, repel = TRUE,
              label.size = 3.5, ncol = 2, raster = FALSE) +
  coord_fixed() + NoLegend() +
  ggtitle("subtypes split (macaque = merged-reclustered)")
print(p3)
ggsave(paste0(out_dir, "UMAP_CCA_dims50_subtype_split_recl.pdf"),
       p3, width = 12, height = 6)

# 合并版
p4 <- DimPlot(classFiltSeurat, reduction = "umap", group.by = "subtype_recl",
              label = TRUE, repel = TRUE, label.size = 3, raster = FALSE) +
  coord_fixed() + NoLegend()
print(p4)
ggsave(paste0(out_dir, "UMAP_CCA_dims50_subtype_all_recl.pdf"),
       p4, width = 7, height = 6)

## =========================================================================
## 9. Overlap 热图（clusterName vs seurat_clusters）
## =========================================================================
suppressMessages({
  library(ComplexHeatmap)
  library(circlize)
  library(grid)
})

compare_cl <- function(cl, ref.cl, plot.title = NA, plot.silent = TRUE,
                       heat.colors = colorRampPalette(c("white", "grey70", "black"))(100),
                       row.cl.num = min(length(unique(cl)), length(unique(ref.cl)))) {
  conf1 <- table(cl, ref.cl)
  conf1 <- sweep(conf1, 1, rowSums(conf1), "/")
  conf2 <- reorder_matrix(conf1)
  cl.prop.cocl <- apply(conf1, 2, function(x) {
    grid1 <- expand.grid(x, x)
    min.prop <- apply(grid1, 1, min)
  })
  cl.prop.cocl.total <- apply(cl.prop.cocl, 1, sum)
  cl.prop.cocl.m <- matrix(cl.prop.cocl.total, nrow(conf1), nrow(conf1),
                           dimnames = list(rownames(conf1), rownames(conf1)))
  ph1 <- pheatmap::pheatmap(conf2, cutree_rows = row.cl.num,
                            clustering_method = "ward.D2",
                            color = heat.colors, fontsize = 6,
                            main = plot.title, silent = plot.silent)
  return(list(conf = conf2, cocl = cl.prop.cocl.m, ph = ph1))
}

reorder_matrix <- function(matrix1, by.rows = TRUE) {
  if (by.rows == TRUE) {
    conf.order <- order(apply(matrix1, 1, which.max))
    matrix1.reordered <- matrix1[conf.order, ]
  } else {
    conf.order <- order(apply(matrix1, 2, which.max))
    matrix1.reordered <- matrix1[, conf.order]
  }
}

cl.conf <- compare_cl(classFiltSeurat$clusterName,
                      classFiltSeurat$seurat_clusters)
cocl <- cl.conf$cocl

cocl.subset <- cocl[grepl("macaque", rownames(cocl)),
                    grepl("mouse",   rownames(cocl))]

heat.colors <- colorRampPalette(c("white", "grey70", "black"))(100)

row_max <- apply(cocl.subset, 1, max)
bar_colors <- ifelse(row_max > 0.3, "#D6251F", "#3778AE")

left_anno <- rowAnnotation(
  "Max Overlap\nscore" = anno_barplot(
    row_max,
    gp = gpar(fill = bar_colors, col = NA),
    bar_width = 0.9,
    border = FALSE,
    axis_param = list(direction = "reverse")
  ),
  width = unit(3, "cm")
)

bar_legend <- Legend(
  title = "Type",
  at = c("macaque-specific", "macaque/mouse-conserved"),
  legend_gp = gpar(fill = c("#3778AE", "#D6251F")),
  border = "black",
  labels_gp = gpar(fontsize = 10)
)

ht <- Heatmap(
  cocl.subset,
  name = "Overlap score",
  col = heat.colors,
  cluster_rows = FALSE,
  cluster_columns = FALSE,
  left_annotation = left_anno,
  row_names_side = "right",
  row_names_gp = gpar(fontsize = 10),
  column_names_gp = gpar(fontsize = 10),
  width = unit(5, "cm"),
  height = unit(7, "cm"),
)

png(paste0(out_dir, "overlap_cca_dims50_batch913.png"),
    width = 8, height = 7, units = "in", res = 300)

draw(ht, annotation_legend_side = "bottom",
     heatmap_legend_list = list(bar_legend))

decorate_annotation("Max Overlap\nscore", {
  n <- length(row_max)
  for (j in seq_len(n)) {
    grid.text(label = sprintf("%.2f", row_max[j]),
              x = unit(row_max[j], "native"),
              y = unit(1 - (j - 0.5) / n, "npc"),
              just = "right",
              gp = gpar(fontsize = 8))
  }
})

decorate_annotation("Max Overlap\nscore", {
  grid.lines(x = unit(0.7, "native"), y = unit(c(0, 1), "npc"),
             gp = gpar(lty = 2, col = "black"))
})

dev.off()

write.csv(cocl.subset, paste0(out_dir, "overlap_cca_dims50_batch913.csv"))
cat("Overlap 热图与数值表已保存\n")

# ## =========================================================================
# ## 10. 保存对象
# ## =========================================================================
# saveRDS(classFiltSeurat,
#         paste0(out_dir, "crossSpecies_CCA_dims50_batch913.rds"))
# writeLines(capture.output(sessionInfo()),
#            paste0(out_dir, "sessionInfo_CCA_dims50_batch913.txt"))
# message("完成，共 ", ncol(classFiltSeurat), " 细胞（5 batch × 913）")